# FloodCastBench - the corrected study (Colab)

**This is the current notebook.** `flood_forecasting_colab.ipynb` and `flood_forecasting_multires_colab.ipynb` reproduce the ORIGINAL model (v1) and
its original evaluation; a later audit found three problems with them (see `PROJECT_LOG.md`): (1) the rainfall input carried up to 30 minutes of rain
that had **not yet fallen**, (2) a trivial per-pixel linear model beat the original CNN-LSTM, and (3) v1 could not use the depth history at all.
This notebook re-runs the depth-forecasting study with those problems fixed.

**What it does** (per experiment: `E60` Australia -> UK at 60 m, `E30` the same at 30 m, `E480` Pakistan -> Mozambique at 480 m):

1. builds a cache of every wet window of the training region and of the never-seen test region;
2. runs the **baseline ladder** against persistence ("copy the last frame"): linear models, a two-feature lookup table, a per-pixel MLP;
3. trains the **neural models** (a 4.1k-parameter local CNN, and a small full-resolution ConvLSTM = the project's CNN-LSTM) in **three rainfall regimes**;
4. prints skill vs persistence with confidence intervals on **held-out tiles** and on the **never-seen region**, plus what each model uses
   (test-time removal of rainfall, terrain and depth history).

**The three rainfall regimes** (rainfall file `j` is the rain falling during `[1800 j, 1800 (j+1))` s of the simulation - verified physically):
`causal` = only rain that has already fallen; `hybrid` = the original input (the interval containing each frame, partly future);
`forced` = hybrid plus the rainfall of the 5 forecast intervals (a rain-forced surrogate, the setting the paper's FNO+ works in).

**Cost.** `E60` downloads about 3.4 GB + rainfall; `E30` about 13.5 GB; `E480` about 9.5 GB (Pakistan). Defaults run `E60` only.

## Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "rasterio", "scipy"], check=False)
import warnings
import tensorflow as tf
import rasterio
warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)
print("GPU available:", tf.config.list_physical_devices('GPU'))
print("TensorFlow version:", tf.__version__)

## Configuration

`EPOCHS = 30` and `N_TRAIN = 300` are the frozen recipe of the study (`completion/PREREGISTRATION.md`); reduce them only for a quick smoke test.

In [ ]:
# CONFIG (overridable)
EXPERIMENTS = ["E60"]                      # any of "E60", "E30", "E480"
REGIMES = ["hybrid", "causal", "forced"]
MODELS = ["local", "clstm"]                # any of "local1", "local", "local11", "clstm"
FOLDS = [0, 1, 2]
EPOCHS = 30
N_TRAIN = 300
THREADS = 4                                # CPU threads per training process
RUN_BASELINES = True
RUN_SEVERITY = False                       # the severity / rules / Grad-CAM study needs E30 (Australia at 30 m)

## Pull the data from Hugging Face

In [ ]:
from huggingface_hub import snapshot_download

PATTERNS = ["Relevant data/Georeferenced files/*"]
def add(regions, fid):
    global PATTERNS
    for r in regions:
        PATTERNS += [f"{fid}/{r}/*", f"Relevant data/DEM/{r}_DEM.*", f"Relevant data/Land use and land cover/{r}.*", f"Relevant data/Rainfall/{r} flood/*"]
if "E60" in EXPERIMENTS: add(["Australia", "UK"], "High-fidelity flood forecasting/60m")
if "E30" in EXPERIMENTS or RUN_SEVERITY: add(["Australia", "UK"], "High-fidelity flood forecasting/30m")
if "E480" in EXPERIMENTS: add(["Pakistan", "Mozambique"], "Low-fidelity flood forecasting/480m")
DATA_ROOT = snapshot_download(repo_id="mahajananhad/floodcastbench", repo_type="dataset", allow_patterns=PATTERNS, max_workers=16)
print("Downloaded to:", DATA_ROOT)

## Modules (verified locally; identical to the ones in the code handoff)

In [ ]:
import os
for d in ("completion", "completion/results", "completion/results/runs", "completion/logs", "completion/cache", "data/extracted"):
    os.makedirs(d, exist_ok=True)
if not os.path.exists("data/extracted/FloodCastBench"):
    os.symlink(DATA_ROOT, "data/extracted/FloodCastBench")
print("data linked at data/extracted/FloodCastBench")

In [ ]:
%%writefile real_data_loader.py
"""
Real FloodCastBench data loader for one region (Stage 1 of the build plan).

Loads a spatial patch + temporal window of depth, DEM, and rainfall for a
single region, aligns rainfall's separate 30-minute calendar-date series to
depth's 5-minute simulation-second series, and derives a severity label from
future depth using the fixed thresholds decided in the project plan.

Verified against the real downloaded files:
  - depth: <region>/<t>.tif, t = unpadded seconds since sim start, step 300s
  - rainfall: Relevant data/Rainfall/<region> flood/<YYYYMMDD>-S<HHMMSS>.tif, step 1800s
  - Australia: depth, rainfall, DEM and land use all share the same 1073x1073 grid.
  - UK: depth is 170x275 but the other layers are 180x285 - a measured (5, 5) pixel
    offset is applied (see LAYER_GEOMETRY).
  - 60 m and 480 m depth rasters (Australia/UK at 60 m; Pakistan/Mozambique at 480 m) are the 30 m
    simulation resampled to 2x2 / 16x16 base-pixel cells and cropped, while DEM / land use / rainfall
    stay on the 30 m base grid. LAYER_GEOMETRY records each combination's scale and crop offset
    (recovered from the data and cross-checked, see its comment); DEM / rainfall are block-averaged
    to the depth cells. A combination with no registered geometry is refused, never guessed.
    (An earlier version of this loader claimed all layers matched - checked only on Australia - and
    later refused Pakistan/Mozambique as "not alignable" - a failure of the first search, not a
    property of the data.)

CONFIRMED PHYSICALLY (completion/a_rain_lag.py, round 2): rainfall file j is the rain that FALLS DURING [1800 j, 1800 (j+1)) s of the simulation; the values are mm/h
intensities (depth gain over an interval = 0.494-0.502 x the file value; correlation 1.00 at lag 0 in all four regions). The archive itself gives no clock time for
t = 0 (no TIFF tag or readme ties it to a calendar date); none is used. Original reasoning: rainfall's first file corresponds to depth's t=0. Evidence for this: for all four events the rainfall series starts on the
paper's stated event start date, and its duration equals the depth duration to the second
(e.g. Australia: 481 files at 1800 s = 10 days = 864000 s of depth at 300 s). Date-level
agreement was checked first; the 30-minute alignment is now measured (see above).
"""
import os
import re
import glob
from datetime import datetime

import numpy as np
import rasterio

DEPTH_STEP_SECONDS = 300
RAINFALL_STEP_SECONDS = 1800

# Pixel offset (rows, cols) of the depth grid inside the DEM / land-use / rainfall grid,
# for regions where those layers are larger than the depth raster. Measured, not assumed:
# UK depth is 170x275 but its DEM/land-use/rainfall are 180x285. Both an independent
# depth-vs-DEM correlation search (best at dy=5,dx=5, r=-0.27 vs a max of +0.03 over all
# 121 candidate offsets) and a wet-pixels-on-baseline-water enrichment search (~5x at
# dy=5..6,dx=5..6) peak there - consistent with a centered 5px border crop on each side.
# Australia's layers all match the depth grid exactly (verified) so it needs no entry.
#
# Geometry of EVERY (region, resolution) relative to the 30 m base grid that the DEM / land use / rainfall live on:
#   (scale, (row_offset, col_offset))  = base pixels per depth cell, and the base-pixel position of depth cell (0, 0).
# The 60 m and 480 m depth rasters are the 30 m results resampled to 2x2 and 16x16 base-pixel cells and CROPPED (the paper:
# "resampled ... using bilinear interpolation"). Each geometry below was recovered from the data with a probe that was first
# validated on the known cases (Australia 30 m -> (0,0), UK 30 m -> (5,5)), and cross-checked by independent signals:
#   Mozambique 480 m  (16, (150, 50))  terrain-structure r=-0.27 (5.2 robust-sd), DEM NoData all outside the crop
#   Pakistan   480 m  (16, (42, 178))  terrain-structure r=-0.09 (5.5 robust-sd) AND baseline-water r=+0.10 agree; NoData outside
#   UK          60 m  (2,  (5, 5))     terrain-structure r=-0.44 at (5..6, 5..6)
#   Australia   60 m  (2,  (0, 0))     terrain-structure r=-0.26 at (0..1, 0..1)
# Precision is about +-2 base pixels (60 m), i.e. < 6% of a 480 m cell. Scripts and logs: verification_outputs/audit/.
# (An earlier version of this loader refused Pakistan and Mozambique as "not alignable"; that was a failure of the first
# search - it did not remove the large-scale terrain trend - not a property of the data.)
LAYER_GEOMETRY = {
    ("Australia", 30): (1, (0, 0)), ("UK", 30): (1, (5, 5)),
    ("Australia", 60): (2, (0, 0)), ("UK", 60): (2, (5, 5)),
    ("Pakistan", 480): (16, (42, 178)), ("Mozambique", 480): (16, (150, 50)),
}
LAYER_OFFSETS = {"UK": (5, 5)}  # kept for older callers: 30 m offsets only

# The DEM / rainfall rasters carry the float32 NoData sentinel (-3.4028235e38): the UK layers have a padding ring of it
# outside the depth-aligned crop. A window that touched it would feed ~1e38 into the network (and the flat-DEM filter
# in windows.py would count such a patch as "valid", its DEM std being enormous). None of the windows this project
# uses touch it (verification_outputs/audit/investigate_uk_nodata.py), but the loader now refuses instead of trusting that.
NODATA_BELOW = -1e30


def has_nodata(arr: np.ndarray) -> bool:
    """True if `arr` holds the NoData sentinel or any non-finite value."""
    return bool((arr < NODATA_BELOW).any() or not np.isfinite(arr).all())

# Fixed absolute depth thresholds (meters), decided in the project plan.
SEVERITY_THRESHOLDS = [0.3, 1.0, 2.0]  # -> none / minor / moderate / severe
SEVERITY_CLASSES = ["none", "minor", "moderate", "severe"]


def depth_to_severity(max_depth_m: float) -> int:
    """Bin a max-depth value (meters) into one of the 4 fixed severity classes."""
    idx = 0
    for t in SEVERITY_THRESHOLDS:
        if max_depth_m >= t:
            idx += 1
    return idx  # 0=none, 1=minor, 2=moderate, 3=severe


class RegionData:
    """Indexes one region's depth/rainfall/DEM files without loading pixel data."""

    def __init__(self, root_dir: str, region: str, fidelity_dir: str):
        """
        root_dir: path to the extracted FloodCastBench folder
        region: e.g. "Australia"
        fidelity_dir: e.g. "High-fidelity flood forecasting/30m"
        """
        self.root_dir = root_dir
        self.region = region

        depth_dir = os.path.join(root_dir, fidelity_dir, region)
        depth_files = glob.glob(os.path.join(depth_dir, "*.tif"))
        if not depth_files:
            raise FileNotFoundError(f"No depth TIFFs found in {depth_dir}")

        def depth_seconds(path):
            name = os.path.splitext(os.path.basename(path))[0]
            m = re.match(r"^(\d+)$", name)
            if not m:
                raise ValueError(f"Expected purely numeric depth filename, got: {path}")
            return int(m.group(1))

        self.depth_files = sorted(depth_files, key=depth_seconds)
        self.depth_seconds = [depth_seconds(p) for p in self.depth_files]

        step = self.depth_seconds[1] - self.depth_seconds[0]
        for i in range(1, len(self.depth_seconds)):
            gap = self.depth_seconds[i] - self.depth_seconds[i - 1]
            if gap != step:
                raise ValueError(
                    f"Depth sequence is not evenly spaced at index {i}: "
                    f"gap={gap}s, expected {step}s"
                )
        self.depth_step_seconds = step

        rain_dir = os.path.join(root_dir, "Relevant data", "Rainfall", f"{region} flood")
        rain_files = glob.glob(os.path.join(rain_dir, "*.tif"))
        if not rain_files:
            raise FileNotFoundError(f"No rainfall TIFFs found in {rain_dir}")

        def rain_datetime(path):
            name = os.path.splitext(os.path.basename(path))[0]
            m = re.match(r"^(\d{8})-S(\d{6})$", name)
            if not m:
                raise ValueError(f"Unexpected rainfall filename format: {path}")
            return datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M%S")

        rain_with_dt = sorted(((rain_datetime(p), p) for p in rain_files), key=lambda x: x[0])
        self.rain_files = [p for _, p in rain_with_dt]
        self.rain_start = rain_with_dt[0][0]
        rain_step = (rain_with_dt[1][0] - rain_with_dt[0][0]).total_seconds()
        for i in range(1, len(rain_with_dt)):
            gap = (rain_with_dt[i][0] - rain_with_dt[i - 1][0]).total_seconds()
            if gap != rain_step:
                raise ValueError(f"Rainfall sequence is not evenly spaced at index {i}")
        self.rain_step_seconds = rain_step

        self.dem_path = os.path.join(root_dir, "Relevant data", "DEM", f"{region}_DEM.tif")
        if not os.path.isfile(self.dem_path):
            raise FileNotFoundError(self.dem_path)

        self.landuse_path = os.path.join(root_dir, "Relevant data", "Land use and land cover", f"{region}.tif")
        if not os.path.isfile(self.landuse_path):
            raise FileNotFoundError(self.landuse_path)

        with rasterio.open(self.depth_files[0]) as src:
            self.grid_height, self.grid_width = src.height, src.width

        m = re.search(r"(\d+)m$", fidelity_dir.rstrip("/"))
        if not m:
            raise ValueError(
                f"Cannot parse pixel size from fidelity_dir={fidelity_dir!r} "
                f"(expected it to end in e.g. '30m')"
            )
        self.pixel_size_m = float(m.group(1))

        key = (region, int(round(self.pixel_size_m)))
        if key not in LAYER_GEOMETRY:
            raise ValueError(
                f"{region} at {self.pixel_size_m:.0f} m has no registered layer geometry in LAYER_GEOMETRY - windows read "
                f"at the same (row, col) would cover different geography. Refusing to load rather than return misaligned data."
            )
        self.layer_scale, self.layer_offset = LAYER_GEOMETRY[key]
        dy, dx = self.layer_offset
        s = self.layer_scale
        for label, path in (("DEM", self.dem_path), ("land use", self.landuse_path),
                            ("rainfall", self.rain_files[0])):
            with rasterio.open(path) as src:
                layer_h, layer_w = src.height, src.width
            if dy + s * self.grid_height > layer_h or dx + s * self.grid_width > layer_w:
                raise ValueError(
                    f"{region}: {label} grid is {layer_h}x{layer_w}, too small for the registered geometry "
                    f"(scale {s}, offset {self.layer_offset}) of a {self.grid_height}x{self.grid_width} depth grid."
                )

    def _layer_window(self, row: int, col: int, patch_size: int):
        """Window (in 30 m base pixels) into the DEM / land-use / rainfall grids for a depth-grid (row, col) patch."""
        dy, dx = self.layer_offset
        s = self.layer_scale
        return ((dy + s * row, dy + s * (row + patch_size)), (dx + s * col, dx + s * (col + patch_size)))

    def _read_layer(self, path: str, row: int, col: int, patch_size: int, nearest: bool = False) -> np.ndarray:
        """Read a base-grid layer for a depth-grid patch, reduced to (patch, patch): block-mean over the cell's
        scale x scale base pixels (nearest = the cell-centre pixel, for categorical land use)."""
        s = self.layer_scale
        with rasterio.open(path) as src:
            a = src.read(1, window=self._layer_window(row, col, patch_size), boundless=True, fill_value=0)
        if s == 1:
            return a
        a = a.reshape(patch_size, s, patch_size, s)
        if nearest:
            return a[:, s // 2, :, s // 2]
        return a.astype(np.float64).mean(axis=(1, 3)).astype(np.float32)

    def load_dem_patch(self, row: int, col: int, patch_size: int) -> np.ndarray:
        return self._read_layer(self.dem_path, row, col, patch_size).astype(np.float32)

    def rainfall_index_for_depth_second(self, depth_second: int) -> int:
        """Map a depth timestep (seconds since sim start) to its rainfall file index,
        holding each 30-min rainfall value constant across the depth steps inside it.
        NOTE (round 2): file j covers [1800 j, 1800 (j+1)) s, so this returns rain that has NOT YET FALLEN for a frame inside the interval (the 'hybrid' regime: partly
        future rain for the last observed frame). For past-only input use file floor(t/1800) - 1 ('causal'; see completion/engine.py)."""
        idx = int(depth_second // self.rain_step_seconds)
        return min(idx, len(self.rain_files) - 1)

    def window_frame_indices(self, start_idx: int, t_in: int, horizon: int, stride: int = 1):
        """Depth-file indices for one window: (history_indices, future_indices).

        History is start_idx, start_idx+stride, ...; the forecast continues at the same
        stride. stride=1 is the native 5-minute step; stride=6 is 30 minutes, which is
        rainfall's native resolution (at stride 1, five history frames span only 25
        minutes, so the rainfall channel is essentially constant across the sequence).
        The severity scan in windows.py uses this same method so scan and loader can
        never disagree about which frames a window covers."""
        history = [start_idx + stride * k for k in range(t_in)]
        future = [start_idx + stride * (t_in + k) for k in range(horizon)]
        if future[-1] >= len(self.depth_files):
            raise ValueError(
                f"start_idx={start_idx} t_in={t_in} horizon={horizon} stride={stride} runs past "
                f"the end of the depth sequence ({len(self.depth_files)} files)"
            )
        return history, future

    def max_start_index(self, t_in: int, horizon: int, stride: int = 1) -> int:
        """Largest start_idx for which a whole window still fits."""
        return len(self.depth_files) - 1 - stride * (t_in + horizon - 1)

    def load_patch_sequence(self, row: int, col: int, patch_size: int,
                             start_idx: int, t_in: int, horizon: int, stride: int = 1):
        """Read one (row, col) patch across the window described by window_frame_indices():
        the history frames are the model input, the future frames are both the regression
        targets and the source of the severity label.

        Returns: input_sequence (t_in, H, W, 3) [depth, DEM, rainfall (mm/h; for the last frame this includes up to 30 min of not-yet-fallen rain)],
                 depth_targets (horizon, H, W),
                 severity_label (int, 0-3), max_future_depth (float)
        """
        window = ((row, row + patch_size), (col, col + patch_size))
        history_idx, future_idx = self.window_frame_indices(start_idx, t_in, horizon, stride)

        dem_patch = self.load_dem_patch(row, col, patch_size)

        depth_frames = []
        rainfall_frames = []
        for i in history_idx:
            with rasterio.open(self.depth_files[i]) as src:
                depth_frames.append(src.read(1, window=window, boundless=True, fill_value=0).astype(np.float32))
            rain_idx = self.rainfall_index_for_depth_second(self.depth_seconds[i])
            rainfall_frames.append(self._read_layer(self.rain_files[rain_idx], row, col, patch_size).astype(np.float32))

        depth_hist = np.stack(depth_frames)          # (t_in, H, W)
        rainfall_hist = np.stack(rainfall_frames)     # (t_in, H, W)
        dem_seq = np.broadcast_to(dem_patch, depth_hist.shape)  # (t_in, H, W)
        input_sequence = np.stack([depth_hist, dem_seq, rainfall_hist], axis=-1)  # (t_in, H, W, 3)

        depth_targets = []
        for i in future_idx:
            with rasterio.open(self.depth_files[i]) as src:
                depth_targets.append(src.read(1, window=window, boundless=True, fill_value=0).astype(np.float32))
        depth_targets = np.stack(depth_targets)  # (horizon, H, W)

        for what, arr in (("depth history", depth_hist), ("rainfall", rainfall_hist),
                          ("DEM", dem_patch), ("depth targets", depth_targets)):
            if has_nodata(arr):
                raise ValueError(
                    f"{self.region}: {what} window at depth-grid (row={row}, col={col}) contains NoData/NaN pixels "
                    f"(the layers have a NoData padding ring outside the aligned crop). Refusing to return it."
                )

        max_future_depth = float(depth_targets.max())
        severity_label = depth_to_severity(max_future_depth)

        return input_sequence, depth_targets, severity_label, max_future_depth

    def load_landuse_patch(self, row: int, col: int, patch_size: int) -> np.ndarray:
        """Real Esri/Impact Observatory Sentinel-2 land-cover codes (verified against
        this exact product's published legend): 1=Water, 2=Trees, 4=Flooded Vegetation,
        5=Crops, 7=Built Area, 8=Bare Ground, 9=Snow/Ice, 10=Clouds, 11=Rangeland.
        At 60 m / 480 m the class of each cell's centre pixel is returned (nearest neighbour)."""
        return self._read_layer(self.landuse_path, row, col, patch_size, nearest=True).astype(np.int32)


In [ ]:
%%writefile windows.py
"""
Window scanning and sampling for FloodCastBench regions.

A "window" is one (spatial patch, start time). Severity is computed EXACTLY the way
RegionData.load_patch_sequence computes it - max depth in the patch over the future
frames from RegionData.window_frame_indices - and both use that same method, so a scan
can never disagree with what the loader later returns.

Each scanned window is a tuple:
    (row, col, start_idx, severity, baseline_severity)
where baseline_severity is what you'd get by just applying the severity thresholds to the
max depth of the LAST OBSERVED frame (a "persistence" classifier that needs no model).
Measured on real Australia data, that trivial classifier is 99.9% accurate at a 5-minute
step and 96.7% at a 30-minute stride - so any severity model has to be judged against it.
"""
import numpy as np
import rasterio

from real_data_loader import SEVERITY_CLASSES, RegionData, depth_to_severity, has_nodata


def all_positions(rd: RegionData, patch_size: int):
    """Non-overlapping patch origins that fit entirely inside the depth grid."""
    return [(r, c) for r in range(0, rd.grid_height - patch_size + 1, patch_size)
                   for c in range(0, rd.grid_width - patch_size + 1, patch_size)]


def valid_positions(rd: RegionData, patch_size: int, dem_nodata_std: float = 1.0):
    """Drop patches whose DEM is (near-)flat - those lie outside the simulation domain - and patches whose
    DEM window touches the NoData sentinel (its std is enormous, so the flat-DEM test alone would keep it)."""
    keep = []
    for (r, c) in all_positions(rd, patch_size):
        dem = rd.load_dem_patch(r, c, patch_size)
        if not has_nodata(dem) and dem.std() > dem_nodata_std:
            keep.append((r, c))
    return keep


def start_indices_for(rd: RegionData, count: int, t_in: int, horizon: int, stride: int):
    """`count` start indices spread evenly over every start at which a whole window fits."""
    last = rd.max_start_index(t_in, horizon, stride)
    return np.unique(np.linspace(0, last, count).astype(int))


def scan_windows(rd: RegionData, positions, start_indices, patch_size: int,
                 t_in: int, horizon: int, stride: int):
    out = []
    for start_idx in start_indices:
        history_idx, future_idx = rd.window_frame_indices(int(start_idx), t_in, horizon, stride)
        with rasterio.open(rd.depth_files[history_idx[-1]]) as src:
            last_observed = src.read(1)
        future_max = None
        for i in future_idx:
            with rasterio.open(rd.depth_files[i]) as src:
                frame = src.read(1)
            future_max = frame if future_max is None else np.maximum(future_max, frame)
        for (row, col) in positions:
            sl = (slice(row, row + patch_size), slice(col, col + patch_size))
            out.append((row, col, int(start_idx),
                        depth_to_severity(float(future_max[sl].max())),
                        depth_to_severity(float(last_observed[sl].max()))))
    return out


def class_fractions(windows):
    total = len(windows)
    return {c: sum(1 for w in windows if w[3] == i) / total for i, c in enumerate(SEVERITY_CLASSES)}


def balanced_sample(pool, per_class: int, rng):
    """Up to `per_class` randomly-chosen windows of each severity class."""
    chosen = []
    for cls_idx, cls in enumerate(SEVERITY_CLASSES):
        members = [w for w in pool if w[3] == cls_idx]
        take = min(per_class, len(members))
        if take < per_class:
            print(f"  WARNING: only {len(members)} '{cls}' windows available (wanted {per_class})")
        pick = rng.choice(len(members), size=take, replace=False)
        chosen.extend(members[i] for i in pick)
    return chosen


In [ ]:
%%writefile capsule_layers.py
"""
Capsule Network layers (Sabour, Frosst & Hinton 2017, arXiv:1710.09829) for the
severity-classification branch: PrimaryCaps -> routing-by-agreement -> per-class
output capsules, whose vector lengths give the 4-way severity probability.

Kept as plain tf.keras.layers.Layer subclasses (not a separate framework) so the
whole model - depth-forecasting branch and severity branch alike - trains as one
TensorFlow/Keras graph: gradients from the severity loss reach back through
routing, through PrimaryCaps, into the SAME shared CNN encoder the LSTM branch
uses. This is the piece that was previously impossible when the capsule layer
lived in a separate PyTorch model (see project plan, Stage 3).
"""
import tensorflow as tf
from tensorflow.keras import layers


def squash(vectors: tf.Tensor, axis: int = -1, epsilon: float = 1e-7) -> tf.Tensor:
    """Capsule nonlinearity: shrinks short vectors to (near) zero and long vectors
    to just under length 1, so a capsule's output length can be read as a
    probability. squash(s) = (|s|^2 / (1 + |s|^2)) * (s / |s|)."""
    squared_norm = tf.reduce_sum(tf.square(vectors), axis=axis, keepdims=True)
    scale = squared_norm / (1.0 + squared_norm) / tf.sqrt(squared_norm + epsilon)
    return scale * vectors


class PrimaryCaps(layers.Layer):
    """First capsule layer: a Conv2D projects a spatial feature map into
    (num_capsules * capsule_dim) channels, then reshapes each spatial location's
    channel group into one capsule vector, and squashes it."""

    def __init__(self, num_capsules: int, capsule_dim: int, kernel_size: int = 3,
                 strides: int = 2, **kwargs):
        super().__init__(**kwargs)
        self.num_capsules = num_capsules
        self.capsule_dim = capsule_dim
        self.conv = layers.Conv2D(
            num_capsules * capsule_dim, kernel_size, strides=strides,
            padding="same", activation=None, name="primary_caps_conv",
        )

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        """inputs: (batch, H, W, C) spatial feature map.
        Returns: (batch, n_primary_caps, capsule_dim) - flattened over space and
        the num_capsules groups, each a squashed vector."""
        x = self.conv(inputs)  # (batch, h, w, num_capsules * capsule_dim)
        batch = tf.shape(x)[0]
        h, w = x.shape[1], x.shape[2]
        n_primary_caps = h * w * self.num_capsules
        x = tf.reshape(x, (batch, n_primary_caps, self.capsule_dim))
        return squash(x, axis=-1)


class RoutingCaps(layers.Layer):
    """Routing-by-agreement between a PrimaryCaps layer and this layer's output
    capsules (one per class). Each input capsule casts a weighted "vote" for each
    output capsule (via a learned transform W); routing coefficients are refined
    over `n_routing` iterations based on how well each vote agrees with the
    current output capsule (dot-product agreement), per Sabour et al.'s
    Algorithm 1 - this is what "routing-by-agreement" in the architecture
    diagram refers to, not a fixed pooling/summation."""

    def __init__(self, num_output_capsules: int, output_capsule_dim: int,
                 n_routing: int = 3, **kwargs):
        super().__init__(**kwargs)
        self.num_output_capsules = num_output_capsules
        self.output_capsule_dim = output_capsule_dim
        self.n_routing = n_routing

    def build(self, input_shape):
        n_input_caps, input_capsule_dim = input_shape[1], input_shape[2]
        # W: one learned transform matrix per (input capsule, output capsule) pair
        self.W = self.add_weight(
            name="routing_weights",
            shape=(n_input_caps, self.num_output_capsules, self.output_capsule_dim, input_capsule_dim),
            initializer="glorot_uniform",
            trainable=True,
        )

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        """inputs: (batch, n_input_caps, input_capsule_dim)
        Returns: (batch, num_output_capsules, output_capsule_dim)"""
        batch = tf.shape(inputs)[0]
        n_input_caps = inputs.shape[1]

        # votes[b,i,j,:] = W[i,j] @ inputs[b,i,:]  -> (batch, n_in, n_out, out_dim).
        # einsum broadcasts the batch dim automatically (W has none) - no manual
        # tiling needed, which is exactly where the shape bug above came from.
        votes = tf.einsum("ijod,bid->bijo", self.W, inputs)

        routing_logits = tf.zeros((batch, n_input_caps, self.num_output_capsules))
        votes_stopped = tf.stop_gradient(votes)

        outputs = None
        for i in range(self.n_routing):
            routing_weights = tf.nn.softmax(routing_logits, axis=2)  # agreement coefficients c_ij, sum over j=1
            routing_weights = routing_weights[..., tf.newaxis]        # (b, n_in, n_out, 1)

            use_votes = votes if i == self.n_routing - 1 else votes_stopped
            weighted_sum = tf.reduce_sum(routing_weights * use_votes, axis=1)  # (b, n_out, out_dim)
            outputs = squash(weighted_sum, axis=-1)

            if i < self.n_routing - 1:
                agreement = tf.reduce_sum(votes_stopped * outputs[:, tf.newaxis, :, :], axis=-1)  # (b, n_in, n_out)
                routing_logits = routing_logits + agreement

        return outputs


def margin_loss(labels: tf.Tensor, lengths: tf.Tensor, m_plus: float = 0.9,
                m_minus: float = 0.1, down_weight: float = 0.5) -> tf.Tensor:
    """The margin loss from Sabour et al. 2017, eq. 4 - the loss capsule lengths were designed for.
    labels: (batch,) integer class ids; lengths: (batch, n_classes) in [0, 1).

    Why not softmax cross-entropy on the lengths: lengths are bounded in [0, 1), so a softmax over
    them can never exceed ~e/(e+3) = 0.475 for one of 4 classes - the loss floors near 0.74 and the
    gradient signal is weak (measured locally: predictions collapsed to one class). The margin loss
    instead pushes the true class's length above m_plus and every other class's below m_minus."""
    targets = tf.one_hot(labels, depth=lengths.shape[-1], dtype=lengths.dtype)
    present = targets * tf.square(tf.nn.relu(m_plus - lengths))
    absent = down_weight * (1.0 - targets) * tf.square(tf.nn.relu(lengths - m_minus))
    return tf.reduce_mean(tf.reduce_sum(present + absent, axis=-1))


def capsule_lengths(output_capsules: tf.Tensor, epsilon: float = 1e-7) -> tf.Tensor:
    """(batch, n_classes, dim) -> (batch, n_classes) vector lengths, read as
    per-class confidence. Softmax over these gives a proper severity probability
    distribution for use with standard categorical cross-entropy."""
    return tf.sqrt(tf.reduce_sum(tf.square(output_capsules), axis=-1) + epsilon)


In [ ]:
%%writefile model.py
"""
CNN -> LSTM encoder-decoder for spatiotemporal flood depth forecasting.

Input: (batch, T_in, H, W, C=3) - depth[t], static DEM, rainfall[t] per historical timestep (rainfall = intensity in mm/h of the interval containing t: partly future for the last frame, the 'hybrid' input).
Output: (batch, T_out, H, W) depth regression only.

This is the depth-forecasting branch only, per the current (Capsule Network + Symbolic
Rule Layer) architecture - the per-pixel dry/flooded classifier this module used to carry
predates that pivot and has been removed. Severity classification now lives entirely in
the separate Capsule Network branch (build_capsule_head, added in Stage 3), which reads
the same shared CNN-encoder features but is trained/evaluated independently of this module.

Architecture follows the dossier's Section 4.5 illustrative design (time-distributed CNN
encoder -> LSTM temporal encoder-decoder -> CNN decoder), itself following the
time-distributed-CNN-then-LSTM pattern from Malik et al. 2024 and the CNN/LSTM-fused-at-
bottleneck pattern from Wang et al. 2026 (see research dossier, Section 2.B).

CO mapping:
  CO1 - dense_bottleneck: Dense layer projecting CNN features to the LSTM's input vector
  CO2 - severity classification head: now the Capsule Network (see Stage 3), not this module
  CO3 - cnn_encoder_*/cnn_decoder_*: convolutional spatial feature extraction/reconstruction
  CO4 - dropout/optimizer/learning_rate exposed as build_models() parameters
  CO5 - encoder_lstm/decoder_lstm: the temporal encoder-decoder generating the forecast sequence
        (round 2: in the trained ORIGINAL model (v1) the depth history was not exploited - see PROJECT_LOG 'Completion, round 2'; the final ConvLSTM is model_v2.py)
"""

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from capsule_layers import PrimaryCaps, RoutingCaps, capsule_lengths

PATCH_SIZE = 128  # H = W = 128, per dossier Section 4.5
N_CHANNELS = 3    # depth, DEM, rainfall
NUM_SEVERITY_CLASSES = 4  # none / minor / moderate / severe, see real_data_loader.py

# Fixed (non-learned) scales so raw physical inputs feed the CNN sanely. Without these,
# DEM values (tens to hundreds of metres) swamp rainfall (0-9 mm/h) and depth (0-18 m).
DEPTH_SCALE_M = 5.0     # input depth is divided by this; the decoder multiplies back to metres
DEM_SCALE_M = 50.0      # DEM is made patch-relative (minus its own mean) then divided by this
RAIN_SCALE_MM = 5.0     # in mm/h (the rainfall files hold intensities)
DELTA_SCALE_M = 0.05    # the decoder's unit of "change from the last observed frame" is 5 cm


class InputNormalizer(layers.Layer):
    """Fixed per-channel scaling applied INSIDE the model, so every caller (training,
    predict_severity, Grad-CAM) keeps passing raw physical units and getting metres back.

    The DEM channel is patch-relative (mean removed) rather than absolute elevation: absolute
    elevation is a region-specific fingerprint that can't transfer to a new region, while
    relative relief (slopes, valleys) can - which matters for the cross-regional test."""

    def call(self, x):  # x: (batch, T, H, W, 3) raw units
        depth = x[..., 0:1] / DEPTH_SCALE_M
        dem = x[..., 1:2]
        dem_rel = (dem - tf.reduce_mean(dem, axis=(2, 3), keepdims=True)) / DEM_SCALE_M
        rain = x[..., 2:3] / RAIN_SCALE_MM
        return tf.concat([depth, dem_rel, rain], axis=-1)

OPTIMIZERS = {
    "adam": keras.optimizers.Adam,
    "rmsprop": keras.optimizers.RMSprop,
    "sgd": keras.optimizers.SGD,
}


def build_encoder(hidden_dim: int = 128, dropout: float = 0.3) -> keras.Model:
    """Shared CNN encoder + LSTM temporal encoder. Outputs both the LSTM's final
    states (for the depth-forecasting decoder) AND the raw per-timestep CNN
    feature maps, BEFORE they're flattened into the LSTM - this second output is
    what the Capsule Network (Stage 3) consumes, so both branches genuinely share
    one encoder and both receive real gradients through it, rather than the
    Capsule Network operating on a separately re-computed copy."""
    inputs = keras.Input(shape=(None, PATCH_SIZE, PATCH_SIZE, N_CHANNELS), name="input_sequence")

    x = InputNormalizer(name="input_normalizer")(inputs)
    x = layers.TimeDistributed(layers.Conv2D(16, 3, strides=2, padding="same", activation="relu"), name="cnn_encoder_1")(x)
    x = layers.TimeDistributed(layers.Conv2D(32, 3, strides=2, padding="same", activation="relu"), name="cnn_encoder_2")(x)
    cnn_features = layers.TimeDistributed(layers.Conv2D(64, 3, strides=2, padding="same", activation="relu"), name="cnn_encoder_3")(x)
    # 128x128 -> 16x16x64 after three stride-2 blocks; cnn_features: (batch, T, 16, 16, 64)

    x = layers.TimeDistributed(layers.Flatten())(cnn_features)
    x = layers.TimeDistributed(layers.Dropout(dropout))(x)
    x = layers.TimeDistributed(layers.Dense(hidden_dim, activation="relu"), name="dense_bottleneck")(x)  # CO1

    _outputs, state_h, state_c = layers.LSTM(
        hidden_dim, return_sequences=True, return_state=True, dropout=dropout, name="encoder_lstm"
    )(x)

    return keras.Model(inputs, [state_h, state_c, cnn_features], name="encoder")


def build_decoder_step(hidden_dim: int = 128, dropout: float = 0.3) -> keras.Model:
    """One decode step. The LSTM takes a constant learned "step" input and relies entirely
    on its (state_h, state_c) carried from the previous step to produce a different output
    each time - standard for this kind of generative decoder, and much cheaper than
    re-encoding the full previous frame through a dense layer just to feed it back in.

    PERSISTENCE-ANCHORED (residual): the output is the most recent OBSERVED depth frame plus
    a learned change, not a depth field rebuilt from scratch. The first version rebuilt the
    whole 128x128 field from a 128-number LSTM state with no link to the observed frame, so
    it could never approach a near-exact copy - and a copy of the last frame is already
    accurate to ~2 mm at a 5-minute step, ~13 mm at 30 minutes (measured on real data).
    That, not only "too little data", is why the early predictions were blurry. The final
    conv is zero-initialized, so training starts exactly at the persistence forecast and
    can only improve on it."""
    step_input = keras.Input(shape=(1,), name="step_input")  # constant scalar, e.g. tf.ones((1,1))
    state_h_in = keras.Input(shape=(hidden_dim,), name="state_h")
    state_c_in = keras.Input(shape=(hidden_dim,), name="state_c")
    anchor_in = keras.Input(shape=(PATCH_SIZE, PATCH_SIZE, 1), name="anchor_depth")  # last observed depth, metres
    skip_in = keras.Input(shape=(16, 16, 64), name="skip_features")  # encoder's last-step spatial features

    step_emb = layers.Dense(hidden_dim, name="step_embedding")(step_input)
    step_emb_seq = layers.Reshape((1, hidden_dim))(step_emb)
    lstm_out, state_h, state_c = layers.LSTM(hidden_dim, return_state=True, name="decoder_lstm")(
        step_emb_seq, initial_state=[state_h_in, state_c_in]
    )
    lstm_out = layers.Dropout(dropout)(lstm_out)

    # project back to a 16x16x64 feature map, then upsample with transposed convs (mirrors the encoder)
    x = layers.Dense(16 * 16 * 64, activation="relu", name="decoder_bottleneck")(lstm_out)
    x = layers.Reshape((16, 16, 64))(x)
    # Skip connection: the LSTM state alone is a 128-number summary, so the decoder can express a
    # global drift but has no way to say WHERE anything changes. Concatenating the encoder's own
    # last-step spatial features gives every 16x16 cell access to the local observed structure.
    # (Measured: without this, held-out skill plateaued at exactly +2% from epoch 2 onward, and
    # was the same +2% on a region never seen in training - a constant drift, not learning.)
    x = layers.Concatenate(name="context_plus_skip")([x, skip_in])
    x = layers.Conv2DTranspose(32, 3, strides=2, padding="same", activation="relu", name="cnn_decoder_1")(x)
    x = layers.Conv2DTranspose(16, 3, strides=2, padding="same", activation="relu", name="cnn_decoder_2")(x)
    x = layers.Conv2DTranspose(16, 3, strides=2, padding="same", activation="relu", name="cnn_decoder_3")(x)
    # 16x16 -> 32x32 -> 64x64 -> 128x128

    delta = layers.Conv2D(1, 1, activation="linear", kernel_initializer="zeros", name="depth_delta")(x)
    delta = layers.Rescaling(DELTA_SCALE_M, name="delta_to_meters")(delta)
    depth_out = layers.Add(name="anchor_plus_delta")([anchor_in, delta])               # (batch,128,128,1) metres

    return keras.Model(
        [step_input, state_h_in, state_c_in, anchor_in, skip_in],
        [depth_out, state_h, state_c],
        name="decoder_step",
    )


def anchor_from_input(x):
    """Most recent observed depth frame from a raw (batch, T, H, W, 3) input -> (batch, H, W, 1)."""
    return x[:, -1, :, :, 0:1]


def decode_horizon(decoder, enc_out, x, horizon: int, training: bool = False):
    """Runs the residual decoder for `horizon` steps. Returns (batch, horizon, H, W) depth in metres.

    enc_out: the encoder's outputs [state_h, state_c, cnn_features]; x: the SAME raw input batch
    the encoder was run on (its last depth frame is the persistence anchor). Every step is
    anchored to that frame and sees the encoder's last-step spatial features; the LSTM state
    evolves the change."""
    state_h, state_c, cnn_features = enc_out
    anchor_depth = anchor_from_input(x)
    skip = cnn_features[:, -1]
    step_input = tf.ones((tf.shape(state_h)[0], 1))
    frames = []
    sh, sc = state_h, state_c
    for _ in range(horizon):
        depth_out, sh, sc = decoder([step_input, sh, sc, anchor_depth, skip], training=training)
        frames.append(depth_out[..., 0])
    return tf.stack(frames, axis=1)


def build_capsule_head(num_primary_capsules: int = 8, primary_capsule_dim: int = 8,
                        severity_capsule_dim: int = 16, n_routing: int = 3,
                        return_lengths: bool = False) -> keras.Model:
    """Severity-classification branch (Stage 3): PrimaryCaps -> routing-by-agreement
    -> one capsule per severity class. Consumes the LAST timestep of the SAME
    cnn_features the encoder outputs - i.e. the shared encoder's own most-recent
    observed spatial representation, not a separately-computed copy - so this
    branch's gradients flow back into the real shared encoder weights.

    return_lengths=False (default): output softmax(lengths), for use with cross-entropy.
    return_lengths=True: output the raw capsule lengths in [0, 1) for use with
    capsule_layers.margin_loss (the loss the paper designed lengths for). argmax is the
    prediction either way."""
    cnn_feat_input = keras.Input(shape=(16, 16, 64), name="cnn_features_last_step")
    primary = PrimaryCaps(num_primary_capsules, primary_capsule_dim, kernel_size=3, strides=2)(cnn_feat_input)
    severity_caps = RoutingCaps(NUM_SEVERITY_CLASSES, severity_capsule_dim, n_routing=n_routing)(primary)
    lengths = layers.Lambda(capsule_lengths, name="capsule_lengths")(severity_caps)
    if return_lengths:
        return keras.Model(cnn_feat_input, lengths, name="capsule_severity_head")
    probs = layers.Softmax(name="severity_probs")(lengths)
    return keras.Model(cnn_feat_input, probs, name="capsule_severity_head")


def build_models(hidden_dim: int = 128, dropout: float = 0.3):
    return build_encoder(hidden_dim, dropout), build_decoder_step(hidden_dim, dropout)


def forecast(encoder, decoder, input_sequence: np.ndarray, horizon: int, hidden_dim: int = 128):
    """Real forward pass: encode T_in history, then decode `horizon` future frames."""
    x = tf.constant(input_sequence[np.newaxis, ...].astype("float32"))  # (1, T_in, H, W, C)
    enc_out = encoder(x, training=False)

    shapes = {"input": tuple(x.shape), "encoder_state_h": tuple(enc_out[0].shape)}
    frames = decode_horizon(decoder, enc_out, x, horizon, training=False)
    shapes["decoded_frames"] = tuple(frames.shape)
    return frames.numpy()[0], shapes


def dummy_training_step(encoder, decoder, input_sequence: np.ndarray, horizon: int,
                         optimizer_name: str = "adam", learning_rate: float = 1e-3, hidden_dim: int = 128):
    """One real gradient step against random fake targets - proves the loss/optimizer/dropout
    wiring (CO4) is mechanically correct. Not real training; the targets are random."""
    optimizer = OPTIMIZERS[optimizer_name](learning_rate=learning_rate)
    x = tf.constant(input_sequence[np.newaxis, ...].astype("float32"))
    rng = np.random.default_rng(0)
    fake_depth_targets = tf.constant(rng.random((horizon, PATCH_SIZE, PATCH_SIZE)).astype("float32"))

    trainable_vars = encoder.trainable_variables + decoder.trainable_variables

    with tf.GradientTape() as tape:
        enc_out = encoder(x, training=True)
        frames = decode_horizon(decoder, enc_out, x, horizon, training=True)
        loss = tf.reduce_mean(tf.square(frames[0] - fake_depth_targets))

    grads = tape.gradient(loss, trainable_vars)
    n_none = sum(1 for g in grads if g is None)
    optimizer.apply_gradients((g, v) for g, v in zip(grads, trainable_vars) if g is not None)

    return float(loss.numpy()), n_none, len(trainable_vars)


In [ ]:
%%writefile model_v2.py
"""
CNN -> ConvLSTM encoder-decoder (architecture v2) for spatiotemporal flood-depth forecasting.

WHY v2 EXISTS.  A per-pixel LINEAR model that sees only the depth history (last frame, last-minus-previous, last-minus-first) scores
skill +0.33 .. +0.36 against persistence on every experiment, about DOUBLE the v1 CNN-LSTM in model.py (+0.05 .. +0.23 on the same tasks,
+0.18 on the never-seen UK at 60 m).  The depth history is very informative, and v1 could not use it, for structural reasons:
  * v1 flattens each frame's 16x16x64 features into ONE dense layer (128 numbers) before the LSTM: no per-pixel temporal difference can
    survive that bottleneck (and depth was scaled by 5 m, so millimetre-scale changes were ~1e-4 in network units);
  * its decoder's only spatial information is the LAST frame's features (skip), never a difference between frames.
v2 keeps what makes it a CNN-LSTM but removes both obstacles:
  * the recurrence is a ConvLSTM2D over the 16x16x64 CNN feature maps (spatial structure preserved; the temporal state is a MAP, not a vector);
  * multi-scale TEMPORAL skips: the 64x64 and 32x32 features of ALL input steps are stacked along channels and convolved, so the decoder can
    form frame-to-frame differences at fine resolution;
  * an explicit frame-to-frame depth-difference channel (in units of 1 cm) is added to the input;
  * the decoder is a ConvLSTM that continues the encoder state for `horizon` steps; a shared upsampling head (with the skips) turns each
    step's state into a change map; output = last observed depth + change (persistence-anchored, zero-initialised final conv, exactly as v1).
The capsule severity branch still reads the last step's 16x16x64 CNN features (`cnn_features`), so the two-branch design is unchanged.
"""
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

PATCH_SIZE = 128
DEPTH_SCALE_M = 5.0
DEM_SCALE_M = 50.0
RAIN_SCALE_MM = 5.0
DIFF_SCALE_M = 0.01     # frame-to-frame depth difference in units of 1 cm
DELTA_SCALE_M = 0.05

OPTIMIZERS = {"adam": keras.optimizers.Adam}


class InputNormalizerV2(layers.Layer):
    """(B, T, H, W, 3) raw [depth m, DEM m, rain mm/h] -> (B, T, H, W, 4): depth/5, patch-relative DEM/50, rain/5, depth difference/1 cm."""

    def call(self, x):
        depth, dem, rain = x[..., 0:1], x[..., 1:2], x[..., 2:3]
        dem_rel = (dem - tf.reduce_mean(dem, axis=(2, 3), keepdims=True)) / DEM_SCALE_M
        prev = tf.concat([depth[:, :1], depth[:, :-1]], axis=1)
        return tf.concat([depth / DEPTH_SCALE_M, dem_rel, rain / RAIN_SCALE_MM, (depth - prev) / DIFF_SCALE_M], axis=-1)


class StackTime(layers.Layer):
    """(B, T, H, W, C) -> (B, H, W, T*C): all time steps side by side along channels (a fixed T)."""

    def call(self, x):
        s = tf.shape(x)
        x = tf.transpose(x, [0, 2, 3, 1, 4])
        return tf.reshape(x, [s[0], s[2], s[3], x.shape[3] * x.shape[4]])


class LinFeatures(layers.Layer):
    """Full-resolution local features for the (optional) local-CNN path, built from the NORMALISED input (B, T, H, W, 4):
    last observed depth, patch-relative DEM, last one-step depth change, total change over the history, wet indicator and the
    rainfall channel of every input step.  -> (B, H, W, T + 5)."""

    def __init__(self, last_obs, **kw):
        super().__init__(**kw)
        self.last_obs = last_obs

    def call(self, x):
        d_last = x[:, self.last_obs, ..., 0:1]
        dem = x[:, self.last_obs, ..., 1:2]
        diff1 = x[:, self.last_obs, ..., 3:4]
        trend = tf.reduce_sum(x[..., 3], axis=1)[..., None]
        wet = tf.cast(d_last * DEPTH_SCALE_M > 0.01, x.dtype)
        rain = tf.transpose(x[..., 2], [0, 2, 3, 1])                  # (B, H, W, T)
        return tf.concat([d_last, dem, diff1, trend, wet, rain], axis=-1)

    def get_config(self):
        return {**super().get_config(), "last_obs": self.last_obs}


class ZerosSeq(layers.Layer):
    """(B, h, w, c) -> zeros (B, horizon, h, w, 1): the constant 'step input' of the decoder ConvLSTM (its state carries the information)."""

    def __init__(self, horizon, **kw):
        super().__init__(**kw)
        self.horizon = horizon

    def call(self, x):
        s = tf.shape(x)
        return tf.zeros([s[0], self.horizon, x.shape[1], x.shape[2], 1], x.dtype)

    def get_config(self):
        return {**super().get_config(), "horizon": self.horizon}


class TileTime(layers.Layer):
    """(B, h, w, c) -> (B, horizon, h, w, c): the same static skip map for every forecast step."""

    def __init__(self, horizon, **kw):
        super().__init__(**kw)
        self.horizon = horizon

    def call(self, x):
        return tf.repeat(x[:, None], self.horizon, axis=1)

    def get_config(self):
        return {**super().get_config(), "horizon": self.horizon}


def build_encoder(T, hidden_dim=64, dropout=0.3, local_path=False, last_obs=4):
    """Outputs [state_h (16,16,hidden), state_c, cnn_features (T,16,16,64), skip64 (64,64,32), skip32 (32,32,32)]."""
    inputs = keras.Input(shape=(T, PATCH_SIZE, PATCH_SIZE, 3), name="input_sequence")
    x = InputNormalizerV2(name="input_normalizer")(inputs)
    c1 = layers.TimeDistributed(layers.Conv2D(16, 3, strides=2, padding="same", activation="relu"), name="cnn_encoder_1")(x)      # 64x64
    c2 = layers.TimeDistributed(layers.Conv2D(32, 3, strides=2, padding="same", activation="relu"), name="cnn_encoder_2")(c1)     # 32x32
    c3 = layers.TimeDistributed(layers.Conv2D(64, 3, strides=2, padding="same", activation="relu"), name="cnn_encoder_3")(c2)     # 16x16
    _, h, c = layers.ConvLSTM2D(hidden_dim, 3, padding="same", return_sequences=True, return_state=True, dropout=dropout,
                                name="encoder_convlstm")(c3)
    s1 = layers.Conv2D(32, 3, padding="same", activation="relu", name="temporal_skip_64")(StackTime(name="stack64")(c1))
    s2 = layers.Conv2D(32, 3, padding="same", activation="relu", name="temporal_skip_32")(StackTime(name="stack32")(c2))
    outs = [h, c, c3, s1, s2]
    if local_path:
        outs.append(LinFeatures(last_obs, name="local_features")(x))
    return keras.Model(inputs, outs, name="encoder_v2")


def build_decoder(horizon, hidden_dim=64, dropout=0.3, local_channels=0):
    h_in = keras.Input(shape=(16, 16, hidden_dim), name="state_h")
    c_in = keras.Input(shape=(16, 16, hidden_dim), name="state_c")
    skip16 = keras.Input(shape=(16, 16, 64), name="skip16")
    skip32 = keras.Input(shape=(32, 32, 32), name="skip32")
    skip64 = keras.Input(shape=(64, 64, 32), name="skip64")
    lin_in = keras.Input(shape=(PATCH_SIZE, PATCH_SIZE, local_channels), name="local_features") if local_channels else None
    anchor = keras.Input(shape=(PATCH_SIZE, PATCH_SIZE, 1), name="anchor_depth")

    steps = layers.ConvLSTM2D(hidden_dim, 3, padding="same", return_sequences=True, name="decoder_convlstm")(
        ZerosSeq(horizon, name="step_input")(h_in), initial_state=[h_in, c_in])                      # (B, H, 16, 16, hidden)
    x = layers.Concatenate(name="context_plus_skip16")([steps, TileTime(horizon, name="tile16")(skip16)])
    x = layers.Dropout(dropout)(x)
    x = layers.TimeDistributed(layers.Conv2DTranspose(64, 3, strides=2, padding="same", activation="relu"), name="cnn_decoder_1")(x)   # 32
    x = layers.Concatenate(name="skip32_cat")([x, TileTime(horizon, name="tile32")(skip32)])
    x = layers.TimeDistributed(layers.Conv2DTranspose(32, 3, strides=2, padding="same", activation="relu"), name="cnn_decoder_2")(x)   # 64
    x = layers.Concatenate(name="skip64_cat")([x, TileTime(horizon, name="tile64")(skip64)])
    x = layers.TimeDistributed(layers.Conv2DTranspose(16, 3, strides=2, padding="same", activation="relu"), name="cnn_decoder_3")(x)   # 128
    delta = layers.TimeDistributed(layers.Conv2D(1, 1, kernel_initializer="zeros"), name="depth_delta")(x)
    delta = layers.Rescaling(DELTA_SCALE_M, name="delta_to_meters")(delta)
    terms = [TileTime(horizon, name="tile_anchor")(anchor), delta]
    if lin_in is not None:                                        # local CNN path: full-resolution, no recurrence, one map per forecast step
        z = layers.Conv2D(16, 5, padding="same", activation="relu", name="local_conv")(lin_in)
        z = layers.Conv2D(horizon, 1, kernel_initializer="zeros", name="local_delta")(z)                  # (B, 128, 128, H)
        z = layers.Rescaling(DELTA_SCALE_M, name="local_to_meters")(z)
        terms.append(layers.Lambda(lambda t: tf.transpose(t, [0, 3, 1, 2])[..., None], name="local_time_first")(z))   # (B, H, 128, 128, 1)
    out = layers.Add(name="anchor_plus_delta")(terms)                                                        # (B, H, 128, 128, 1)
    ins = [h_in, c_in, skip16, skip32, skip64] + ([lin_in] if lin_in is not None else []) + [anchor]
    return keras.Model(ins, out, name="decoder_v2")


def decode_horizon(decoder, enc_out, x, horizon, training=False):
    """(B, horizon, 128, 128) depth in metres. The anchor is the last observed depth frame (for the forced regime the last input
    steps repeat it, so x[:, -1] is the same frame)."""
    h, c, c3, s1, s2 = enc_out[:5]
    anchor = x[:, -1, :, :, 0:1]
    ins = [h, c, c3[:, -1], s2, s1] + ([enc_out[5]] if len(enc_out) > 5 else []) + [anchor]
    y = decoder(ins, training=training)
    return y[..., 0]


# ------------------------------------------------------------------------------------------------- local-only control (no recurrence)
def build_local(T, horizon, last_obs=4, kernel=5):
    """CONTROL MODEL: only the full-resolution local CNN path (LinFeatures -> Conv5x5(16, relu) -> Conv1x1(horizon), zero-initialised),
    added to the persistence anchor.  No ConvLSTM, no encoder, no skips.  Comparing v2lin with this model shows what the ConvLSTM adds."""
    inputs = keras.Input(shape=(T, PATCH_SIZE, PATCH_SIZE, 3), name="input_sequence")
    lin = LinFeatures(last_obs, name="local_features")(InputNormalizerV2(name="input_normalizer")(inputs))
    encoder = keras.Model(inputs, [lin], name="encoder_local")
    lin_in = keras.Input(shape=(PATCH_SIZE, PATCH_SIZE, T + 5), name="local_features_in")
    anchor = keras.Input(shape=(PATCH_SIZE, PATCH_SIZE, 1), name="anchor_depth")
    z = layers.Conv2D(16, kernel, padding="same", activation="relu", name="local_conv")(lin_in)      # kernel 1 = a per-pixel network (no spatial context)
    z = layers.Conv2D(horizon, 1, kernel_initializer="zeros", name="local_delta")(z)
    z = layers.Rescaling(DELTA_SCALE_M, name="local_to_meters")(z)
    z = layers.Lambda(lambda t: tf.transpose(t, [0, 3, 1, 2])[..., None], name="local_time_first")(z)
    out = layers.Add(name="anchor_plus_delta")([layers.Lambda(lambda a: tf.repeat(a[:, None], horizon, axis=1), name="tile_anchor")(anchor), z])
    return encoder, keras.Model([lin_in, anchor], out, name="decoder_local")


def decode_horizon_local(decoder, enc_out, x, horizon, training=False):
    lin = enc_out[0] if isinstance(enc_out, (list, tuple)) else enc_out            # a one-output Keras model returns the tensor itself
    return decoder([lin, x[:, -1, :, :, 0:1]], training=training)[..., 0]


class SliceTime(layers.Layer):
    """(B, T, H, W, C) -> the first `n` time steps."""

    def __init__(self, n, **kw):
        super().__init__(**kw)
        self.n = n

    def call(self, x):
        return x[:, :self.n]

    def get_config(self):
        return {**super().get_config(), "n": self.n}


class FutureRainChannels(layers.Layer):
    """(B, T, H, W, 4) normalised input -> (B, H, W, T - start): the rain channel of the steps AFTER `start`, as side channels."""

    def __init__(self, start, **kw):
        super().__init__(**kw)
        self.start = start

    def call(self, x):
        return tf.transpose(x[:, self.start:, ..., 2], [0, 2, 3, 1])

    def get_config(self):
        return {**super().get_config(), "start": self.start}


def build_clstm(T, horizon, filters=8, kernel=3, future_steps=0):
    """SMALL FULL-RESOLUTION CNN-LSTM: a ConvLSTM2D (`filters` channels, kernel x kernel) runs over the observed input steps at the native 128x128
    resolution; a 3x3 conv + 1x1 conv turns its final hidden map into `horizon` change maps (zero-initialised), added to the persistence
    anchor.  Recurrent like a CNN-LSTM, but with few parameters and no low-resolution bottleneck, so per-pixel temporal change is visible.
    future_steps > 0 (the FORCED regime, T = 5 + future_steps): the ConvLSTM sees only the first T - future_steps steps; the rainfall of the future steps
    is concatenated to its hidden map as `future_steps` side channels (code review F4: five extra time steps handicap the recurrence)."""
    inputs = keras.Input(shape=(T, PATCH_SIZE, PATCH_SIZE, 3), name="input_sequence")
    x = InputNormalizerV2(name="input_normalizer")(inputs)
    hist = SliceTime(T - future_steps, name="observed_steps")(x) if future_steps else x
    h = layers.ConvLSTM2D(filters, kernel, padding="same", return_sequences=False, name="encoder_convlstm")(hist)          # (B, 128, 128, filters)
    if future_steps:
        h = layers.Concatenate(name="hidden_plus_future_rain")([h, FutureRainChannels(T - future_steps, name="future_rain")(x)])
    encoder = keras.Model(inputs, [h], name="encoder_clstm")
    f_in = keras.Input(shape=(PATCH_SIZE, PATCH_SIZE, filters + future_steps), name="hidden_map")
    anchor = keras.Input(shape=(PATCH_SIZE, PATCH_SIZE, 1), name="anchor_depth")
    z = layers.Conv2D(16, 3, padding="same", activation="relu", name="head_conv")(f_in)
    z = layers.Conv2D(horizon, 1, kernel_initializer="zeros", name="head_delta")(z)
    z = layers.Rescaling(DELTA_SCALE_M, name="delta_to_meters")(z)
    z = layers.Lambda(lambda t: tf.transpose(t, [0, 3, 1, 2])[..., None], name="time_first")(z)
    out = layers.Add(name="anchor_plus_delta")([layers.Lambda(lambda a: tf.repeat(a[:, None], horizon, axis=1), name="tile_anchor")(anchor), z])
    return encoder, keras.Model([f_in, anchor], out, name="decoder_clstm")


In [ ]:
%%writefile depth_experiment.py
"""Depth-forecast experiments on the COARSE parts of FloodCastBench, following the paper's own two transfer protocols:

  480 m  low-fidelity :  train on Pakistan, test on Mozambique      (the paper's low-fidelity transfer test)
   60 m  high-fidelity:  train on Australia, test on the UK          (the paper's high-fidelity transfer test)

Depth only. At 480 m and 60 m the severity label is degenerate (a 128-px patch covers 61 km / 7.7 km, so nearly every patch is
"severe"), so no severity is scored here. Same model as the 30 m experiment (persistence-anchored CNN-LSTM), same loss
(MSE / persistence MSE), same metric (skill vs persistence, = 1 - MSE_model / MSE_persistence), so results are comparable
in kind. NOT comparable to the paper's tables: those forecast 19 steps at 300 s from the true depth at t = 1 over the whole domain and report a relative error (not metres);
plain persistence under that protocol is comparable to the paper's best model (completion/f_paper_protocol.py).

Also reports, for each trained model, (a) skill by lead time and (b) what the model uses (test-time ablations), because the
30 m audit of the ORIGINAL model (v1) found its skill was driven by rainfall and not by the depth history (a limitation of v1, not of the flood; see PROJECT_LOG 'Completion, round 2').

Run:  python depth_experiment.py pakistan_mozambique        |   python depth_experiment.py australia60_uk60
"""
import json
import os
import sys
import time

import numpy as np
import rasterio
import tensorflow as tf

import model as flood_model
from model import decode_horizon
from real_data_loader import RegionData
from windows import valid_positions, start_indices_for

PATCH, T_IN, HORIZON, STRIDE = 128, 5, 5, 6          # 6 x 5 min = 30 min: rainfall's native step, same as the 30 m experiment
ROOT = "data/extracted/FloodCastBench"
FID = {30: "High-fidelity flood forecasting/30m", 60: "High-fidelity flood forecasting/60m", 480: "Low-fidelity flood forecasting/480m"}


# ------------------------------------------------------------------------------------------------- windows
def scan_wet_windows(rd, positions, starts, patch=PATCH, wet_m=0.05):
    """(row, col, start) for windows whose patch is wet (>= wet_m) at the last observed OR the last forecast frame.
    Dry windows carry no signal (persistence is exact) and would only dilute the metrics."""
    out = []
    for st in starts:
        hist, fut = rd.window_frame_indices(int(st), T_IN, HORIZON, STRIDE)
        with rasterio.open(rd.depth_files[hist[-1]]) as s:
            last = s.read(1)
        with rasterio.open(rd.depth_files[fut[-1]]) as s:
            end = s.read(1)
        for (r, c) in positions:
            sl = (slice(r, r + patch), slice(c, c + patch))
            if max(float(last[sl].max()), float(end[sl].max())) >= wet_m:
                out.append((r, c, int(st)))
    return out


def load_examples(rd, windows, patch=PATCH):
    ex = []
    for (r, c, st) in windows:
        x, y, _, _ = rd.load_patch_sequence(r, c, patch, st, T_IN, HORIZON, STRIDE)
        ex.append((x, y, np.ones((patch, patch), bool)))
    return ex


def load_padded_examples(rd, positions, starts, size, wet_m=0.05):
    """For a domain smaller than one patch in some direction (UK at 60 m is 85x137): read the largest square that fits, pad it
    to 128x128 (depth 0; terrain and rain edge-replicated) and return a mask of the real pixels, so scoring ignores the padding."""
    ex = []
    for st in starts:
        for (r, c) in positions:
            x, y, _, mx = rd.load_patch_sequence(r, c, size, int(st), T_IN, HORIZON, STRIDE)
            if max(float(x[-1, ..., 0].max()), float(y[-1].max())) < wet_m:
                continue
            p = PATCH - size
            xp = np.pad(x, ((0, 0), (0, p), (0, p), (0, 0)), mode="edge")
            xp[:, size:, :, 0] = 0.0
            xp[:, :, size:, 0] = 0.0
            yp = np.pad(y, ((0, 0), (0, p), (0, p)))
            mask = np.zeros((PATCH, PATCH), bool)
            mask[:size, :size] = True
            ex.append((xp, yp, mask))
    return ex


# ------------------------------------------------------------------------------------------------- model
class WarmupCosine(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Linear warm-up to `peak` over `warmup_steps`, then cosine decay to alpha*peak. The 60 m Australia run collapsed to an
    input-independent network within ~100 steps at a constant-start LR of 1e-3 (Adam + a few extreme windows); warm-up removes that."""

    def __init__(self, peak, total_steps, warmup_steps, alpha=0.02):
        self.peak, self.total, self.warm, self.alpha = float(peak), float(total_steps), float(max(1, warmup_steps)), float(alpha)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warm = self.peak * (step + 1.0) / self.warm
        prog = tf.clip_by_value((step - self.warm) / tf.maximum(1.0, self.total - self.warm), 0.0, 1.0)
        cos = self.peak * (self.alpha + (1.0 - self.alpha) * 0.5 * (1.0 + tf.cos(np.pi * prog)))
        return tf.where(step < self.warm, warm, cos)

    def get_config(self):
        return dict(peak=self.peak, total_steps=self.total, warmup_steps=self.warm, alpha=self.alpha)


def prediction_spread(predict, examples, n=12):
    """Std across windows of the predicted mean change (metres). ~0 means the network answers the same for every input (dead)."""
    d = [(predict(x) - x[-1, ..., 0][None]).mean() for x, _, _ in examples[:n]]
    return float(np.std(d))


def build_and_train(train, val, epochs, seed, lr=1e-3, clipnorm=1.0, warmup_steps=200, log=print):
    tf.keras.utils.set_random_seed(seed)
    encoder = flood_model.build_encoder(hidden_dim=128, dropout=0.3)
    decoder = flood_model.build_decoder_step(hidden_dim=128, dropout=0.3)
    pers_mse = max(float(np.mean([np.mean((y - x[-1, ..., 0][None]) ** 2) for x, y, _ in train])), 1e-12)
    sched = WarmupCosine(lr, epochs * len(train), warmup_steps)
    opt = flood_model.OPTIMIZERS["adam"](learning_rate=sched, clipnorm=clipnorm)
    vars_ = encoder.trainable_variables + decoder.trainable_variables
    xs = tf.TensorSpec([1, T_IN, PATCH, PATCH, 3], tf.float32)
    ys = tf.TensorSpec([HORIZON, PATCH, PATCH], tf.float32)

    @tf.function(input_signature=[xs, ys])
    def step(x, y):
        with tf.GradientTape() as tape:
            frames = decode_horizon(decoder, encoder(x, training=True), x, HORIZON, training=True)
            loss = tf.reduce_mean(tf.square(frames[0] - y)) / pers_mse
        g = tape.gradient(loss, vars_)
        opt.apply_gradients((gi, v) for gi, v in zip(g, vars_) if gi is not None)
        return loss

    @tf.function(input_signature=[xs])
    def pred(x):
        return decode_horizon(decoder, encoder(x, training=False), x, HORIZON, training=False)[0]

    def predict(x):
        return pred(tf.constant(x[None], tf.float32)).numpy()

    def val_skill():
        return skill(all_preds(predict, val))

    log(f"persistence MSE on training windows {pers_mse:.3e} m^2 (RMSE {np.sqrt(pers_mse) * 1000:.1f} mm); "
        f"epoch 0 val skill {val_skill():+.4f} (must be ~0)")
    hist = []
    for ep in range(1, epochs + 1):
        t0 = time.perf_counter()
        losses = []
        for i in np.random.permutation(len(train)):
            x, y, _ = train[i]
            losses.append(float(step(tf.constant(x[None], tf.float32), tf.constant(y, tf.float32))))
        sk = val_skill()
        hist.append((ep, float(np.mean(losses)), sk))
        spread = prediction_spread(predict, val)
        if ep >= 2 and spread < 1e-5:
            raise RuntimeError(f"DEAD NETWORK after epoch {ep}: the predicted change is identical for every validation window "
                               f"(std {spread:.2e} m). Its skill would be an input-independent constant offset, not a result.")
        log(f"epoch {ep:2d}/{epochs}  train/persist {np.mean(losses):.4f}  val skill {sk:+.4f}  ({time.perf_counter() - t0:.1f}s)")
    return predict, hist, pers_mse


# ------------------------------------------------------------------------------------------------- evaluation
def all_preds(predict, examples, transform=None):
    P, O, Q, M = [], [], [], []
    for x, y, m in examples:
        P.append(predict(x if transform is None else transform(x)))
        O.append(y)
        Q.append(np.repeat(x[-1, ..., 0][None], HORIZON, axis=0))
        M.append(np.broadcast_to(m, (HORIZON,) + m.shape))
    return np.stack(P), np.stack(O), np.stack(Q), np.stack(M)


def skill(t):
    p, o, q, m = t
    return 1.0 - np.mean(((p - o) ** 2)[m]) / np.mean(((q - o) ** 2)[m])


def metrics(p, o, m, wet):
    pp, oo = p[m], o[m]
    rmse = float(np.sqrt(np.mean((pp - oo) ** 2)))
    nse = float(1 - np.sum((oo - pp) ** 2) / np.sum((oo - oo.mean()) ** 2))
    r = float(np.corrcoef(pp, oo)[0, 1])
    pw, ow = pp > wet, oo > wet
    csi = float((pw & ow).sum() / max(1, (pw | ow).sum()))
    return dict(RMSE=rmse, NSE=nse, r=r, CSI=csi)


def report(name, predict, examples, log=print):
    p, o, q, m = all_preds(predict, examples)
    row = {"n_windows": len(examples), "skill": float(skill((p, o, q, m)))}
    row["model"] = {f"CSI@{w}": metrics(p, o, m, w) for w in (0.001, 0.1)}
    row["persistence"] = {f"CSI@{w}": metrics(q, o, m, w) for w in (0.001, 0.1)}
    lead = [float(1 - np.mean(((p[:, k] - o[:, k]) ** 2)[m[:, k]]) / np.mean(((q[:, k] - o[:, k]) ** 2)[m[:, k]])) for k in range(HORIZON)]
    pr = [float(np.sqrt(np.mean(((q[:, k] - o[:, k]) ** 2)[m[:, k]])) * 1000) for k in range(HORIZON)]
    row["lead_skill"], row["persistence_rmse_mm"] = lead, pr
    a, b = row["model"]["CSI@0.001"], row["persistence"]["CSI@0.001"]
    log(f"\n{name}: {len(examples)} windows")
    log(f"  skill vs persistence {row['skill']:+.4f}")
    log(f"  RMSE  model {a['RMSE']:.4f} m   persistence {b['RMSE']:.4f} m")
    log(f"  NSE   model {a['NSE']:.4f}     persistence {b['NSE']:.4f}      r  model {a['r']:.4f}  persistence {b['r']:.4f}")
    log(f"  CSI(0.001 m) model {a['CSI']:.4f} persistence {b['CSI']:.4f};  CSI(0.1 m) model {row['model']['CSI@0.1']['CSI']:.4f} "
        f"persistence {row['persistence']['CSI@0.1']['CSI']:.4f}")
    log("  skill by lead time (30 min ... 2.5 h): " + "  ".join(f"{v:+.3f}" for v in lead) +
        "   | persistence RMSE mm: " + " ".join(f"{v:.1f}" for v in pr))
    return row


def ablations(name, predict, examples, log=print):
    def no_rain(x):
        y = x.copy(); y[..., 2] = 0.0; return y

    def flat_dem(x):
        y = x.copy(); y[..., 1] = 0.0; return y

    def scrambled(x):
        return np.concatenate([x[:-1][::-1], x[-1:]], axis=0)

    def frozen(x):
        return np.repeat(x[-1:], x.shape[0], axis=0)

    out = {}
    log(f"\n{name}: what does the model use? (skill when one input is removed at test time)")
    for label, fn in (("none (normal)", None), ("rainfall removed", no_rain), ("terrain removed", flat_dem),
                      ("history order scrambled", scrambled), ("history frozen", frozen)):
        out[label] = float(skill(all_preds(predict, examples, fn)))
        log(f"    {label:26s} {out[label]:+.3f}")
    return out


# ------------------------------------------------------------------------------------------------- the two experiments
def experiment(kind, epochs=40, n_train=400, n_val=150, n_starts=80, seed=42, log=print):
    rng = np.random.default_rng(seed)
    if kind == "pakistan_mozambique":
        res_train, res_test, tr_name, te_name = 480, 480, "Pakistan", "Mozambique"
    elif kind == "australia60_uk60":
        res_train, res_test, tr_name, te_name = 60, 60, "Australia", "UK"
    else:
        raise SystemExit(f"unknown experiment {kind}")
    rd_tr = RegionData(root_dir=ROOT, region=tr_name, fidelity_dir=FID[res_train])
    rd_te = RegionData(root_dir=ROOT, region=te_name, fidelity_dir=FID[res_test])
    log(f"=== {tr_name} ({res_train} m, depth {rd_tr.grid_height}x{rd_tr.grid_width}) -> {te_name} ({res_test} m, depth "
        f"{rd_te.grid_height}x{rd_te.grid_width}); epochs {epochs}, seed {seed}")
    pos = valid_positions(rd_tr, PATCH, 1.0)
    perm = rng.permutation(len(pos))
    nv = max(1, int(round(len(pos) * 0.25)))
    val_pos = [pos[i] for i in perm[:nv]]
    tr_pos = [pos[i] for i in perm[nv:]]
    assert not set(val_pos) & set(tr_pos)
    log(f"{tr_name}: {len(pos)} valid {PATCH}px tiles -> {len(tr_pos)} train / {len(val_pos)} validation (split by tile)")
    starts = start_indices_for(rd_tr, n_starts, T_IN, HORIZON, STRIDE)
    tr_pool = scan_wet_windows(rd_tr, tr_pos, starts)
    va_pool = scan_wet_windows(rd_tr, val_pos, starts)
    log(f"wet windows available: train {len(tr_pool)}, validation {len(va_pool)}")
    tr_w = [tr_pool[i] for i in rng.choice(len(tr_pool), size=min(n_train, len(tr_pool)), replace=False)]
    va_w = [va_pool[i] for i in rng.choice(len(va_pool), size=min(n_val, len(va_pool)), replace=False)]
    train, val = load_examples(rd_tr, tr_w), load_examples(rd_tr, va_w)
    log(f"loaded {len(train)} training and {len(val)} validation windows")

    # test region
    te_starts = start_indices_for(rd_te, 60, T_IN, HORIZON, STRIDE)
    small = min(rd_te.grid_height, rd_te.grid_width) < PATCH
    if small:                                        # the domain is smaller than a patch in one direction: padded + masked
        size = min(rd_te.grid_height, rd_te.grid_width)
        cols = sorted({0, rd_te.grid_width - size})
        rows = sorted({0, rd_te.grid_height - size})
        test = load_padded_examples(rd_te, [(r, c) for r in rows for c in cols], te_starts, size)
        log(f"{te_name}: domain {rd_te.grid_height}x{rd_te.grid_width} < one patch -> {size}px windows padded to {PATCH}, scored on real pixels only")
    else:
        rows = sorted({0, rd_te.grid_height - PATCH})
        cols = sorted({0, rd_te.grid_width - PATCH})
        tw = scan_wet_windows(rd_te, [(r, c) for r in rows for c in cols], te_starts)
        test = load_examples(rd_te, tw)
        log(f"{te_name}: {len(rows) * len(cols)} (overlapping) {PATCH}px tiles x {len(te_starts)} times -> {len(test)} wet windows")

    predict, hist, pers = build_and_train(train, val, epochs, seed, log=log)
    out = {"experiment": kind, "seed": seed, "epochs": epochs, "n_train": len(train), "n_val": len(val), "n_test": len(test),
           "history": hist, "persistence_mse_train": pers}
    out["validation"] = report(f"{tr_name} held-out tiles", predict, val, log)
    out["test"] = report(f"{te_name} (never seen)", predict, test, log)
    out["ablation_validation"] = ablations(f"{tr_name} held-out tiles", predict, val, log)
    out["ablation_test"] = ablations(f"{te_name} (never seen)", predict, test, log)
    return out


if __name__ == "__main__":
    kind = sys.argv[1]
    epochs = int(sys.argv[2]) if len(sys.argv) > 2 else 40
    seed = int(sys.argv[3]) if len(sys.argv) > 3 else 42
    n_train = int(sys.argv[4]) if len(sys.argv) > 4 else 400
    res = experiment(kind, epochs=epochs, seed=seed, n_train=n_train, n_val=min(150, n_train // 2), n_starts=80 if n_train >= 100 else 20)
    os.makedirs("verification_outputs/multires", exist_ok=True)
    path = f"verification_outputs/multires/{kind}_seed{seed}_ep{epochs}.json"
    json.dump(res, open(path, "w"), indent=1, default=float)
    print("\nsaved", path)


In [ ]:
%%writefile symbolic_rule_layer.py
"""
Stage 5: Symbolic Rule Layer (NeuroSymbolic) - post-hoc, deterministic rule logic
applied AFTER the trained model produces its outputs. No weights, no training,
no gradients through this module - it is a plain function, which is exactly
what makes the "NeuroSymbolic" label accurate (see project explainer doc).

All four thresholds below were derived from real Australia (30m, high-fidelity)
data spanning all 4 severity classes (10 examples each, from
train_stage3_balanced.py's class-balanced scan), NOT guessed:

  depth-rate (patch-mean, m/hr):  none=0.0000  minor=0.0002  moderate=0.0028  severe=0.0035
  rainfall (patch-mean, mm/h):    none=0.000   minor=0.189   moderate=2.055   severe=4.056
  slope (% grade):                none=4.85    minor=4.53    moderate=5.79    severe=8.22
  drainage-proximity (m to       none=911.9   minor=774.4   moderate=1030.1  severe=1260.6
  nearest baseline water px):

Two of these ran counter to the naive assumption before checking real data:
higher drainage-proximity and higher slope both correlate with MORE severe
outcomes here, not less. The physical reading: severe flooding means water
reaching the floodplain, farther from the baseline channel than routine minor
flooding - and often on higher-energy, steeper terrain, not just pooling in
flat low spots. The rules below follow the real data's direction, not the
initial (wrong) assumption.
"""
from real_data_loader import SEVERITY_CLASSES

DEPTH_RATE_ESCALATE_THRESHOLD = 0.0030   # m/hr, patch-mean - near the moderate/severe boundary
DEPTH_RATE_RECEDE_THRESHOLD = -0.0001    # m/hr, patch-mean - mildly negative = receding
RAINFALL_CORROBORATE_THRESHOLD = 3.0     # mm/h, patch-mean - near the moderate/severe boundary
SLOPE_HIGH_ENERGY_THRESHOLD = 7.0        # % grade - near the moderate/severe boundary
DRAINAGE_FLOODPLAIN_THRESHOLD = 1000.0   # meters - near the moderate/severe boundary


def apply_symbolic_rules(raw_severity_idx: int, depth_rate_m_per_hr: float, rainfall_mm: float,
                          slope_pct: float, drainage_proximity_m: float):
    """Returns (reasoned_severity_idx, list_of_justification_strings).
    raw_severity_idx: the Capsule Network's own prediction (0=none..3=severe) - the
    starting point these rules adjust, not replace outright."""
    severity_idx = raw_severity_idx
    reasons = []

    if depth_rate_m_per_hr > DEPTH_RATE_ESCALATE_THRESHOLD and drainage_proximity_m > DRAINAGE_FLOODPLAIN_THRESHOLD:
        if severity_idx < len(SEVERITY_CLASSES) - 1:
            severity_idx += 1
        reasons.append(
            f"escalated: depth rising {depth_rate_m_per_hr:.4f} m/hr while {drainage_proximity_m:.0f} m "
            f"from the baseline water extent - a floodplain-inundation pattern, not just channel rise"
        )

    if depth_rate_m_per_hr < DEPTH_RATE_RECEDE_THRESHOLD:
        if severity_idx > 0:
            severity_idx -= 1
        reasons.append(f"de-escalated: depth trend is receding ({depth_rate_m_per_hr:.4f} m/hr)")

    if rainfall_mm > RAINFALL_CORROBORATE_THRESHOLD and depth_rate_m_per_hr > 0:
        reasons.append(
            f"rainfall intensity ({rainfall_mm:.2f} mm/h, current 30-minute interval) accompanies the rising depth trend"
        )

    if slope_pct > SLOPE_HIGH_ENERGY_THRESHOLD and depth_rate_m_per_hr > 0:
        reasons.append(
            f"steep terrain ({slope_pct:.1f}% grade) suggests high-energy flow, not simple pooling"
        )

    if not reasons:
        reasons.append("no rule triggered an adjustment; raw model classification retained")

    return severity_idx, reasons


In [ ]:
%%writefile predict_severity.py
"""
Stage 5b: predict_severity_with_reasoning() - the actual usable inference path,
combining the trained Capsule Network (Stage 3) with the Symbolic Rule Layer
(Stage 5) into one call. This is what a demo, or Stage 6's evaluation, calls -
not the individual pieces tested in isolation.
"""
import warnings

import numpy as np
from scipy import ndimage
import tensorflow as tf

from model import decode_horizon
from real_data_loader import RegionData, SEVERITY_CLASSES
from symbolic_rule_layer import apply_symbolic_rules

# The rule thresholds in symbolic_rule_layer.py were derived from Australia at 30m only.
RULE_CALIBRATION_PIXEL_SIZE_M = 30.0


def _compute_slope_pct(dem_patch: np.ndarray, pixel_size_m: float) -> float:
    gy, gx = np.gradient(dem_patch, pixel_size_m)
    return float(100 * np.mean(np.sqrt(gx ** 2 + gy ** 2)))


def _compute_drainage_proximity_m(landuse_patch: np.ndarray, pixel_size_m: float) -> float:
    water_mask = (landuse_patch == 1)  # verified against the real Esri/Impact Observatory legend
    if not water_mask.any():
        return float("nan")  # no baseline water anywhere in this patch - proximity undefined, not zero
    dist_px = ndimage.distance_transform_edt(~water_mask)
    return float(np.mean(dist_px) * pixel_size_m)


def predict_severity_with_reasoning(encoder, decoder, capsule_head, rd: RegionData,
                                     row: int, col: int, patch_size: int, start_idx: int,
                                     t_in: int, horizon: int, stride: int = 1):
    """Runs the real forward pass + rule layer on one real patch/time window.
    Returns a dict with the raw capsule prediction, the reasoned output, the
    justification, and the underlying feature values (for transparency/debugging).
    `stride` is the window's time stride in depth steps (6 = 30 min); depth-rate is
    computed per hour of real time, so it stays comparable across strides."""
    if rd.pixel_size_m != RULE_CALIBRATION_PIXEL_SIZE_M:
        warnings.warn(
            f"{rd.region} is at {rd.pixel_size_m:.0f}m but symbolic_rule_layer.py's thresholds were "
            f"derived at {RULE_CALIBRATION_PIXEL_SIZE_M:.0f}m (Australia) - a 128px patch covers a very "
            f"different physical area, so the rules are not calibrated for this region."
        )

    input_seq, depth_targets, true_severity, max_future_depth = rd.load_patch_sequence(
        row, col, patch_size, start_idx, t_in, horizon, stride
    )

    x_batch = tf.constant(input_seq[np.newaxis, ...], dtype=tf.float32)
    enc_out = encoder(x_batch, training=False)

    severity_scores = capsule_head(enc_out[2][:, -1], training=False)  # probs or lengths; argmax is the same
    raw_severity_idx = int(tf.argmax(severity_scores[0]))

    pred_depth_frames = decode_horizon(
        decoder, enc_out, x_batch, horizon, training=False
    ).numpy()[0]  # (horizon, H, W) - model's OWN depth prediction,
    # used for depth-rate, not the ground-truth depth_targets - this is what the reasoning layer
    # would actually have access to at real inference time (no ground truth available then)

    # Depth-rate = the model's PREDICTED change in patch-mean depth, from the last observed frame
    # to the last forecast frame, per hour of real time. (An earlier version differenced the
    # first and last FORECAST frames. With the original from-scratch decoder those frames were
    # essentially noise, so the rate that triggered the escalation rule was an artifact; with the
    # persistence-anchored decoder that difference is ~0 by construction. Measuring from the
    # observed frame uses the observation plus the model's predicted trend, which is the quantity
    # the rule thresholds were derived from - the real patch-mean change per hour.)
    last_observed_mean = float(input_seq[-1, ..., 0].mean())
    hours = horizon * stride * rd.depth_step_seconds / 3600.0
    depth_rate_m_per_hr = float((pred_depth_frames[-1].mean() - last_observed_mean) / hours)

    rainfall_mm = float(input_seq[-1, ..., 2].mean())   # mm/h; the hybrid input includes rain of the interval containing the last frame (partly not yet fallen)

    dem_patch = rd.load_dem_patch(row, col, patch_size)
    slope_pct = _compute_slope_pct(dem_patch, rd.pixel_size_m)

    landuse_patch = rd.load_landuse_patch(row, col, patch_size)
    drainage_proximity_m = _compute_drainage_proximity_m(landuse_patch, rd.pixel_size_m)

    reasoned_idx, reasons = apply_symbolic_rules(
        raw_severity_idx, depth_rate_m_per_hr, rainfall_mm, slope_pct,
        drainage_proximity_m if not np.isnan(drainage_proximity_m) else 0.0,
    )

    return {
        "true_severity": SEVERITY_CLASSES[true_severity],
        "raw_severity": SEVERITY_CLASSES[raw_severity_idx],
        "reasoned_severity": SEVERITY_CLASSES[reasoned_idx],
        "justification": "; ".join(reasons),
        "features": {
            "depth_rate_m_per_hr": depth_rate_m_per_hr,
            "rainfall_mm": rainfall_mm,
            "slope_pct": slope_pct,
            "drainage_proximity_m": drainage_proximity_m,
        },
        "pred_depth_frames": pred_depth_frames,
        "depth_targets": depth_targets,
    }


In [ ]:
%%writefile grad_cam.py
"""
Stage 4: Grad-CAM (Selvaraju et al. 2017) for the depth-forecasting branch,
adapted from classification to dense regression - there's no single "class
score" to explain, so the target is the SUM of predicted depth over the output
patch at a chosen forecast step, a scalar standing in for "how much flood did
the model predict, in total." Explains cnn_encoder_3 (the encoder's last, most
semantically-rich conv layer - standard Grad-CAM practice), at the input
sequence's LAST timestep, matching how the capsule branch also reads the
encoder's most recent observed state.

Purely post-hoc: no retraining, gradients here are for the heatmap only and are
never applied to any weights.
"""
import numpy as np
import tensorflow as tf

import model as flood_model


def grad_cam(encoder, decoder, input_sequence: np.ndarray, horizon_step: int = 0):
    """input_sequence: (T_in, H, W, 3). Returns (heatmap, net_change_m): a (H, W) heatmap in
    [0,1], upsampled from the encoder's 16x16 feature-map resolution to the full input
    resolution, and the model's predicted net change in depth (metres, summed over the patch)
    at `horizon_step`. The sign of net_change_m says which direction the map explains:
    positive = evidence for the predicted rise, negative = evidence for the predicted fall."""
    x_batch = tf.constant(input_sequence[np.newaxis, ...], dtype=tf.float32)

    with tf.GradientTape() as tape:
        enc_out = encoder(x_batch, training=False)
        cnn_features = enc_out[2]
        tape.watch(cnn_features)  # the same tensor the decoder's skip connection reads, so both paths count

        frames = flood_model.decode_horizon(decoder, enc_out, x_batch, horizon_step + 1, training=False)
        # The decoder is persistence-anchored, so the gradient of the predicted depth w.r.t. the
        # encoder features reflects what drives the model's predicted CHANGE from the last
        # observed frame (the anchor itself carries no gradient to the features).
        #
        # Direction matters. Grad-CAM's ReLU keeps only evidence that INCREASES the target, so
        # if the model predicts net drying the plain map is entirely negative and ReLU blanks
        # it (measured on a real validation window: net change -0.11 m, 0% of cells positive).
        # So the target is signed by the predicted net change - decided outside the gradient
        # path - and the map shows evidence for the direction the model actually predicts.
        predicted_depth = tf.reduce_sum(frames[0, horizon_step])
        net_change = predicted_depth - tf.reduce_sum(flood_model.anchor_from_input(x_batch))
        direction = tf.stop_gradient(tf.where(net_change >= 0, 1.0, -1.0))
        target = direction * predicted_depth

    grads = tape.gradient(target, cnn_features)  # (1, T_in, 16, 16, 64)
    last_step_grads = grads[:, -1]                # (1, 16, 16, 64) - attribute to the most recent step
    last_step_features = cnn_features[:, -1]

    channel_weights = tf.reduce_mean(last_step_grads, axis=(0, 1, 2))  # (64,) global-avg-pool, per Grad-CAM
    weighted = tf.reduce_sum(last_step_features[0] * channel_weights, axis=-1)  # (16, 16)
    heatmap = tf.nn.relu(weighted)

    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-9)  # normalize to [0,1]
    heatmap_full_res = tf.image.resize(
        heatmap[tf.newaxis, ..., tf.newaxis], (flood_model.PATCH_SIZE, flood_model.PATCH_SIZE), method="bilinear"
    )[0, ..., 0]

    return heatmap_full_res.numpy(), float(net_change)


In [ ]:
%%writefile completion/engine.py
"""Study engine for the completion work: one place that builds windows, composes model inputs for each RAINFALL REGIME, holds out
whole tiles in K folds, trains a persistence-anchored depth forecaster (architectures v1 = the original CNN-LSTM, local / local1 / local11 = small local CNNs,
clstm = a small full-resolution ConvLSTM, v2 / v2lin = heavier ConvLSTM variants; all with the same warm-up recipe as depth_experiment.py) and
scores it against persistence with per-window errors saved (for bootstrap intervals).

RAINFALL REGIMES (why this exists).  A physical test (completion/a_rain_lag.py) showed that rainfall file j is the rain that FALLS during
the 30-minute interval [1800 j, 1800 (j+1)) of the simulation (domain-mean depth gain = 0.50 x file value, correlation 1.00 at lag 0: the
files are mm/h intensities).  The original loader gave the frame at time t the file floor(t/1800), i.e. the interval that CONTAINS t - for
the last observed frame that is rain falling AFTER the observation (up to 30 minutes of it).  Three well-defined regimes:
  causal : every input frame gets the last COMPLETED interval (file floor(t/1800) - 1). Only rain that has already fallen. A strictly
           real-time forecast without rainfall forecasts.
  hybrid : the ORIGINAL setup (file floor(t/1800), the interval containing t; partly future). Kept for comparison with all earlier results.
  forced : hybrid inputs PLUS the rainfall files j+1 .. j+5 that follow the last observed frame's file as five extra input steps (the minimal complete cover of the
           2.5 h horizon; at phase 0 the last one starts at the final forecast instant) (depth channel = the last observed frame,
           i.e. "unknown future depth = persistence", DEM, rain = the file for that step). A rainfall-FORCED surrogate: the setting the
           FloodCastBench paper's FNO+ operates in (it is given the rainfall series). Rain is treated as a known forecast.

VARIANTS (training-time ablations): full | nodepthhist | norainhist | nohist | norain | noterrain

Run:   python completion/engine.py cache  <EXP>
       python completion/engine.py run    <EXP> <regime> <fold> [--variant V] [--folds 3] [--epochs 40] [--seed S] [--n-train 400] [--save-weights]
       python completion/engine.py selftest
       (more flags: --arch A --batch B --lr L --blocked --augment --blind --no-ablate --no-guard (finish the run even if the dead-network check fires; the run id gets '_noguard'))
EXP: E480 (Pakistan -> Mozambique), E60 (Australia -> UK), E30 (Australia -> UK), and the reverse directions R480, R60, R30.
"""
import argparse
import hashlib
import json
import os
import sys
import time

import numpy as np
import rasterio
from rasterio.enums import Resampling

HERE = os.path.dirname(os.path.abspath(__file__))
PROJ = os.path.dirname(HERE)
os.chdir(PROJ)
sys.path.insert(0, PROJ)

PATCH, T_IN, HORIZON, STRIDE = 128, 5, 5, 6
ROOT = "data/extracted/FloodCastBench"
FID = {30: "High-fidelity flood forecasting/30m", 60: "High-fidelity flood forecasting/60m", 480: "Low-fidelity flood forecasting/480m"}
CACHE = os.path.join("completion", "cache")
RUNS = os.path.join("completion", "results", "runs")
WET_M = 0.05
EXPS = {
    "E480": dict(train=("Pakistan", 480), test=("Mozambique", 480)),
    "E60": dict(train=("Australia", 60), test=("UK", 60)),
    "E30": dict(train=("Australia", 30), test=("UK", 30)),
    "R480": dict(train=("Mozambique", 480), test=("Pakistan", 480)),
    "R60": dict(train=("UK", 60), test=("Australia", 60)),
    "R30": dict(train=("UK", 30), test=("Australia", 30)),
}
REGIMES = ("causal", "hybrid", "forced")
VARIANTS = {
    "full": {}, "nodepthhist": dict(depth_hist="frozen"), "norainhist": dict(rain_hist="frozen"),
    "nohist": dict(depth_hist="frozen", rain_hist="frozen"), "norain": dict(rain="none"), "noterrain": dict(terrain="off"),
}


def _region(region, res):
    from real_data_loader import RegionData
    return RegionData(root_dir=ROOT, region=region, fidelity_dir=FID[res])


# ================================================================================================= region-level arrays
def region_arrays(region, res, log=print):
    """DEM (H, W) and the whole rainfall stack (n_files, H, W) at DEPTH-GRID resolution (block mean over scale x scale base pixels,
    exactly what RegionData._read_layer returns per patch). Built once and cached as .npy (memory-mapped afterwards)."""
    os.makedirs(CACHE, exist_ok=True)
    key = f"{region}_{res}"
    dem_p, rain_p = os.path.join(CACHE, f"{key}_dem.npy"), os.path.join(CACHE, f"{key}_rain.npy")
    if os.path.exists(dem_p) and os.path.exists(rain_p) and os.path.exists(rain_p + ".done"):
        return np.load(dem_p, mmap_mode="r"), np.load(rain_p, mmap_mode="r")
    rd = _region(region, res)
    H, W, s = rd.grid_height, rd.grid_width, rd.layer_scale
    dy, dx = rd.layer_offset
    win = ((dy, dy + s * H), (dx, dx + s * W))

    def read(path):
        with rasterio.open(path) as src:
            if s == 1:
                a = src.read(1, window=win)
            else:
                a = src.read(1, window=win, out_shape=(H, W), resampling=Resampling.average)
        return a.astype(np.float32)

    dem = read(rd.dem_path)
    assert (dem > -1e30).all() and np.isfinite(dem).all(), f"{key}: NoData inside the depth-covered DEM window"
    np.save(dem_p, dem)
    n = len(rd.rain_files)
    out = np.lib.format.open_memmap(rain_p, mode="w+", dtype=np.float32, shape=(n, H, W))
    t0 = time.perf_counter()
    for j, p in enumerate(rd.rain_files):
        a = read(p)
        assert (a > -1e30).all() and np.isfinite(a).all(), f"{key}: NoData inside rainfall file {j}"
        out[j] = a
        if j % 50 == 0:
            log(f"  {key} rainfall {j}/{n}  ({time.perf_counter() - t0:.0f}s)")
    out.flush()
    del out
    open(rain_p + ".done", "w").write("ok")
    return np.load(dem_p, mmap_mode="r"), np.load(rain_p, mmap_mode="r")


# ================================================================================================= window pools
def test_positions(rd):
    """Test-domain tiles: the non-overlapping grid of valid tiles for a large domain, the four CORNER tiles for a small one
    (Mozambique 151x138, UK 30 m 170x275: overlapping), and two/one padded square windows if the domain is smaller than a patch (UK 60 m)."""
    from windows import valid_positions
    H, W = rd.grid_height, rd.grid_width
    if min(H, W) < PATCH:
        size = min(H, W)
        return [(r, c) for r in sorted({0, H - size}) for c in sorted({0, W - size})], size
    if H >= 2 * PATCH and W >= 2 * PATCH:
        return valid_positions(rd, PATCH, 1.0), PATCH
    return [(r, c) for r in sorted({0, H - PATCH}) for c in sorted({0, W - PATCH})], PATCH


def build_pool(region, res, role, log=print):
    """All WET windows (patch wet at the last observed OR the last forecast frame, >= 0.05 m) for every tile x start.
    role 'train': every valid non-overlapping tile, 80 start times; role 'test': test_positions(), 60 start times."""
    from windows import valid_positions, start_indices_for
    os.makedirs(CACHE, exist_ok=True)
    key = f"pool_{region}_{res}_{role}"
    meta_p = os.path.join(CACHE, key + "_meta.npz")
    if os.path.exists(meta_p):
        return load_pool(region, res, role)
    rd = _region(region, res)
    assert all(sec == 300 * i for i, sec in enumerate(rd.depth_seconds)), "depth files are not at t = 300*index"
    if role == "train":
        positions, size = valid_positions(rd, PATCH, 1.0), PATCH
        starts = start_indices_for(rd, 80, T_IN, HORIZON, STRIDE)
    else:
        positions, size = test_positions(rd)
        starts = start_indices_for(rd, 60, T_IN, HORIZON, STRIDE)
    log(f"{key}: {len(positions)} tiles of {size}px x {len(starts)} start times")
    dh_l, df_l, meta = [], [], []
    t0 = time.perf_counter()
    for n_done, st in enumerate(starts):
        hist, fut = rd.window_frame_indices(int(st), T_IN, HORIZON, STRIDE)
        frames = {}
        for i in hist + fut:
            with rasterio.open(rd.depth_files[i]) as s:
                frames[i] = s.read(1)
        for (r, c) in positions:
            sl = (slice(r, r + size), slice(c, c + size))
            if max(float(frames[hist[-1]][sl].max()), float(frames[fut[-1]][sl].max())) < WET_M:
                continue
            dh_l.append(np.stack([frames[i][sl] for i in hist]).astype(np.float32))
            df_l.append(np.stack([frames[i][sl] for i in fut]).astype(np.float32))
            meta.append((r, c, int(st)))
        if n_done % 20 == 0:
            log(f"  {key}: start {n_done}/{len(starts)}  windows so far {len(meta)}  ({time.perf_counter() - t0:.0f}s)")
    n = len(meta)
    dh_p, df_p = os.path.join(CACHE, key + "_dh.npy"), os.path.join(CACHE, key + "_df.npy")
    dh = np.lib.format.open_memmap(dh_p, mode="w+", dtype=np.float32, shape=(n, T_IN, size, size))
    df = np.lib.format.open_memmap(df_p, mode="w+", dtype=np.float32, shape=(n, HORIZON, size, size))
    for i in range(n):
        dh[i], df[i] = dh_l[i], df_l[i]
    dh.flush(); df.flush()
    del dh, df, dh_l, df_l
    np.savez(meta_p, meta=np.array(meta, dtype=np.int64).reshape(-1, 3), size=np.array(size), n_tiles=np.array(len(positions)))
    log(f"{key}: {n} wet windows saved")
    return load_pool(region, res, role)


def load_pool(region, res, role):
    key = f"pool_{region}_{res}_{role}"
    m = np.load(os.path.join(CACHE, key + "_meta.npz"))
    dem, rain = region_arrays(region, res)
    return dict(region=region, res=res, role=role, meta=m["meta"], size=int(m["size"]),
                dh=np.load(os.path.join(CACHE, key + "_dh.npy"), mmap_mode="r"),
                df=np.load(os.path.join(CACHE, key + "_df.npy"), mmap_mode="r"), dem=dem, rain=rain)


# ================================================================================================= composing inputs
def rain_intervals(start, regime):
    """Rainfall-file indices for the input steps of a window that starts at depth index `start` (history frame s at depth index
    start + 6 s, time 1800 (start/6 + s) s -> interval start//6 + s in the ORIGINAL mapping)."""
    j = [start // 6 + s for s in range(T_IN)]
    if regime == "hybrid":
        return j
    if regime == "causal":
        return [k - 1 for k in j]
    if regime == "forced":
        return j + [j[-1] + m for m in range(1, HORIZON + 1)]
    raise ValueError(regime)


def compose(pool, i, regime, variant="full", regime_override=None, rain_shift=None):
    """One window -> (x [T,128,128,3], y [5,128,128], mask [128,128]).  x = [depth, DEM, rain] per step."""
    r, c, st = (int(v) for v in pool["meta"][i])
    size = pool["size"]
    sl = (slice(r, r + size), slice(c, c + size))
    fl = VARIANTS[variant]
    dh = np.array(pool["dh"][i])                                   # (5, size, size) observed depth
    y = np.array(pool["df"][i])
    dem = np.array(pool["dem"][sl]) if fl.get("terrain") != "off" else np.zeros((size, size), np.float32)
    idx = rain_intervals(st, regime_override or regime)
    nf = pool["rain"].shape[0]
    rain = np.zeros((len(idx), size, size), np.float32)
    rsl = sl
    if rain_shift:                                                 # test-time ablation: read the rainfall from a window shifted by whole depth cells
        Hh, Ww = pool["rain"].shape[1:]
        rr = int(np.clip(r + rain_shift[0], 0, Hh - size)); cc = int(np.clip(c + rain_shift[1], 0, Ww - size))
        rsl = (slice(rr, rr + size), slice(cc, cc + size))
    for k, j in enumerate(idx):
        if j < 0:
            continue                                               # before the simulation starts: no rain
        rain[k] = pool["rain"][min(j, nf - 1), rsl[0], rsl[1]]
    depth = dh
    if regime_override == "forced" or regime == "forced":
        depth = np.concatenate([dh, np.repeat(dh[-1:], HORIZON, axis=0)], axis=0)   # unknown future depth := last observation
    if fl.get("depth_hist") == "frozen":
        depth = depth.copy(); depth[:T_IN - 1] = depth[T_IN - 1]
    if fl.get("rain_hist") == "frozen":
        rain = rain.copy(); rain[:T_IN - 1] = rain[T_IN - 1]
    if fl.get("rain") == "none":
        rain = np.zeros_like(rain)
    T = depth.shape[0]
    x = np.stack([depth, np.broadcast_to(dem, depth.shape), rain], axis=-1).astype(np.float32)
    mask = np.ones((PATCH, PATCH), bool)
    if size < PATCH:                                               # padded window: depth 0 in the padding; DEM / rain edge-replicated
        p = PATCH - size
        x = np.pad(x, ((0, 0), (0, p), (0, p), (0, 0)), mode="edge")
        x[:, size:, :, 0] = 0.0
        x[:, :, size:, 0] = 0.0
        y = np.pad(y, ((0, 0), (0, p), (0, p)))
        mask = np.zeros((PATCH, PATCH), bool)
        mask[:size, :size] = True
    return x, y, mask


# ================================================================================================= training / scoring
def _tf():
    import tensorflow as tf
    tf.config.threading.set_intra_op_parallelism_threads(int(os.environ.get("TF_THREADS", "3")))
    tf.config.threading.set_inter_op_parallelism_threads(1)
    return tf


def build_nets(arch, T):
    import model as m1
    import model_v2 as m2
    if arch == 'v1':
        return m1.build_encoder(hidden_dim=128, dropout=0.3), m1.build_decoder_step(hidden_dim=128, dropout=0.3), m1.decode_horizon, m1.OPTIMIZERS
    if arch == 'v2':
        return m2.build_encoder(T, hidden_dim=64, dropout=0.3), m2.build_decoder(HORIZON, hidden_dim=64, dropout=0.3), m2.decode_horizon, m2.OPTIMIZERS
    if arch == 'v2lin':
        return (m2.build_encoder(T, hidden_dim=64, dropout=0.3, local_path=True), m2.build_decoder(HORIZON, hidden_dim=64, dropout=0.3, local_channels=T + 5),
                m2.decode_horizon, m2.OPTIMIZERS)
    if arch == 'clstm':
        enc, dec = m2.build_clstm(T, HORIZON)
        return enc, dec, m2.decode_horizon_local, m2.OPTIMIZERS
    if arch == 'clstmf':                                    # ConvLSTM + future rain as side channels (only differs from clstm when T > 5, i.e. forced)
        enc, dec = m2.build_clstm(T, HORIZON, future_steps=T - T_IN)
        return enc, dec, m2.decode_horizon_local, m2.OPTIMIZERS
    if arch in ('local', 'local1', 'local11'):
        enc, dec = m2.build_local(T, HORIZON, kernel={'local': 5, 'local1': 1, 'local11': 11}[arch])
        return enc, dec, m2.decode_horizon_local, m2.OPTIMIZERS
    raise ValueError(arch)


def build_and_train(pool_tr, tr_idx, va_idx, regime, variant, epochs, seed, lr=1e-3, clipnorm=1.0, warmup_steps=200, log=print, n_epoch_val=48, arch='v1', augment=False, batch=1, guard=True):
    tf = _tf()
    from depth_experiment import WarmupCosine
    tf.keras.utils.set_random_seed(seed)
    T = T_IN + (HORIZON if regime == "forced" else 0)
    encoder, decoder, decode_horizon, OPT = build_nets(arch, T)
    pers = []
    for i in tr_idx:
        pers.append(np.mean((np.array(pool_tr["df"][i]) - np.array(pool_tr["dh"][i][-1])[None]) ** 2))
    pers_mse = max(float(np.mean(pers)), 1e-12)
    sched = WarmupCosine(lr, epochs * (len(tr_idx) // batch), warmup_steps)
    opt = OPT["adam"](learning_rate=sched, clipnorm=clipnorm)
    vars_ = encoder.trainable_variables + decoder.trainable_variables
    xs = tf.TensorSpec([batch, T, PATCH, PATCH, 3], tf.float32)
    ys = tf.TensorSpec([batch, HORIZON, PATCH, PATCH], tf.float32)

    @tf.function(input_signature=[xs, ys])
    def step(x, y):
        with tf.GradientTape() as tape:
            frames = decode_horizon(decoder, encoder(x, training=True), x, HORIZON, training=True)
            loss = tf.reduce_mean(tf.square(frames - y)) / pers_mse
        g = tape.gradient(loss, vars_)
        opt.apply_gradients((gi, v) for gi, v in zip(g, vars_) if gi is not None)
        return loss

    xs1 = tf.TensorSpec([1, T, PATCH, PATCH, 3], tf.float32)

    @tf.function(input_signature=[xs1])
    def pred(x):
        return decode_horizon(decoder, encoder(x, training=False), x, HORIZON, training=False)[0]

    def predict(x):
        return pred(tf.constant(x[None], tf.float32)).numpy()

    val_small = list(va_idx[:n_epoch_val])
    val_cache = [compose(pool_tr, i, regime, variant) for i in val_small]
    guard_idx = [va_idx[k] for k in np.unique(np.linspace(0, len(va_idx) - 1, min(12, len(va_idx))).astype(int))] if len(va_idx) else []   # dead-network check on windows spread over the WHOLE held-out period (the first 48 are the earliest, near-static ones: false alarms)
    guard_cache = [compose(pool_tr, i, regime, variant) for i in guard_idx]

    def val_skill():
        sm = sp = 0.0
        for x, y, m in val_cache:
            p = predict(x)
            q = x[T_IN - 1, ..., 0][None]
            sm += float(((p - y) ** 2)[:, m].sum())
            sp += float(((q - y) ** 2)[:, m].sum())
        return 1.0 - sm / sp

    log(f"arch {arch}, regime {regime}, variant {variant}, T={T}, {sum(int(np.prod(v.shape)) for v in vars_):,} trainable params; persistence MSE on training windows {pers_mse:.3e} m^2 (RMSE {np.sqrt(pers_mse) * 1000:.1f} mm); "
        f"epoch 0 val skill {val_skill():+.4f} (must be 0)")
    hist = []
    rng = np.random.default_rng(seed)
    for ep in range(1, epochs + 1):
        t0 = time.perf_counter()
        losses = []
        perm = rng.permutation(len(tr_idx))
        for b0 in range(0, len(perm) - batch + 1, batch):
            xb, yb = [], []
            for k in perm[b0:b0 + batch]:
                x, y, _ = compose(pool_tr, tr_idx[k], regime, variant)
                if augment:                               # one of the 8 dihedral transforms, identical for inputs and targets
                    q, fl = int(rng.integers(4)), bool(rng.integers(2))
                    x, y = np.rot90(x, q, axes=(1, 2)), np.rot90(y, q, axes=(1, 2))
                    if fl:
                        x, y = x[:, :, ::-1], y[:, :, ::-1]
                    x, y = np.ascontiguousarray(x), np.ascontiguousarray(y)
                xb.append(x); yb.append(y)
            losses.append(float(step(tf.constant(np.stack(xb), tf.float32), tf.constant(np.stack(yb), tf.float32))))
        sk = val_skill()
        hist.append((ep, float(np.mean(losses)), sk))
        if len(guard_cache) >= 2:
            D = np.stack([predict(x) - x[T_IN - 1, ..., 0][None] for x, _, _ in guard_cache])          # (<=12, 5, H, W) predicted change
            spread = float(np.std(D, axis=0).mean())        # 0 only if every window gets the identical change map
        else:
            spread = 1.0                                     # fewer than 2 check windows: the check cannot be evaluated, never abort
        if guard and ep * len(tr_idx) >= 400 and ep >= 2 and spread < 1e-5:
            raise RuntimeError(f"DEAD NETWORK after epoch {ep}: the predicted change map is identical for every validation window (mean per-pixel std {spread:.2e} m)")
        log(f"epoch {ep:2d}/{epochs}  train/persist {np.mean(losses):.4f}  val skill {sk:+.4f}  ({time.perf_counter() - t0:.1f}s)")
    return predict, (encoder, decoder), hist, pers_mse


def evaluate(predict, pool, idx, regime, variant="full", regime_override=None, transform=None, keep_windows=True, rain_shift=None):
    """Per-window sums of squared error (model / persistence, per lead) and CSI counts; nothing else is kept in memory."""
    n = len(idx)
    sse_m, sse_p, npx = np.zeros((n, HORIZON)), np.zeros((n, HORIZON)), np.zeros(n)
    csi = np.zeros((n, 2, 2, 3))                              # window, threshold (1 mm, 0.1 m), (model, persistence), (tp, fp, fn)
    for a, i in enumerate(idx):
        x, y, m = compose(pool, i, regime, variant, regime_override, rain_shift)
        xin = x if transform is None else transform(x)
        p = predict(xin)
        q = np.repeat(x[T_IN - 1, ..., 0][None], HORIZON, axis=0)
        mm = np.broadcast_to(m, y.shape)
        for k in range(HORIZON):
            sse_m[a, k] = ((p[k] - y[k]) ** 2)[m].sum()
            sse_p[a, k] = ((q[k] - y[k]) ** 2)[m].sum()
        npx[a] = m.sum()
        for ti, thr in enumerate((0.001, 0.1)):
            for si, arr in enumerate((p, q)):
                aw, ow = (arr > thr) & mm, (y > thr) & mm
                csi[a, ti, si] = ((aw & ow).sum(), (aw & ~ow).sum(), (~aw & ow).sum())
    return dict(sse_m=sse_m, sse_p=sse_p, npx=npx, csi=csi)


def summarize(ev):
    sm, sp, n = ev["sse_m"], ev["sse_p"], ev["npx"]
    tot_m, tot_p = sm.sum(), sp.sum()
    lead = [float(1 - sm[:, k].sum() / sp[:, k].sum()) for k in range(HORIZON)]
    npix = n.sum() * HORIZON
    out = dict(n_windows=int(len(n)), skill=float(1 - tot_m / tot_p), lead_skill=lead,
               rmse_model_mm=float(np.sqrt(tot_m / npix) * 1000), rmse_persistence_mm=float(np.sqrt(tot_p / npix) * 1000))
    c = ev["csi"].sum(axis=0)
    for ti, name in enumerate(("CSI_1mm", "CSI_0.1m")):
        out[name] = dict(model=float(c[ti, 0, 0] / max(1, c[ti, 0].sum())), persistence=float(c[ti, 1, 0] / max(1, c[ti, 1].sum())))
    return out


def skill_of(ev):
    return float(1 - ev["sse_m"].sum() / ev["sse_p"].sum())


TEST_ABLATIONS = ("none", "rain_removed", "terrain_removed", "depth_hist_scrambled", "depth_hist_frozen", "rain_hist_frozen", "future_rain_removed", "rain_shifted_cells", "rain_shifted_earlier")


def ablation_skills(predict, pool, idx, regime, variant):
    def rain_removed(x):
        y = x.copy(); y[..., 2] = 0.0; return y

    def terrain_removed(x):
        y = x.copy(); y[..., 1] = 0.0; return y

    def scr(x):
        y = x.copy(); y[:T_IN - 1, ..., 0] = x[:T_IN - 1, ..., 0][::-1]; return y

    def frz(x):
        y = x.copy(); y[:T_IN - 1, ..., 0] = x[T_IN - 1, ..., 0][None]; return y

    def rfrz(x):
        y = x.copy(); y[:T_IN - 1, ..., 2] = x[T_IN - 1, ..., 2][None]; return y

    def fut_rain_removed(x):
        y = x.copy(); y[T_IN:, ..., 2] = 0.0; return y

    out = {}
    out["none"] = skill_of(evaluate(predict, pool, idx, regime, variant))
    for name, fn in (("rain_removed", rain_removed), ("terrain_removed", terrain_removed), ("depth_hist_scrambled", scr),
                     ("depth_hist_frozen", frz), ("rain_hist_frozen", rfrz)):
        out[name] = skill_of(evaluate(predict, pool, idx, regime, variant, transform=fn))
    if regime == "forced":
        out["future_rain_removed"] = skill_of(evaluate(predict, pool, idx, regime, variant, transform=fut_rain_removed))
    cells = 1 if pool["res"] == 480 else 2                     # 480 m: one cell = 8x the +-2 base-pixel geometry uncertainty; 30/60 m: 2 cells = 60/120 m
    out["rain_shifted_cells"] = skill_of(evaluate(predict, pool, idx, regime, variant, rain_shift=(cells, cells)))
    if regime == "hybrid":                                     # what if the input rain were strictly past?  (a model trained on hybrid, fed causal rain)
        out["rain_shifted_earlier"] = skill_of(evaluate(predict, pool, idx, regime, variant, regime_override="causal"))
    return out


def _ser(ev, pool, idx):
    meta = pool["meta"][idx]
    return dict(tile_r=meta[:, 0].tolist(), tile_c=meta[:, 1].tolist(), start=meta[:, 2].tolist(), phase=(meta[:, 2] % 6).tolist(),
                sse_m=ev["sse_m"].tolist(), sse_p=ev["sse_p"].tolist(), npx=ev["npx"].tolist(), csi=ev["csi"].tolist())


def folds_of_tiles(tiles, K, seed=0):
    perm = np.random.default_rng(seed).permutation(len(tiles))
    return [sorted(tiles[i] for i in perm[k::K]) for k in range(K)]


def split_for(exp, fold, K=3, seed=None, n_train=400, n_val=300, log=print, blocked=False):
    """Pools and window indices of one fold: train windows from the tiles NOT in the fold, held-out windows from the fold's tiles,
    every window of the never-seen test region. Deterministic in (exp, fold, K, seed)."""
    cfg = EXPS[exp]
    (tr_reg, tr_res), (te_reg, te_res) = cfg["train"], cfg["test"]
    seed = 1000 + 10 * fold + K if seed is None else seed
    tr_pool = build_pool(tr_reg, tr_res, "train", log)
    te_pool = build_pool(te_reg, te_res, "test", log)
    tiles = sorted({(int(r), int(c)) for r, c, _ in tr_pool["meta"]})
    folds = folds_of_tiles(tiles, K, seed=0)
    held = set(map(tuple, folds[fold]))
    tile_of = [(int(r), int(c)) for r, c, _ in tr_pool["meta"]]
    starts_of = tr_pool["meta"][:, 2]
    if blocked:                                            # hold out TILES and a TIME BLOCK: held-out windows are unseen in space AND in time
        blocks = np.array_split(np.array(sorted(set(starts_of.tolist()))), K)
        blk = set(blocks[fold].tolist())
        b_lo, b_hi = min(blk), max(blk)
        # EMBARGO: a window spans 54 depth steps, so also drop training windows whose span touches the held-out time block (code review F9)
        tr_all = [i for i, t in enumerate(tile_of) if t not in held and not (b_lo - 54 <= int(starts_of[i]) <= b_hi + 54)]
        va_all = [i for i, t in enumerate(tile_of) if t in held and int(starts_of[i]) in blk]
    else:
        tr_all = [i for i, t in enumerate(tile_of) if t not in held]
        va_all = [i for i, t in enumerate(tile_of) if t in held]
    rng = np.random.default_rng(seed)
    tr_idx = sorted(rng.choice(tr_all, size=min(n_train, len(tr_all)), replace=False).tolist())
    va_idx = sorted(rng.choice(va_all, size=min(n_val, len(va_all)), replace=False).tolist())
    te_idx = list(range(len(te_pool["meta"])))
    log(f"{tr_reg}: {len(tiles)} valid tiles -> fold {fold}/{K}: {len(tiles) - len(held)} train tiles, {len(held)} held-out tiles; wet windows: train pool {len(tr_all)}, "
        f"held-out pool {len(va_all)}; using {len(tr_idx)} train / {len(va_idx)} held-out / {len(te_idx)} {te_reg} windows")
    return tr_pool, te_pool, tr_idx, va_idx, te_idx, held, seed


def run(exp, regime, fold, variant="full", K=3, epochs=40, seed=None, n_train=400, n_val=300, n_test_ablate=True, save_weights=False, log=print, arch="v1", augment=False, blind=False, blocked=False, batch=1, lr=1e-3, guard=True):
    cfg = EXPS[exp]
    (tr_reg, tr_res), (te_reg, te_res) = cfg["train"], cfg["test"]
    seed = 1000 + 10 * fold + K if seed is None else seed
    rid = f"{exp}_{arch}_{regime}_{variant}{'_aug' if augment else ''}{'_blind' if blind else ''}{'_blk' if blocked else ''}{'_noguard' if not guard else ''}{'_b%d' % batch if batch > 1 else ''}{'_lr%g' % lr if lr != 1e-3 else ''}_f{fold}of{K}_s{seed}_ep{epochs}"
    os.makedirs(RUNS, exist_ok=True)
    out_p = os.path.join(RUNS, rid + ".json")
    if os.path.exists(out_p):
        log(f"{rid}: already done"); return json.load(open(out_p))
    log(f"=== {rid}: {tr_reg} {tr_res} m -> {te_reg} {te_res} m")
    tr_pool, te_pool, tr_idx, va_idx, te_idx, held, seed = split_for(exp, fold, K, seed, n_train, n_val, log, blocked=blocked)
    t0 = time.perf_counter()
    predict, nets, hist, pers = build_and_train(tr_pool, tr_idx, va_idx, regime, variant, epochs, seed, log=log, arch=arch, augment=augment, batch=batch, lr=lr, guard=guard)
    ev_va = evaluate(predict, tr_pool, va_idx, regime, variant)
    ev_te = None if blind else evaluate(predict, te_pool, te_idx, regime, variant)      # blind: the never-seen region is not touched (design phase)
    res = dict(id=rid, exp=exp, arch=arch, augment=augment, blocked=blocked, batch=batch, lr=lr, guard=guard, regime=regime, variant=variant, fold=fold, K=K, seed=seed, epochs=epochs, n_train=len(tr_idx),
               train_region=tr_reg, test_region=te_reg, held_out_tiles=sorted(held), history=hist, persistence_mse_train=pers,
               heldout=summarize(ev_va), test=None if blind else summarize(ev_te), blind=blind, train_subset=summarize(evaluate(predict, tr_pool, tr_idx[:100], regime, variant)), heldout_windows=_ser(ev_va, tr_pool, va_idx), test_windows=None if blind else _ser(ev_te, te_pool, te_idx),
               train_minutes=(time.perf_counter() - t0) / 60,
               code_md5={f: hashlib.md5(open(f, "rb").read()).hexdigest()[:10] for f in ("completion/engine.py", "model.py", "model_v2.py", "depth_experiment.py")})
    for nm in ("train_subset", "heldout", "test"):
        r = res[nm]
        if r is None:
            continue
        log(f"{nm}: skill {r['skill']:+.4f}  rmse model {r['rmse_model_mm']:.2f} mm / persistence {r['rmse_persistence_mm']:.2f} mm; "
            f"CSI(1 mm) {r['CSI_1mm']['model']:.3f}/{r['CSI_1mm']['persistence']:.3f}; lead skill " + " ".join(f"{v:+.3f}" for v in r["lead_skill"]))
    if n_test_ablate:
        res["ablation_heldout"] = ablation_skills(predict, tr_pool, va_idx, regime, variant)
        res["ablation_test"] = None if blind else ablation_skills(predict, te_pool, te_idx, regime, variant)
        for nm in ("ablation_heldout", "ablation_test"):
            if res[nm] is None:
                continue
            log(nm + ": " + "  ".join(f"{k} {v:+.3f}" for k, v in res[nm].items()))
    if save_weights:
        wd = os.path.join(CACHE, "weights"); os.makedirs(wd, exist_ok=True)
        nets[0].save_weights(os.path.join(wd, rid + "_enc.weights.h5")); nets[1].save_weights(os.path.join(wd, rid + "_dec.weights.h5"))
    json.dump(res, open(out_p, "w"), default=float)
    log("saved " + out_p)
    return res


# ================================================================================================= self-test
def selftest(log=print):
    """(1) compose('hybrid') equals RegionData.load_patch_sequence for random windows of every region/resolution (train and test pools);
    (2) causal = hybrid shifted by one interval, forced future rain equals the loader's file for those intervals;
    (3) an untrained model equals persistence exactly in every regime (zero-initialised final conv)."""
    rng = np.random.default_rng(0)
    worst = 0.0
    exps = os.environ.get("SELFTEST_EXPS", "E480,E60,E30,R480").split(",")
    for exp in exps:
        for role, (reg, res) in (("train", EXPS[exp]["train"]), ("test", EXPS[exp]["test"])):
            pool = load_pool(reg, res, role) if os.path.exists(os.path.join(CACHE, f"pool_{reg}_{res}_{role}_meta.npz")) else build_pool(reg, res, role, log)
            rd = _region(reg, res)
            for i in rng.choice(len(pool["meta"]), size=min(8, len(pool["meta"])), replace=False):
                r, c, st = (int(v) for v in pool["meta"][i])
                size = pool["size"]
                x_ref, y_ref, _, _ = rd.load_patch_sequence(r, c, size, st, T_IN, HORIZON, STRIDE)
                x, y, m = compose(pool, i, "hybrid")
                xs = x[:, :size, :size]
                d = max(float(np.abs(xs[..., 0] - x_ref[..., 0]).max()), float(np.abs(xs[..., 1] - x_ref[..., 1]).max()),
                        float(np.abs(xs[..., 2] - x_ref[..., 2]).max()), float(np.abs(y[:, :size, :size] - y_ref).max()))
                worst = max(worst, d)
                assert d < 1e-3, (exp, role, i, d)
                # causal / forced intervals against the raw rainfall files
                j = st // 6
                xc, _, _ = compose(pool, i, "causal")
                xf, _, _ = compose(pool, i, "forced")
                for s in range(T_IN):
                    raw = rd._read_layer(rd.rain_files[min(max(j + s - 1, 0), len(rd.rain_files) - 1)], r, c, size) if j + s - 1 >= 0 else np.zeros((size, size), np.float32)
                    assert np.abs(xc[s, :size, :size, 2] - raw).max() < 1e-3, ("causal", exp, role, i, s)
                assert xf.shape[0] == T_IN + HORIZON and np.array_equal(xf[:T_IN], compose(pool, i, "hybrid")[0])
                for mm in range(1, HORIZON + 1):
                    raw = rd._read_layer(rd.rain_files[min(j + T_IN - 1 + mm, len(rd.rain_files) - 1)], r, c, size)
                    assert np.abs(xf[T_IN - 1 + mm, :size, :size, 2] - raw).max() < 1e-3, ("forced", exp, role, i, mm)
                    assert np.array_equal(xf[T_IN - 1 + mm, ..., 0], xf[T_IN - 1, ..., 0])
            log(f"selftest {exp} {role} ({reg} {res} m): compose == loader / causal / forced OK (max abs diff so far {worst:.2e})")
    # untrained model == persistence
    tf = _tf()
    import model as flood_model
    from model import decode_horizon
    pool = load_pool(*EXPS[exps[0]]["train"], "train")
    for arch in ('v1', 'v2', 'v2lin', 'local', 'local1', 'local11', 'clstm', 'clstmf'):
        for regime in REGIMES:
            x, y, m = compose(pool, 5, regime)
            enc, dec, dech, _ = build_nets(arch, x.shape[0])
            f = dech(dec, enc(tf.constant(x[None]), training=False), tf.constant(x[None]), HORIZON, training=False).numpy()[0]
            assert np.abs(f - x[T_IN - 1, ..., 0][None]).max() < 1e-6, (arch, regime)
    log("selftest: untrained model equals persistence in all three regimes")
    log("SELFTEST PASSED")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["cache", "run", "selftest"])
    ap.add_argument("exp", nargs="?")
    ap.add_argument("regime", nargs="?")
    ap.add_argument("fold", nargs="?", type=int)
    ap.add_argument("--variant", default="full")
    ap.add_argument("--arch", default="v1", choices=["v1", "v2", "v2lin", "local", "local1", "local11", "clstm", "clstmf"])
    ap.add_argument("--augment", action="store_true")
    ap.add_argument("--batch", type=int, default=1)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--blocked", action="store_true", help="hold out a time block as well as the fold's tiles")
    ap.add_argument("--blind", action="store_true", help="do not evaluate the never-seen test region (design phase)")
    ap.add_argument("--folds", type=int, default=3)
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--seed", type=int, default=None)
    ap.add_argument("--n-train", type=int, default=400)
    ap.add_argument("--no-ablate", action="store_true")
    ap.add_argument("--no-guard", action="store_true", help="finish the run even if the dead-network guard fires (post hoc, labelled)")
    ap.add_argument("--save-weights", action="store_true")
    a = ap.parse_args()
    if a.cmd == "cache":
        for role, key in (("train", "train"), ("test", "test")):
            reg, res = EXPS[a.exp][key]
            build_pool(reg, res, role, log=lambda s: print(s, flush=True))
    elif a.cmd == "selftest":
        selftest(log=lambda s: print(s, flush=True))
    else:
        run(a.exp, a.regime, a.fold, variant=a.variant, K=a.folds, epochs=a.epochs, seed=a.seed, n_train=a.n_train,
            n_test_ablate=not a.no_ablate, save_weights=a.save_weights, log=lambda s: print(s, flush=True), arch=a.arch, augment=a.augment, blind=a.blind, blocked=a.blocked, batch=a.batch, lr=a.lr, guard=not a.no_guard)


In [ ]:
%%writefile completion/b_baselines.py
"""Baseline ladder (S5 two-feature lookup table, S1-S3 linear, S1c-S3c the same clipped at zero depth, S4/S4c a per-pixel MLP): how much of the neural network's skill can simple per-pixel LINEAR models reproduce, and does depth HISTORY carry
any information at all?  All models are scored on exactly the windows the neural models use (engine.split_for), against persistence.

  S0 persistence            (skill 0 by definition)
  S1 trend only             depth state + depth trend (last-minus-previous, last-minus-first) + 15x15 local means      (no rainfall, no terrain)
  S2 rain + state           depth state + wetness + terrain + rainfall features (+ wet x rain) [+ future rain in the forced regime]
  S3 rain + state + trend   S2 + the depth trend features
Each is a ridge regression of (future depth - last depth) at each of the 5 lead times on the features of the same pixel (plus a
15x15 local mean of the depth state and trend).  No spatial deep model, no LSTM: if S3 ~ the CNN-LSTM then the deep model adds little;
if S3 >> S2 then depth history is informative and a good model should use it.

Run: python completion/b_baselines.py <EXP> <regime> <fold> [--folds 3]
"""
import argparse
import json
import os
import sys

import numpy as np
from scipy.ndimage import uniform_filter

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import engine as E  # noqa: E402  (chdirs to the project root)

T_IN, HORIZON = E.T_IN, E.HORIZON


def features(x, regime, sets):
    """Feature stack (H*W, F) for one composed window, for feature-set name in `sets` (a list of names)."""
    d = x[:T_IN, ..., 0]
    rain = x[..., 2]
    dT, tr1, tr4 = d[-1], d[-1] - d[-2], d[-1] - d[0]
    dem = x[0, ..., 1]
    dem_rel = (dem - dem.mean()) / 50.0
    wet = (dT > 0.01).astype(np.float32)
    loc = lambda a: uniform_filter(a, size=15, mode="nearest")
    ones = np.ones_like(dT)
    R = [rain[T_IN - 1], rain[:T_IN].mean(0)]
    if regime == "forced":
        R += [rain[T_IN + m] for m in range(HORIZON)]
    cols = {
        "S1": [ones, dT, tr1, tr4, wet, loc(dT), loc(tr1)],
        "S2": [ones, dT, wet, dem_rel, loc(dT)] + R + [wet * r for r in R],
        "S3": [ones, dT, wet, dem_rel, loc(dT), tr1, tr4, loc(tr1)] + R + [wet * r for r in R],
        # S4 (per-pixel MLP): the S3 features plus all four one-step depth changes, in centimetres
        "S4": [dT, wet, dem_rel, loc(dT), loc(tr1)] + [100.0 * (d[k + 1] - d[k]) for k in range(T_IN - 1)] + R + [wet * r for r in R],
    }
    return {k: np.stack([c.reshape(-1) for c in cols[k]], axis=1).astype(np.float32) for k in sets}


def fit(pool, tr_idx, regime, sets, px_per_window=3000, lam=1e-3, seed=0):
    rng = np.random.default_rng(seed)
    XS = {k: [] for k in sets}
    Y = []
    for i in tr_idx:
        x, y, m = E.compose(pool, i, regime)
        F = features(x, regime, sets)
        sel = rng.choice(x.shape[1] * x.shape[2], size=px_per_window, replace=False)
        for k in sets:
            XS[k].append(F[k][sel])
        Y.append((y - x[T_IN - 1, ..., 0][None]).reshape(HORIZON, -1)[:, sel].T)
    Y = np.concatenate(Y)
    coefs = {}
    for k in sets:
        X = np.concatenate(XS[k]).astype(np.float64)
        A = X.T @ X + lam * len(X) * np.eye(X.shape[1])
        coefs[k] = np.linalg.solve(A, X.T @ Y.astype(np.float64))
    return coefs


def predictor(coef, regime, name, clip=False):
    def predict(x):
        F = features(x, regime, [name])[name]
        delta = (F @ coef).T.reshape(HORIZON, x.shape[1], x.shape[2])
        out = x[T_IN - 1, ..., 0][None] + delta
        return np.maximum(out, 0.0) if clip else out              # depth cannot be negative
    return predict


def fit_mlp(pool, tr_idx, regime, px_per_window=2000, epochs=12, seed=0):
    """S4: a small per-pixel MLP (2 x 64 ReLU) on the S4 features -> the 5 lead-time changes (in cm); NO spatial convolution, NO recurrence."""
    tf = E._tf()
    rng = np.random.default_rng(seed)
    X, Y = [], []
    for i in tr_idx:
        x, y, m = E.compose(pool, i, regime)
        F = features(x, regime, ["S4"])["S4"]
        sel = rng.choice(x.shape[1] * x.shape[2], size=px_per_window, replace=False)
        X.append(F[sel])
        Y.append((y - x[T_IN - 1, ..., 0][None]).reshape(HORIZON, -1)[:, sel].T * 100.0)
    X, Y = np.concatenate(X).astype("float32"), np.concatenate(Y).astype("float32")
    mu, sd = X.mean(0), X.std(0) + 1e-6
    tf.keras.utils.set_random_seed(seed)
    net = tf.keras.Sequential([tf.keras.layers.Input((X.shape[1],)), tf.keras.layers.Dense(64, activation="relu"), tf.keras.layers.Dense(64, activation="relu"),
                               tf.keras.layers.Dense(HORIZON)])
    net.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
    net.fit((X - mu) / sd, Y, batch_size=2048, epochs=epochs, verbose=0)
    return net, mu, sd


def mlp_predictor(net, mu, sd, regime, clip=False):
    def predict(x):
        F = features(x, regime, ["S4"])["S4"]
        delta = (net.predict((F - mu) / sd, batch_size=16384, verbose=0) / 100.0).T.reshape(HORIZON, x.shape[1], x.shape[2])
        out = x[T_IN - 1, ..., 0][None] + delta
        return np.maximum(out, 0.0) if clip else out
    return predict


D_EDGES = np.array([0.0, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 1e9])
T_EDGES = np.array([-1e9, -0.20, -0.10, -0.05, -0.02, -0.01, -0.005, -0.002, -0.0005, 0.0005, 0.002, 0.005, 0.01, 0.02, 0.05, 0.10, 0.20, 1e9])   # metres per 30 min


def _cell(d_last, tr1):
    return np.clip(np.digitize(d_last, D_EDGES) - 1, 0, len(D_EDGES) - 2), np.clip(np.digitize(tr1, T_EDGES) - 1, 0, len(T_EDGES) - 2)


def fit_table(pool, tr_idx, regime, px_per_window=3000, seed=0):
    """S5: a TWO-FEATURE LOOKUP TABLE. Mean future change (each lead) in each (current depth bin) x (last one-step change bin) cell of the training
    pixels. No learning machinery, no spatial context, no rainfall, no terrain."""
    rng = np.random.default_rng(seed)
    nd, nt = len(D_EDGES) - 1, len(T_EDGES) - 1
    S, N = np.zeros((nd, nt, HORIZON)), np.zeros((nd, nt))
    for i in tr_idx:
        x, y, m = E.compose(pool, i, regime)
        sel = rng.choice(x.shape[1] * x.shape[2], size=px_per_window, replace=False)
        d_last, tr1 = x[T_IN - 1, ..., 0].reshape(-1)[sel], (x[T_IN - 1, ..., 0] - x[T_IN - 2, ..., 0]).reshape(-1)[sel]
        delta = (y - x[T_IN - 1, ..., 0][None]).reshape(HORIZON, -1)[:, sel].T
        a, b = _cell(d_last, tr1)
        np.add.at(S, (a, b), delta)
        np.add.at(N, (a, b), 1.0)
    M = np.where(N[..., None] > 0, S / np.maximum(N[..., None], 1), np.nan)
    marg = np.array([np.nansum(S[a], axis=0) / max(1.0, N[a].sum()) for a in range(nd)])       # depth-bin marginal for empty cells
    for a in range(nd):
        for b in range(nt):
            if N[a, b] < 5:
                M[a, b] = marg[a]
    return M


def table_predictor(M):
    def predict(x):
        d_last = x[T_IN - 1, ..., 0]
        a, b = _cell(d_last, d_last - x[T_IN - 2, ..., 0])
        return d_last[None] + np.moveaxis(M[a, b], -1, 0)
    return predict


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("exp"); ap.add_argument("regime"); ap.add_argument("fold", type=int)
    ap.add_argument("--folds", type=int, default=3)
    ap.add_argument("--n-train", type=int, default=400)
    ap.add_argument("--blocked", action="store_true")
    a = ap.parse_args()
    out_p = f"completion/results/baselines_{a.exp}_{a.regime}_f{a.fold}of{a.folds}_n{a.n_train}{'_blk' if a.blocked else ''}.json"
    tr_pool, te_pool, tr_idx, va_idx, te_idx, held, seed = E.split_for(a.exp, a.fold, a.folds, n_train=a.n_train, log=lambda s: print(s, flush=True), blocked=a.blocked)
    sets = ["S1", "S2", "S3"]
    coefs = fit(tr_pool, tr_idx, a.regime, sets)
    out = {"exp": a.exp, "regime": a.regime, "fold": a.fold, "K": a.folds, "n_train": len(tr_idx), "blocked": a.blocked, "S0": {"heldout": 0.0, "test": 0.0}}
    for k in sets:
        for clip in (False, True):
            pr = predictor(coefs[k], a.regime, k, clip)
            key = k + ("c" if clip else "")
            out[key] = {"heldout": E.summarize(E.evaluate(pr, tr_pool, va_idx, a.regime))["skill"],
                        "test": E.summarize(E.evaluate(pr, te_pool, te_idx, a.regime))["skill"]}
            print(f"{a.exp} {a.regime} fold {a.fold}: {key} heldout {out[key]['heldout']:+.4f}  test {out[key]['test']:+.4f}", flush=True)
    pr = table_predictor(fit_table(tr_pool, tr_idx, a.regime))
    out["S5"] = {"heldout": E.summarize(E.evaluate(pr, tr_pool, va_idx, a.regime))["skill"], "test": E.summarize(E.evaluate(pr, te_pool, te_idx, a.regime))["skill"]}
    print(f"{a.exp} {a.regime} fold {a.fold}: S5 (2-feature lookup table) heldout {out['S5']['heldout']:+.4f}  test {out['S5']['test']:+.4f}", flush=True)
    net, mu, sd = fit_mlp(tr_pool, tr_idx, a.regime)
    for clip in (False, True):
        pr = mlp_predictor(net, mu, sd, a.regime, clip)
        key = "S4" + ("c" if clip else "")
        out[key] = {"heldout": E.summarize(E.evaluate(pr, tr_pool, va_idx, a.regime))["skill"],
                    "test": E.summarize(E.evaluate(pr, te_pool, te_idx, a.regime))["skill"]}
        print(f"{a.exp} {a.regime} fold {a.fold}: {key} heldout {out[key]['heldout']:+.4f}  test {out[key]['test']:+.4f}", flush=True)
    os.makedirs("completion/results", exist_ok=True)
    json.dump(out, open(out_p, "w"), indent=1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile completion/d_aggregate.py
"""Aggregate completion/results/runs/*.json (per-fold neural runs) and completion/results/baselines_*.json (baseline ladder) into tables.

Only runs of the FROZEN recipe (batch 1, lr 1e-3, the default seed of each fold, epochs/n_train as set by FINAL_EPOCHS / FINAL_NTRAIN) enter the headline groups;
runs with another batch size, learning rate or seed (batches OPT and F) are kept apart (`load_runs(extra=True)`) so they can never merge into a headline cell.

Uncertainty (revised after the independent code review, finding F2): neighbouring test windows overlap strongly in time (a window spans 54 depth steps, start times are
14-50 steps apart) and every never-seen region is ONE flood event, so start times are NOT independent.  Intervals therefore use BLOCK bootstraps:
  * never-seen region: start times sorted in time and cut into non-overlapping blocks of ceil(54 / median spacing) + 1 starts; blocks are resampled;
  * held-out tiles: a TWO-WAY cluster bootstrap that resamples tiles AND time blocks (weight of a window = count of its tile x count of its block);
  * paired differences (e.g. clstm - local) resample the same time blocks for both models.
Even so, each region is one event, and E60 / E30 share the same Australia and UK events: there are two independent train -> test pairs, not nine.
Usage: python completion/d_aggregate.py [--md out.md]
"""
import argparse
import collections
import glob
import json
import math
import os

import numpy as np

os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
NB = 2000
EPOCHS_F = int(os.environ.get("FINAL_EPOCHS", 30))      # the frozen recipe (override only for reduced re-runs, e.g. the Colab notebook)
NTR_F = int(os.environ.get("FINAL_NTRAIN", 300))
SPAN = 54                                               # depth steps covered by one window (5 input + 5 forecast frames at stride 6, minus one)


def _default_seed(r):
    return r["seed"] == 1000 + 10 * r["fold"] + r["K"]


def load_runs(extra=False):
    groups = collections.defaultdict(list)
    for f in sorted(glob.glob("completion/results/runs/*.json")):
        r = json.load(open(f))
        small_pool = r["exp"].startswith("R") and r["n_train"] <= NTR_F                    # reverse transfers train on ALL windows of a small region's pool (fewer than 300)
        if r.get("test") is None or r["epochs"] != EPOCHS_F or (r["n_train"] != NTR_F and not small_pool):      # design-phase / blind runs are not part of the final tables
            continue
        frozen = int(r.get("batch", 1)) == 1 and float(r.get("lr", 1e-3)) == 1e-3 and _default_seed(r)
        if frozen == extra:
            continue
        key = (r["exp"], r.get("arch", "v1"), r["regime"], r["variant"], bool(r.get("augment", False)), bool(r.get("blocked", False)), r["epochs"], NTR_F if small_pool else r["n_train"])
        if extra:
            key += (int(r.get("batch", 1)), float(r.get("lr", 1e-3)), int(r["seed"]) if not _default_seed(r) else "default", bool(r.get("guard", True)))
        groups[key].append(r)
    return groups


def load_baselines():
    b = collections.defaultdict(list)
    for f in glob.glob(f"completion/results/baselines_*_n{NTR_F}*.json"):
        d = json.load(open(f))
        b[(d["exp"], d["regime"], bool(d.get("blocked", False)))].append(d)
    return b


def complete(runs):
    """True only if the group holds exactly the K folds 0..K-1, once each."""
    return bool(runs) and sorted(r["fold"] for r in runs) == list(range(runs[0]["K"]))


def pooled_skill(win_list):
    m = sum(np.sum(w["sse_m"]) for w in win_list)
    p = sum(np.sum(w["sse_p"]) for w in win_list)
    return 1 - m / p


def _blocks(starts):
    """start index -> time-block id, blocks of ceil(SPAN / median spacing) + 1 consecutive start times; also the number of blocks."""
    u = sorted(set(int(s) for s in starts))
    sp = float(np.median(np.diff(u))) if len(u) > 1 else float(SPAN)
    L = int(math.ceil(SPAN / max(sp, 1.0))) + 1
    return {s: i // L for i, s in enumerate(u)}, int(math.ceil(len(u) / L))


def tile_bootstrap(runs, n=NB):
    """Pooled held-out skill: two-way cluster bootstrap over tiles x time blocks (every tile is held out in exactly one fold)."""
    rng = np.random.default_rng(12345)
    rows = []                                           # (tile id, start, sse_m, sse_p)
    for r in runs:
        w = r["heldout_windows"]
        for tr, tc, st, m, p in zip(w["tile_r"], w["tile_c"], w["start"], np.sum(w["sse_m"], axis=1), np.sum(w["sse_p"], axis=1)):
            rows.append(((tr, tc), st, m, p))
    tiles = sorted({t for t, *_ in rows})
    if len(tiles) < 2:
        return None
    tid = {t: i for i, t in enumerate(tiles)}
    blk, nb = _blocks([st for _, st, _, _ in rows])
    T = np.array([tid[t] for t, *_ in rows]); B = np.array([blk[int(st)] for _, st, _, _ in rows])
    M = np.array([m for *_, m, _ in rows]); P = np.array([p for *_, p in rows])
    out = []
    for _ in range(n):
        ct = np.bincount(rng.integers(0, len(tiles), len(tiles)), minlength=len(tiles))
        cb = np.bincount(rng.integers(0, nb, nb), minlength=nb)
        w = ct[T] * cb[B]
        out.append(1 - (w * M).sum() / max((w * P).sum(), 1e-30))
    return [float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))]


def _per_fold_starts(runs):
    per = []
    for r in runs:
        w = r["test_windows"]
        d = collections.defaultdict(lambda: [0.0, 0.0])
        for s, m, p in zip(w["start"], np.sum(w["sse_m"], axis=1), np.sum(w["sse_p"], axis=1)):
            d[s][0] += m
            d[s][1] += p
        per.append(d)
    return per


def start_bootstrap(runs, n=NB):
    """Mean over folds of the never-seen-region skill; resamples TIME BLOCKS of start times (non-overlapping blocks)."""
    rng = np.random.default_rng(12345)
    per = _per_fold_starts(runs)
    starts = sorted(set(runs[0]["test_windows"]["start"]))
    blk, nb = _blocks(starts)
    B = np.array([blk[s] for s in starts])
    Mf = np.array([[d[s][0] for s in starts] for d in per]); Pf = np.array([[d[s][1] for s in starts] for d in per])
    out = []
    for _ in range(n):
        cb = np.bincount(rng.integers(0, nb, nb), minlength=nb)[B]
        out.append(np.mean(1 - (Mf * cb).sum(1) / np.maximum((Pf * cb).sum(1), 1e-30)))
    return [float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))]


def paired_start_bootstrap(runs_a, runs_b, n=NB):
    """(mean difference, [95% CI]) of the fold-averaged never-seen skill of group A minus group B, resampling the SAME time blocks for both."""
    rng = np.random.default_rng(12345)
    pa, pb = _per_fold_starts(runs_a), _per_fold_starts(runs_b)
    starts = sorted(set(runs_a[0]["test_windows"]["start"]) & set(runs_b[0]["test_windows"]["start"]))
    blk, nb = _blocks(starts)
    B = np.array([blk[s] for s in starts])

    def arr(per, i):
        return np.array([[d[s][i] for s in starts] for d in per])
    Ma, Pa, Mb, Pb = arr(pa, 0), arr(pa, 1), arr(pb, 0), arr(pb, 1)
    point = float(np.mean(1 - Ma.sum(1) / Pa.sum(1)) - np.mean(1 - Mb.sum(1) / Pb.sum(1)))
    out = []
    for _ in range(n):
        cb = np.bincount(rng.integers(0, nb, nb), minlength=nb)[B]
        out.append(np.mean(1 - (Ma * cb).sum(1) / np.maximum((Pa * cb).sum(1), 1e-30)) - np.mean(1 - (Mb * cb).sum(1) / np.maximum((Pb * cb).sum(1), 1e-30)))
    return point, [float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--md", default="completion/results/summary.md")
    a = ap.parse_args()
    groups, base = load_runs(), load_baselines()
    lines = ["| exp | arch | regime | variant | aug | blocked | K | held-out skill per fold | held-out pooled [95% CI] | test skill per fold | test mean [95% CI] | S1 / S2 / S3 (held-out; test) |",
             "|---|---|---|---|---|---|---|---|---|---|---|---|"]
    summary = {}
    for key, runs in sorted(groups.items()):
        runs = sorted(runs, key=lambda r: r["fold"])
        K = runs[0]["K"]
        exp, arch, regime, variant, aug, blk, ep, ntr = key
        ho = [r["heldout"]["skill"] for r in runs]
        te = [r["test"]["skill"] for r in runs]
        ok = complete(runs)
        ci_h = tile_bootstrap(runs) if ok else None
        ci_t = start_bootstrap(runs) if ok else None
        b = base.get((exp, regime, blk), [])
        bs = " / ".join(f"{np.mean([x[k]['heldout'] for x in b if k in x]):+.3f};{np.mean([x[k]['test'] for x in b if k in x]):+.3f}" for k in ("S1", "S2", "S3")) if b else "-"
        fmt = lambda c: f"[{c[0]:+.3f}, {c[1]:+.3f}]" if c else "(incomplete)"
        lines.append(f"| {exp} | {arch} | {regime} | {variant} | {'y' if aug else 'n'} | {'y' if blk else 'n'} | {len(runs)}/{K} | {' '.join(f'{v:+.3f}' for v in ho)} | "
                     f"{pooled_skill([r['heldout_windows'] for r in runs]):+.3f} {fmt(ci_h)} | {' '.join(f'{v:+.3f}' for v in te)} | {np.mean(te):+.3f} {fmt(ci_t)} | {bs} |")
        summary["|".join(map(str, key))] = dict(heldout=ho, test=te, ci_heldout=ci_h, ci_test=ci_t,
                                                ablation_heldout={k: float(np.mean([r["ablation_heldout"][k] for r in runs if r.get("ablation_heldout") and k in r["ablation_heldout"]]))
                                                                  for k in (runs[0].get("ablation_heldout") or {})},
                                                ablation_test={k: float(np.mean([r["ablation_test"][k] for r in runs if r.get("ablation_test") and k in r["ablation_test"]]))
                                                               for k in (runs[0].get("ablation_test") or {})})
    md = "\n".join(lines)
    extra = load_runs(extra=True)
    if extra:
        md += "\n\n### Non-frozen runs (other batch size / learning rate / seed): reported separately, never merged\n\n| exp | arch | regime | batch | lr | seed | n runs | test skill per fold | held-out skill per fold |\n|---|---|---|---|---|---|---|---|---|\n"
        for key, runs in sorted(extra.items(), key=lambda kv: str(kv[0])):
            runs = sorted(runs, key=lambda r: r["fold"])
            tk = " ".join("%+.3f" % r["test"]["skill"] for r in runs)
            hk = " ".join("%+.3f" % r["heldout"]["skill"] for r in runs)
            md += f"| {key[0]} | {key[1]} | {key[2]} | {key[8]} | {key[9]} | {key[10]} | {len(runs)} | {tk} | {hk} |\n"
    open(a.md, "w", encoding="utf-8").write(md + "\n")
    json.dump(summary, open(a.md.replace(".md", ".json"), "w"), indent=1)
    print(md)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile completion/g_report.py
"""Turn the raw results into the final tables and the verdicts on the pre-registered hypotheses (completion/PREREGISTRATION.md).
Reads completion/results/runs/*.json (neural runs) and completion/results/baselines_*_n300*.json (baseline ladder); writes completion/results/REPORT.md
and completion/results/report.json.  Missing groups are simply skipped, so it can be run at any time while the queue is still working.
Usage: python completion/g_report.py
"""
import collections
import glob
import json
import os
import sys

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import d_aggregate as A  # noqa: E402  (chdirs to the project root)

EXPS = ["E60", "E30", "E480"]
EXP_NAME = {"E60": "Australia -> UK, 60 m", "E30": "Australia -> UK, 30 m", "E480": "Pakistan -> Mozambique, 480 m"}
REG = ["causal", "hybrid", "forced"]
NN = [("v1", "original CNN-LSTM (v1)"), ("local1", "CNN kernel 1 (per pixel)"), ("local", "local CNN 5x5, 4.1k params"), ("local11", "CNN kernel 11"),
      ("clstm", "small ConvLSTM (CNN-LSTM), 4.7k params"), ("clstmf", "ConvLSTM, future rain as side channels (forced only)"), ("v2lin", "heavy ConvLSTM + local path")]
BASE = [("S1", "linear: depth trend only"), ("S3c", "linear: rain + state + trend (>=0)"), ("S5", "2-feature lookup table (depth x last change)"), ("S4c", "per-pixel MLP")]


def fmt(v, ci=None):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "-"
    return f"{v:+.3f}" + (f" [{ci[0]:+.3f}, {ci[1]:+.3f}]" if ci else "")


def main():
    groups, base = A.load_runs(), A.load_baselines()
    out, js = [], {}

    def grp(exp, arch, regime, variant="full", aug=False, blk=False):
        for k, v in groups.items():
            if k[:6] == (exp, arch, regime, variant, aug, blk):
                return sorted(v, key=lambda r: r["fold"])
        return []

    def stat(runs):
        if not A.complete(runs):
            return None
        te = [r["test"]["skill"] for r in runs]
        return dict(test_mean=float(np.mean(te)), test_folds=te, test_ci=A.start_bootstrap(runs), held_pooled=float(A.pooled_skill([r["heldout_windows"] for r in runs])),
                    held_ci=A.tile_bootstrap(runs), held_folds=[r["heldout"]["skill"] for r in runs])

    def bl(exp, regime, key, part, blk=False):
        L = base.get((exp, regime, blk), [])
        return float(np.mean([x[key][part] for x in L if key in x])) if L and all(key in x for x in L) and len(L) == 3 else None

    S = {}
    for exp in EXPS:
        for reg in REG:
            for arch, _ in NN:
                S[(exp, arch, reg)] = stat(grp(exp, arch, reg))

    # ------------------------------------------------------------- headline table per experiment
    out.append("# Results report (generated by completion/g_report.py)\n")
    for exp in EXPS:
        out.append(f"## {exp}: {EXP_NAME[exp]}  (skill vs persistence; mean over 3 tile folds; [95% CI]: bootstrap over start times / tiles)\n")
        out.append("| model | " + " | ".join(f"{r}: never-seen test" for r in REG) + " | " + " | ".join(f"{r}: held-out tiles (neural: pooled; baselines: mean of folds)" for r in REG) + " |")
        out.append("|---|" + "---|" * (2 * len(REG)))
        out.append("| persistence (copy the last frame) | " + " | ".join("0 (by definition)" for _ in range(2 * len(REG))) + " |")
        for key, label in BASE:
            cells = [fmt(bl(exp, r, key, "test")) for r in REG] + [fmt(bl(exp, r, key, "heldout")) for r in REG]
            out.append(f"| {label} | " + " | ".join(cells) + " |")
        _dag_used = []
        for arch, label in NN:
            cells_t, cells_h = [], []
            for r in REG:
                s = S[(exp, arch, r)]
                cells_t.append(fmt(s["test_mean"], s["test_ci"]) if s else "-")
                cells_h.append(fmt(s["held_pooled"], s["held_ci"]) if s else "-")
            if any(c != "-" for c in cells_t + cells_h):
                dag = " \u2020" if any(not r_.get("guard", True) for rg in REG for r_ in grp(exp, arch, rg)) else ""
                _dag_used.append(bool(dag))
                out.append(f"| **{label}{dag}** | " + " | ".join(cells_t + cells_h) + " |")
        if any(_dag_used):
            out.append("\n\u2020 contains runs that were finished with the dead-network check disabled after it aborted them falsely (same seed and recipe; see the table 'Runs aborted by the dead-network guard').")
        out.append("")
    js["headline"] = {f"{e}|{a}|{r}": v for (e, a, r), v in S.items() if v}
    js["baselines"] = {f"{e}|{r}|{k}": {"test": bl(e, r, k, "test"), "heldout": bl(e, r, k, "heldout")}
                       for e in EXPS for r in REG for k in ("S1", "S3c", "S5", "S4c") if bl(e, r, k, "test") is not None}

    # ------------------------------------------------------------- hypotheses
    out.append("## Pre-registered hypotheses\n")
    lines = []
    # H1: clstm and local beat persistence on held-out tiles AND never-seen regions, every regime (95% CI lower bound > 0)
    ok, tot, bad = 0, 0, []
    for arch in ("local", "clstm"):
        for exp in EXPS:
            for reg in REG:
                s = S[(exp, arch, reg)]
                if not s:
                    continue
                tot += 1
                if s["test_ci"][0] > 0 and s["held_ci"][0] > 0:
                    ok += 1
                else:
                    bad.append(f"{arch}/{exp}/{reg} (test CI {s['test_ci'][0]:+.3f}, held-out CI {s['held_ci'][0]:+.3f})")
    lines.append(f"* **H1** (local and clstm beat persistence with both intervals above 0, every regime/experiment): {ok}/{tot} cells pass" + (f"; not passing: {'; '.join(bad)}" if bad else "") + ".")
    # H2: beats best linear baseline S3c on never-seen region
    ok, tot = 0, 0
    for arch in ("local", "clstm"):
        for exp in EXPS:
            for reg in REG:
                s, b = S[(exp, arch, reg)], bl(exp, reg, "S3c", "test")
                if s and b is not None:
                    tot += 1; ok += int(s["test_mean"] > b)
    lines.append(f"* **H2** (beat the best linear baseline S3c on the never-seen region): {ok}/{tot} cells pass.")
    # H2b (added): beat the per-pixel MLP S4c?
    ok, tot, diffs = 0, 0, []
    for arch in ("local", "clstm"):
        for exp in EXPS:
            for reg in REG:
                s, b = S[(exp, arch, reg)], bl(exp, reg, "S4c", "test")
                if s and b is not None:
                    tot += 1; ok += int(s["test_mean"] > b); diffs.append(s["test_mean"] - b)
    if tot:
        lines.append(f"* **H2b** (exploratory, added after seeing the per-pixel baseline: do the CNN/ConvLSTM beat the per-pixel MLP S4c on the never-seen region?): {ok}/{tot} cells; mean difference {np.mean(diffs):+.3f} (range {min(diffs):+.3f} .. {max(diffs):+.3f}).")
    # H3 (pre-registered rule): skill without depth history is lower than with it by MORE THAN the fold-to-fold sd of the full model
    for arch3, var in (("local", "nodepthhist"), ("local", "nohist"), ("clstm", "nodepthhist")):
        rows = []
        for exp in EXPS:
            for reg in REG:
                a, b = grp(exp, arch3, reg, "full"), grp(exp, arch3, reg, var)
                if A.complete(a) and A.complete(b):
                    d = [x["test"]["skill"] - y["test"]["skill"] for x, y in zip(a, b)]
                    sd = float(np.std([x["test"]["skill"] for x in a], ddof=1))
                    rows.append((exp, reg, float(np.mean(d)), sd, float(np.mean(d)) > sd))
        if rows:
            lines.append(f"* **H3** (depth history matters; {arch3} trained {var} vs full, never-seen region): full - {var}, mean over folds (fold sd of the full model): " +
                         "; ".join(f"{e}/{r} {m:+.3f} ({sd:.3f})" for e, r, m, sd, _ in rows) + f". Rule (difference > fold sd) holds in {sum(1 for *_, ok in rows if ok)}/{len(rows)} cells.")
    # H4 (pre-registered rule): recurrence: clstm - local; EQUIVALENT if |mean difference| < 0.02 and the 95% block-bootstrap interval lies inside +-0.03, otherwise the larger model wins
    rows = []
    for exp in EXPS:
        for reg in REG:
            a, b = grp(exp, "clstm", reg), grp(exp, "local", reg)
            if A.complete(a) and A.complete(b):
                d, ci = A.paired_start_bootstrap(a, b)
                verdict = "equivalent" if (abs(d) < 0.02 and ci[0] > -0.03 and ci[1] < 0.03) else ("clstm larger" if d > 0 else "local larger")
                if verdict != "equivalent" and ci[0] < 0 < ci[1]:
                    verdict += " (by the rule, but the interval includes 0)"
                rows.append((exp, reg, d, ci, verdict))
    if rows:
        lines.append("* **H4** (recurrence matters: clstm - local, never-seen region, paired block bootstrap; pre-registered rule): " +
                     "; ".join(f"{e}/{r} {d:+.3f} [{ci[0]:+.3f}, {ci[1]:+.3f}] -> {v}" for e, r, d, ci, v in rows) + ".")
    # H5: ordering
    ok, tot = 0, 0
    for arch, _ in NN:
        for exp in EXPS:
            v = [S[(exp, arch, r)] for r in REG]
            if all(v):
                tot += 1; ok += int(v[0]["test_mean"] <= v[1]["test_mean"] <= v[2]["test_mean"])
    if tot:
        lines.append(f"* **H5** (causal <= hybrid <= forced on the never-seen region, means): {ok}/{tot} (arch, experiment) cases hold. Caution (code review F4): the recurrent model reads the forced rain as five extra steps AFTER the observed history, which handicaps it; a failure for clstm is an input-layout effect, not evidence about rain physics.")
    # H6: blocked vs standard held-out
    rows = []
    for exp in EXPS:
        for arch in ("local", "clstm"):
            a, b = grp(exp, arch, "hybrid"), grp(exp, arch, "hybrid", blk=True)
            if A.complete(a) and A.complete(b):
                pa = A.pooled_skill([r["heldout_windows"] for r in a]); pb = A.pooled_skill([r["heldout_windows"] for r in b])
                rows.append((exp, arch, pa, pb))
    if rows:
        lines.append("* **H6** (strict space+time hold-out within 0.05 of the tile-only hold-out): " + "; ".join(f"{e}/{a}: {pa:+.3f} -> {pb:+.3f}" for e, a, pa, pb in rows) +
                     f". Pass in {sum(1 for *_, pa, pb in rows if abs(pa - pb) < 0.05)}/{len(rows)}.")
    out += lines + [""]
    js["hypotheses"] = lines

    # ------------------------------------------------------------- test-time ablations (mean over folds), hybrid
    out.append("## What the trained models use (test-time ablations, never-seen region, hybrid regime, mean over folds)\n")
    keys = ["none", "rain_removed", "terrain_removed", "depth_hist_scrambled", "depth_hist_frozen", "rain_hist_frozen", "rain_shifted_earlier"]
    out.append("| experiment | model | " + " | ".join(keys) + " |")
    out.append("|---|---|" + "---|" * len(keys))
    for exp in EXPS:
        for arch, label in NN:
            runs = grp(exp, arch, "hybrid")
            if len(runs) == 3 and all(r.get("ablation_test") for r in runs):
                out.append(f"| {exp} | {label} | " + " | ".join(f"{np.mean([r['ablation_test'][k] for r in runs if k in r['ablation_test']]):+.3f}" if all(k in r['ablation_test'] for r in runs) else "-" for k in keys) + " |")
    out.append("")

    # ------------------------------------------------------------- training-time ablations
    out.append("## Training-time ablations (models trained without one ingredient; never-seen region skill, mean over folds)\n")
    variants = ["full", "nodepthhist", "norainhist", "nohist", "norain", "noterrain"]
    out.append("| experiment | model | regime | " + " | ".join(variants) + " |")
    out.append("|---|---|---|" + "---|" * len(variants))
    for arch_t, lab_t in (("local", "local CNN"), ("clstm", "ConvLSTM")):
        for exp in EXPS:
            for reg in REG:
                cells = []
                for v in variants:
                    r = grp(exp, arch_t, reg, v)
                    cells.append(f"{np.mean([x['test']['skill'] for x in r]):+.3f}" if A.complete(r) else "-")
                if any(c != "-" for c in cells[1:]):
                    out.append(f"| {exp} | {lab_t} | {reg} | " + " | ".join(cells) + " |")
    out.append("")

    out.append("## What the models do with KNOWN future rain (test-time ablations, forced regime, never-seen region, mean over folds)\n")
    keys_f = ["none", "rain_removed", "future_rain_removed", "depth_hist_frozen"]
    out.append("| experiment | model | " + " | ".join(keys_f) + " |")
    out.append("|---|---|" + "---|" * len(keys_f))
    for exp in EXPS:
        for arch, label in NN:
            runs = grp(exp, arch, "forced")
            if len(runs) == 3 and all(r.get("ablation_test") and "future_rain_removed" in r["ablation_test"] for r in runs):
                out.append(f"| {exp} | {label} | " + " | ".join(f"{np.mean([r['ablation_test'][k] for r in runs]):+.3f}" for k in keys_f) + " |")
    out.append("")

    # ------------------------------------------------------------- exploratory batches (per-fold records; never merged into the headline groups)
    extra = A.load_runs(extra=True)

    def every(exp, arch, regime, variant="full", blocked=False):
        """(run, is_frozen) for every final-size run of this configuration, frozen recipe or not."""
        res = []
        for fl, gs in ((True, A.load_runs()), (False, extra)):
            for k, v in gs.items():
                if k[:6] == (exp, arch, regime, variant, False, blocked):
                    res += [(r, fl) for r in v]
        return sorted(res, key=lambda t: (t[0]["fold"], not t[1], t[0]["seed"]))

    out.append("## Exploratory: ConvLSTM with the future rain as side channels (clstmf, forced regime), per fold (frozen recipe unless marked)\n")
    out.append("| experiment | model | fold | seed | never-seen skill | held-out tiles skill | note |")
    out.append("|---|---|---|---|---|---|---|")
    for exp in ("E60", "E30", "E480"):
        for arch in ("clstm", "clstmf"):
            seen = set()
            for r, fl in every(exp, arch, "forced"):
                seen.add((r["fold"], fl))
                note = ("guard disabled after a false abort (post hoc; identical recipe and seed)" if (fl and not r.get("guard", True)) else "") if fl else (f"RETRY after the frozen-seed run was aborted by the guard (post hoc): seed {r['seed']}" + (f", learning rate {float(r.get('lr', 1e-3)):g}" if abs(float(r.get('lr', 1e-3)) - 1e-3) > 1e-12 else ""))
                out.append(f"| {exp} | {arch} | {r['fold']} | {r['seed']} | {fmt(r['test']['skill'])} | {fmt(r['heldout']['skill'])} | {note} |")
            if arch == "clstmf":
                for f in range(3):
                    lf = f"completion/logs/runs/{exp}_clstmf_forced_full_f{f}of3_s{1000 + 10 * f + 3}_ep30.log"
                    if (f, True) not in seen and os.path.exists(lf):
                        txt = open(lf, errors="replace").read()
                        if "DEAD NETWORK" in txt:
                            ep = txt.split("DEAD NETWORK after epoch ")[-1].split(":")[0]
                            out.append(f"| {exp} | clstmf | {f} | {1000 + 10 * f + 3} | - | - | ABORTED by the dead-network guard after epoch {ep} (a false alarm: see the guard-disabled row); no result of its own |")
                        for tag, sd_, extra_ in (("retry", 4, ""), ("retry2", 5, ", learning rate 0.0005")):
                            lf2 = f"completion/logs/runs/{exp}_clstmf_forced_{tag}_f{f}.log"
                            if os.path.exists(lf2) and "DEAD NETWORK" in open(lf2, errors="replace").read():
                                ep2 = open(lf2, errors="replace").read().split("DEAD NETWORK after epoch ")[-1].split(":")[0]
                                out.append(f"| {exp} | clstmf | {f} | {1000 + 10 * f + sd_} | - | - | RETRY (post hoc){extra_}: also ABORTED by the guard after epoch {ep2}; not pursued further |")
    out.append("")

    out.append("## The original model (v1) against the final models, same folds, hybrid regime, per fold (never-seen / held-out tiles)\n")
    out.append("| experiment | fold | original v1 | local CNN | small ConvLSTM |")
    out.append("|---|---|---|---|---|")
    for exp in EXPS:
        for f in range(3):
            cells = []
            v1s = [(r, fl) for r, fl in every(exp, "v1", "hybrid") if r["fold"] == f]
            txt = "; ".join(f"{fmt(r['test']['skill'])} / {fmt(r['heldout']['skill'])}" + ("" if r.get("guard", True) else " (guard disabled: the old check looked at the earliest, near-static windows and aborted it falsely)") for r, fl in v1s)
            for arch in ("local", "clstm"):
                rr = [r for r in grp(exp, arch, "hybrid") if r["fold"] == f]
                cells.append(f"{fmt(rr[0]['test']['skill'])} / {fmt(rr[0]['heldout']['skill'])}" if rr else "-")
            if txt or any(c != "-" for c in cells):
                out.append(f"| {exp} | {f} | {txt or '-'} | " + " | ".join(cells) + " |")
    out.append("")

    out.append("## Exploratory: optimisation robustness (batch 8, learning rate 3e-3, hybrid; a deviation from the frozen recipe)\n")
    out.append("| experiment | model | frozen recipe: never-seen / held-out | batch 8, lr 3e-3: never-seen / held-out | folds |")
    out.append("|---|---|---|---|---|")
    for exp in EXPS:
        for arch in ("local", "clstm"):
            fz = stat(grp(exp, arch, "hybrid"))
            op = [r for r, fl in every(exp, arch, "hybrid") if not fl and int(r.get("batch", 1)) == 8 and abs(float(r.get("lr", 1e-3)) - 3e-3) < 1e-9]
            if op:
                done_ok = A.complete(op)
                a_ = f"{fmt(fz['test_mean'])} / {fmt(fz['held_pooled'])}" if fz else "-"
                b_ = (f"{fmt(float(np.mean([r['test']['skill'] for r in op])))} / {fmt(float(A.pooled_skill([r['heldout_windows'] for r in op])))}" if done_ok
                      else "incomplete: " + ", ".join(f"f{r['fold']} {fmt(r['test']['skill'])}/{fmt(r['heldout']['skill'])}" for r in op))
                out.append(f"| {exp} | {arch} | {a_} | {b_} | {len(op)} |")
    out.append("")

    out.append("## Exploratory: seed repeats of the local CNN (hybrid; the frozen-seed run plus seeds 11-14)\n")
    out.append("| experiment | fold | never-seen skill per seed (frozen seed first) | sd over seeds | held-out tiles skill per seed (seed AND data draw change) | sd over seeds |")
    out.append("|---|---|---|---|---|---|")
    seed_sd, fold_sd = {}, {}
    for exp in EXPS:
        per_fold_t, per_fold_h, within_t, within_h = [], [], [], []
        for f in range(3):
            rs = [r for r, fl in every(exp, "local", "hybrid") if r["fold"] == f and int(r.get("batch", 1)) == 1 and abs(float(r.get("lr", 1e-3)) - 1e-3) < 1e-9]
            if len(rs) >= 2:
                t = [r["test"]["skill"] for r in rs]
                h = [r["heldout"]["skill"] for r in rs]
                out.append(f"| {exp} | {f} | " + " ".join(f"{x:+.3f}" for x in t) + f" | {np.std(t, ddof=1):.3f} | " + " ".join(f"{x:+.3f}" for x in h) + f" | {np.std(h, ddof=1):.3f} |")
                per_fold_t.append(float(np.mean(t))); per_fold_h.append(float(np.mean(h)))
                within_t.append(float(np.std(t, ddof=1))); within_h.append(float(np.std(h, ddof=1)))
        if len(per_fold_t) == 3:
            seed_sd[exp] = (float(np.mean(within_t)), float(np.mean(within_h)))
            fold_sd[exp] = (float(np.std(per_fold_t, ddof=1)), float(np.std(per_fold_h, ddof=1)))
    for exp in seed_sd:
        out.append(f"\n**{exp}:** sd over seeds within a fold (mean of the 3 folds): never-seen {seed_sd[exp][0]:.3f}, held-out tiles {seed_sd[exp][1]:.3f}; sd of the seed-averaged skill over the 3 folds: never-seen {fold_sd[exp][0]:.3f}, held-out tiles {fold_sd[exp][1]:.3f}.")
    js["seed_repeats"] = {"seed_sd": seed_sd, "fold_sd": fold_sd}
    out.append("")

    out.append("## Exploratory: the strict space+time split on Pakistan (E480, local CNN, fold 0): other seeds and the batch-8 / lr 3e-3 recipe\n")
    out.append("| recipe | seed | held-out (tiles AND a time block hidden) | never-seen (Mozambique) |")
    out.append("|---|---|---|---|")
    rs_s = [(r, fl) for r, fl in every("E480", "local", "hybrid", blocked=True) if r["fold"] == 0]
    for r, fl in rs_s:
        rec = "frozen (batch 1, lr 1e-3)" if (int(r.get("batch", 1)) == 1 and abs(float(r.get("lr", 1e-3)) - 1e-3) < 1e-12) else f"batch {int(r.get('batch', 1))}, lr {float(r.get('lr', 1e-3)):g}"
        out.append(f"| {rec} | {r['seed']} | {fmt(r['heldout']['skill'])} | {fmt(r['test']['skill'])} |")
    out.append("")

    # ------------------------------------------------------------- learning curve and training length (final-size runs of other n_train / epochs; d_aggregate filters those out)
    def raw_runs(exp, arch, regime="hybrid", n_train=None, epochs=30, variant="full"):
        res = []
        for f in sorted(glob.glob("completion/results/runs/*.json")):
            r = json.load(open(f))
            if (r["exp"], r.get("arch", "v1"), r["regime"], r["variant"], bool(r.get("augment", False)), bool(r.get("blocked", False))) != (exp, arch, regime, variant, False, False):
                continue
            if r.get("test") is None or r["epochs"] != epochs or int(r.get("batch", 1)) != 1 or abs(float(r.get("lr", 1e-3)) - 1e-3) > 1e-12 or (n_train is not None and r["n_train"] != n_train):
                continue
            res.append(r)
        return res

    def fold_mean(rs, key="test", frozen_seed=None):
        by = {}
        for r in rs:
            if frozen_seed is not None and (r["seed"] == 1000 + 10 * r["fold"] + r["K"]) != frozen_seed:
                continue
            by[r["fold"]] = r
        return (float(np.mean([by[f][key]["skill"] for f in range(3)])), [by[f][key]["skill"] for f in range(3)]) if len(by) == 3 else None

    def base_n(exp, key, n, part="test"):
        v = []
        for f in range(3):
            p = f"completion/results/baselines_{exp}_hybrid_f{f}of3_n{n}.json"
            if os.path.exists(p):
                d = json.load(open(p))
                if key in d:
                    v.append(d[key][part]["skill"] if isinstance(d[key][part], dict) else d[key][part])
        return float(np.mean(v)) if len(v) == 3 else None

    NS = (25, 50, 100, 300, 600)
    out.append("## Exploratory: learning curve (never-seen skill vs number of training windows; hybrid; mean over 3 folds; the frozen recipe uses 300; FIXED 30 epochs, so the number of optimiser steps grows with n - see the equal-steps table below)\n")
    out.append("| experiment | model | " + " | ".join(f"n = {n}" for n in NS) + " |")
    out.append("|---|---|" + "---|" * len(NS))
    lc = {}
    for exp in EXPS:
        row_cnn = []
        for n in NS:
            rs = raw_runs(exp, "local", n_train=n)
            fm = fold_mean(rs, frozen_seed=True if n == 300 else False)
            row_cnn.append(fm[0] if fm else None)
        out.append(f"| {exp} | local CNN | " + " | ".join(fmt(v) for v in row_cnn) + " |")
        lc[f"{exp}|local"] = row_cnn
        for key, lab in (("S1", "linear trend"), ("S5", "lookup table"), ("S4c", "per-pixel MLP")):
            row = [base_n(exp, key, n) for n in NS]
            if any(v is not None for v in row):
                out.append(f"| {exp} | {lab} | " + " | ".join(fmt(v) for v in row) + " |")
                lc[f"{exp}|{key}"] = row
    js["learning_curve"] = lc
    out.append("")
    out.append("## Exploratory: learning curve with EQUAL optimiser steps (local CNN, hybrid, never-seen, mean of 3 folds; steps = epochs x windows)\n")
    out.append("| experiment | n=300, 30 epochs (9,000 steps) | n=600, 15 epochs (9,000 steps) | n=300, 60 epochs (18,000 steps) | n=600, 30 epochs (18,000 steps) |")
    out.append("|---|---|---|---|---|")
    for exp in EXPS:
        c1 = fold_mean(raw_runs(exp, "local", n_train=300, epochs=30), frozen_seed=True)
        c2 = fold_mean(raw_runs(exp, "local", n_train=600, epochs=15))
        c3 = fold_mean(raw_runs(exp, "local", n_train=300, epochs=60))
        c4 = fold_mean(raw_runs(exp, "local", n_train=600, epochs=30))
        if c2 or c3:
            out.append(f"| {exp} | {fmt(c1[0] if c1 else None)} | {fmt(c2[0] if c2 else None)} | {fmt(c3[0] if c3 else None)} | {fmt(c4[0] if c4 else None)} |")
            js.setdefault("equal_steps", {})[exp] = [x[0] if x else None for x in (c1, c2, c3, c4)]
    out.append("")

    out.append("## Exploratory: training length (hybrid; 30 epochs = frozen recipe versus 90 epochs; mean over 3 folds)\n")
    out.append("| experiment | model | never-seen, 30 epochs | never-seen, 90 epochs | held-out tiles (mean of fold skills), 30 | held-out tiles, 90 |")
    out.append("|---|---|---|---|---|---|")
    for exp in EXPS:
        for arch_e, lab_e in (("local", "local CNN"), ("clstm", "small ConvLSTM")):
            a = fold_mean(raw_runs(exp, arch_e, n_train=300, epochs=30), frozen_seed=True)
            b = fold_mean(raw_runs(exp, arch_e, n_train=300, epochs=90))
            a2 = fold_mean(raw_runs(exp, arch_e, n_train=300, epochs=30), key="heldout", frozen_seed=True)
            b2 = fold_mean(raw_runs(exp, arch_e, n_train=300, epochs=90), key="heldout")
            if b:
                out.append(f"| {exp} | {lab_e} | {fmt(a[0] if a else None)} | {fmt(b[0])} | {fmt(a2[0] if a2 else None)} | {fmt(b2[0] if b2 else None)} |")
                js.setdefault("epochs90", {})[f"{exp}|{arch_e}"] = dict(ep30=a[0] if a else None, ep90=b[0])
    out.append("")
    for exp in EXPS:
        ra = sorted(raw_runs(exp, "clstm", n_train=300, epochs=90), key=lambda r: r["fold"])
        rb = sorted(raw_runs(exp, "local", n_train=300, epochs=90), key=lambda r: r["fold"])
        if A.complete(ra) and A.complete(rb):
            d90, ci90 = A.paired_start_bootstrap(ra, rb)
            out.append(f"* Recurrence with EQUAL training length (90 epochs both), {exp}: ConvLSTM minus local CNN = {d90:+.3f} [{ci90[0]:+.3f}, {ci90[1]:+.3f}] (paired block bootstrap over time blocks)")
            js.setdefault("h4_epochs90", {})[exp] = dict(diff=d90, ci=ci90)
    out.append("")

    # ------------------------------------------------------------- reverse-direction transfers (EXPLORATORY; time-block folds instead of tile folds)
    RNAME = {"R480": "Mozambique -> Pakistan, 480 m", "R60": "UK -> Australia, 60 m", "R30": "UK -> Australia, 30 m"}
    have_rev = any(grp(e, "local", "hybrid") for e in RNAME)
    if have_rev:
        out.append("## Exploratory: reverse-direction transfers (train on the small region, test on the big one; 3 TIME-block folds, no tile folds - see completion/reverse_transfer.py)\n")
        out.append("Never-seen skill (mean over the 3 folds, [95% block-bootstrap interval]); 'time-held-out' = windows of the training region from a hidden third of the start times (pooled). Not comparable with the forward experiments cell by cell.\n")
        out.append("| experiment | model | " + " | ".join(f"{r}: never-seen" for r in REG) + " | " + " | ".join(f"{r}: time-held-out" for r in REG) + " |")
        out.append("|---|---|" + "---|" * (2 * len(REG)))
        for exp, nm in RNAME.items():
            for key, label in BASE:
                cells = [fmt(bl(exp, r, key, "test")) for r in REG] + [fmt(bl(exp, r, key, "heldout")) for r in REG]
                if any(c != "-" for c in cells):
                    out.append(f"| {exp} ({nm}) | {label} | " + " | ".join(cells) + " |")
            for arch, label in (("local", "local CNN 5x5"), ("clstm", "small ConvLSTM")):
                cells_t, cells_h = [], []
                for r in REG:
                    s = stat(grp(exp, arch, r))
                    cells_t.append(fmt(s["test_mean"], s["test_ci"]) if s else "-")
                    cells_h.append(fmt(s["held_pooled"]) if s else "-")
                if any(c != "-" for c in cells_t + cells_h):
                    out.append(f"| {exp} ({nm}) | **{label}** | " + " | ".join(cells_t + cells_h) + " |")
                    js.setdefault("reverse", {})[f"{exp}|{arch}"] = dict(test=cells_t, heldout=cells_h)
        out.append("")
    # ------------------------------------------------------------- where the failures are: the windows at the very start of the flood (start index 0)
    out.append("## Where the failures are: the windows at the very START of the flood (start index 0)\n")
    out.append("Pooled over the 3 folds. 'start 0' = windows whose first observed frame is the first frame of the simulation (the flood is just beginning).\n")
    out.append("| what | windows (of which start 0) | skill, all windows | share of the model's squared error in the start-0 windows | skill without the start-0 windows |")
    out.append("|---|---|---|---|---|")
    specs = [("Mozambique -> Pakistan (R480), local CNN, never-seen", "R480", "local", False, "test_windows"),
             ("Mozambique -> Pakistan (R480), ConvLSTM, never-seen", "R480", "clstm", False, "test_windows"),
             ("E480 strict split, local CNN, held-out", "E480", "local", True, "heldout_windows"),
             ("E480 strict split, ConvLSTM, held-out", "E480", "clstm", True, "heldout_windows"),
             ("E480 tile-only hold-out, local CNN, held-out", "E480", "local", False, "heldout_windows"),
             ("E480 tile-only hold-out, ConvLSTM, held-out", "E480", "clstm", False, "heldout_windows"),
             ("E480 never-seen (Mozambique), local CNN", "E480", "local", False, "test_windows"),
             ("UK -> Australia 60 m (R60), local CNN, never-seen", "R60", "local", False, "test_windows"),
             ("UK -> Australia 30 m (R30), local CNN, never-seen", "R30", "local", False, "test_windows")]
    onset = {}
    for lab, exp_, arch_, blk_, which_ in specs:
        rs_ = grp(exp_, arch_, "hybrid", blk=blk_)
        if not A.complete(rs_):
            continue
        n_ = n_on = 0
        sm = sp = sm_on = sp_on = 0.0
        for r in rs_:
            w = r[which_]
            st = np.array(w["start"])
            m = np.sum(np.array(w["sse_m"]), axis=1)
            p = np.sum(np.array(w["sse_p"]), axis=1)
            on = st == 0
            n_ += len(st); n_on += int(on.sum())
            sm += m.sum(); sp += p.sum(); sm_on += m[on].sum(); sp_on += p[on].sum()
        onset[f"{exp_}|{arch_}|{blk_}|{which_}"] = dict(skill=1 - sm / sp, share=sm_on / sm, skill_wo=1 - (sm - sm_on) / (sp - sp_on))
        out.append(f"| {lab} | {n_} ({n_on}) | {fmt(1 - sm / sp)} | {sm_on / sm:.0%} | {fmt(1 - (sm - sm_on) / (sp - sp_on))} |")
    js["onset_windows"] = onset
    out.append("")

    # ------------------------------------------------------------- runs aborted by the dead-network guard (never silently dropped)
    import re
    dead = []
    for lf in sorted(glob.glob("completion/logs/runs/*.log")):
        name = os.path.basename(lf)[:-4]
        txt = open(lf, errors="replace").read()
        if "DEAD NETWORK after epoch " not in txt:
            continue
        m = re.findall(r"DEAD NETWORK after epoch (\d+): .*?std ([0-9.e+-]+) m", txt)
        if not m:
            continue
        if os.path.exists(f"completion/results/runs/{name}.json"):
            continue
        dead.append((name, int(m[-1][0]), m[-1][1]))
    out.append("## Runs aborted by the dead-network guard (the OLD check: per-pixel spread of the predicted change below 1e-5 m across the 12 EARLIEST held-out windows; an aborted run gives no result of its own)\n")
    if dead:
        out.append("| run | epoch of abort | per-pixel spread (m) | outcome of a re-run with the guard disabled (post hoc) |")
        out.append("|---|---|---|---|")
        for name, ep, sp in dead:
            ng = name.replace("_full_", "_full_noguard_", 1)
            pth = f"completion/results/runs/{ng}.json"
            outc = "not re-run" if "retry" in name else "not run"
            if os.path.exists(pth):
                rr = json.load(open(pth))
                outc = f"finished: never-seen {fmt(rr['test']['skill'])}, held-out {fmt(rr['heldout']['skill'])}, final train/persistence loss {rr['history'][-1][1]:.2f}"
            out.append(f"| {name} | {ep} | {sp} | {outc} |")
        out.append("\nThe check is a safety net against a network that has died (constant output). All eight aborts listed here came from its first version, which looked only at the 12 EARLIEST held-out windows: the flood has barely started there, so even a healthy network predicts almost no change (and a slow-starting model such as the original one looks dead for its first epochs). Seven of the eight were re-run with the check disabled (same seed and recipe) and trained normally; the tables mark those runs with a dagger. The check now samples 12 windows across the whole held-out set (changed 21 September, about 00:00). The eighth (a retry with the next seed) was not re-run.")
    else:
        out.append("None.")
    js["dead_runs"] = dead
    out.append("")
    warn = []
    for k_, v_ in groups.items():
        if not A.complete(v_):
            warn.append(f"{k_[0]} / {k_[1]} / {k_[2]} / {k_[3]}{' / strict split' if k_[5] else ''}{' / augmented' if k_[4] else ''}: folds present {sorted(r['fold'] for r in v_)} (each of 0..{v_[0]['K'] - 1} is needed exactly once)")
    out.append("## Data warnings: result groups that are incomplete or hold duplicate folds (their cells are shown as '-' above)\n")
    out += [f"* {w_}" for w_ in warn] if warn else ["None."]
    js["data_warnings"] = warn
    out.append("")
    md = "\n".join(out)
    open("completion/results/REPORT.md", "w", encoding="utf-8").write(md)
    json.dump(js, open("completion/results/report.json", "w"), indent=1, default=float)
    print(md)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile completion/severity_study.py
"""Severity branch, symbolic rule layer and Grad-CAM, re-evaluated on the frozen depth architecture (clstm) with tile folds.

Joint model = the shared encoder (a full-resolution ConvLSTM, `model_v2.build_clstm`) + the depth head (persistence-anchored) + a severity head on the
encoder's hidden map: three stride-2 convolutions down to 16x16x64, then EITHER the capsule head (PrimaryCaps -> routing-by-agreement -> class capsules,
margin loss; the project's design) OR a plain dense head (Flatten -> Dense(64) -> Dense(4), cross-entropy; the control).  Severity label = max depth in the
patch over the forecast horizon, binned 0.3 / 1 / 2 m (as before); Australia 30 m only (the label is degenerate elsewhere).

Reports on held-out TILES (fold k of 3): accuracy on class-balanced (60 per class) and natural-distribution windows, macro recall, the "class-changing"
subset (windows whose label differs from the last frame's class, where the threshold baseline is wrong by construction), against
  B0  threshold the LAST OBSERVED frame,
  B1  threshold the model's OWN PREDICTED depth (max over the forecast frames),
and the symbolic rule layer (fixed original thresholds vs thresholds re-derived on the training windows of the fold): windows changed / fixed / broken,
plus whether the rule reasons are TRUE statements (sign agreement of the stated depth trend with the actual one).  Grad-CAM: wet-vs-dry heat ratio,
a deletion (faithfulness) test against random masking, and saved maps for stability across seeds.

Run: python completion/severity_study.py <regime hybrid|forced> <head capsule|dense> <fold> [--seed S] [--epochs 30] [--per-class 100]
"""
import argparse
import json
import os
import sys
import time

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import engine as E  # noqa: E402  (chdirs to the project root)

T_IN, HORIZON = E.T_IN, E.HORIZON
CLASSES = ["none", "minor", "moderate", "severe"]
OUT = "completion/results/severity"


def sev_labels(pool):
    from real_data_loader import depth_to_severity
    p = os.path.join(E.CACHE, f"sev_labels_{pool['region']}_{pool['res']}.npz")
    if os.path.exists(p):
        d = np.load(p)
        return d["lab"], d["base"]
    n = len(pool["meta"])
    lab = np.array([depth_to_severity(float(np.max(pool["df"][i]))) for i in range(n)])
    base = np.array([depth_to_severity(float(np.max(pool["dh"][i][-1]))) for i in range(n)])
    np.savez(p, lab=lab, base=base)
    return lab, base


def balanced(idx, lab, per_class, rng):
    out = []
    for c in range(4):
        m = [i for i in idx if lab[i] == c]
        out += list(rng.choice(m, size=min(per_class, len(m)), replace=False)) if m else []
    return sorted(int(i) for i in out)


def build_joint(T, head):
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    import model as m1
    import model_v2 as m2
    enc, dec = m2.build_clstm(T, HORIZON)
    h_in = keras.Input(shape=(128, 128, 8))
    z = layers.Conv2D(16, 3, strides=2, padding="same", activation="relu")(h_in)
    z = layers.Conv2D(32, 3, strides=2, padding="same", activation="relu")(z)
    z = layers.Conv2D(64, 3, strides=2, padding="same", activation="relu")(z)
    feat = keras.Model(h_in, z, name="severity_features")
    if head == "capsule":
        sev = m1.build_capsule_head(return_lengths=True)
    else:
        i = keras.Input(shape=(16, 16, 64))
        o = layers.Dense(4, name="dense_logits")(layers.Dense(64, activation="relu")(layers.Dropout(0.3)(layers.Flatten()(i))))
        sev = keras.Model(i, o, name="dense_severity_head")
    return enc, dec, feat, sev


def rule_inputs(pool, rd_lu, i, x, pred, hours):
    """Depth rate (predicted), rainfall (last observed step), slope, drainage proximity of one window."""
    from predict_severity import _compute_slope_pct, _compute_drainage_proximity_m
    r, c, _ = (int(v) for v in pool["meta"][i])
    last = x[T_IN - 1, ..., 0]
    rate = float((pred[-1].mean() - last.mean()) / hours)
    rain = float(x[T_IN - 1, ..., 2].mean())
    slope = _compute_slope_pct(np.array(pool["dem"][r:r + 128, c:c + 128]), 30.0)
    lu = rd_lu.load_landuse_patch(r, c, 128)
    dr = _compute_drainage_proximity_m(lu, 30.0)
    return rate, rain, slope, (0.0 if np.isnan(dr) else dr)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("regime"); ap.add_argument("head"); ap.add_argument("fold", type=int)
    ap.add_argument("--seed", type=int, default=None)
    ap.add_argument("--epochs", type=int, default=30)
    ap.add_argument("--per-class", type=int, default=100)
    ap.add_argument("--K", type=int, default=3)
    ap.add_argument("--augment", action="store_true")
    a = ap.parse_args()
    seed = 2000 + 10 * a.fold + a.K if a.seed is None else a.seed
    rid = f"SEV_{a.regime}_{a.head}{'_aug' if a.augment else ''}_f{a.fold}of{a.K}_s{seed}_ep{a.epochs}"
    os.makedirs(OUT, exist_ok=True)
    out_p = os.path.join(OUT, rid + ".json")
    if os.path.exists(out_p):
        print(rid, "already done"); return
    tf = E._tf()
    from depth_experiment import WarmupCosine
    from capsule_layers import margin_loss
    from real_data_loader import depth_to_severity
    from symbolic_rule_layer import apply_symbolic_rules
    import symbolic_rule_layer as SRL
    log = lambda s: print(s, flush=True)
    pool = E.build_pool("Australia", 30, "train", log)
    lab, base = sev_labels(pool)
    tiles = sorted({(int(r), int(c)) for r, c, _ in pool["meta"]})
    held = set(map(tuple, E.folds_of_tiles(tiles, a.K, seed=0)[a.fold]))
    tile_of = [(int(r), int(c)) for r, c, _ in pool["meta"]]
    tr_all = [i for i, t in enumerate(tile_of) if t not in held]
    va_all = [i for i, t in enumerate(tile_of) if t in held]
    rng = np.random.default_rng(seed)
    tr_idx = balanced(tr_all, lab, a.per_class, rng)
    va_bal = balanced(va_all, lab, 60, rng)
    va_nat = sorted(int(i) for i in rng.choice(va_all, size=min(600, len(va_all)), replace=False))
    log(f"{rid}: {len(tiles)} tiles, held-out {len(held)}; train {len(tr_idx)} balanced windows (class counts {np.bincount(lab[tr_idx], minlength=4).tolist()}); "
        f"held-out balanced {len(va_bal)}, natural {len(va_nat)} (class fractions {(np.bincount(lab[va_nat], minlength=4) / len(va_nat)).round(3).tolist()}); "
        f"last-frame baseline on natural {float(np.mean(base[va_nat] == lab[va_nat])):.3f}")

    T = T_IN + (HORIZON if a.regime == "forced" else 0)
    tf.keras.utils.set_random_seed(seed)
    enc, dec, feat, sev = build_joint(T, a.head)
    vars_ = enc.trainable_variables + dec.trainable_variables + feat.trainable_variables + sev.trainable_variables
    pers = max(float(np.mean([np.mean((np.array(pool["df"][i]) - np.array(pool["dh"][i][-1])[None]) ** 2) for i in tr_idx])), 1e-12)
    opt = tf.keras.optimizers.Adam(learning_rate=WarmupCosine(1e-3, a.epochs * len(tr_idx), 200), clipnorm=1.0)
    import model_v2 as m2
    xs = tf.TensorSpec([1, T, 128, 128, 3], tf.float32); ys = tf.TensorSpec([HORIZON, 128, 128], tf.float32); ss = tf.TensorSpec([1], tf.int32)

    def sev_out(h, training):
        return sev(feat(h, training=training), training=training)

    @tf.function(input_signature=[xs, ys, ss])
    def step(x, y, s):
        with tf.GradientTape() as tape:
            h = enc(x, training=True)
            frames = m2.decode_horizon_local(dec, h, x, HORIZON, training=True)
            dl = tf.reduce_mean(tf.square(frames[0] - y)) / pers
            o = sev_out(h if not isinstance(h, (list, tuple)) else h[0], True)
            sl = margin_loss(s, o) if a.head == "capsule" else tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(s, o, from_logits=True))
            loss = dl + sl
        g = tape.gradient(loss, vars_)
        opt.apply_gradients((gi, v) for gi, v in zip(g, vars_) if gi is not None)
        return dl, sl

    @tf.function(input_signature=[xs])
    def pred(x):
        h = enc(x, training=False)
        hh = h if not isinstance(h, (list, tuple)) else h[0]
        frames = m2.decode_horizon_local(dec, h, x, HORIZON, training=False)
        return frames[0], tf.argmax(sev_out(hh, False)[0])

    def predict(x):
        f, s = pred(tf.constant(x[None], tf.float32))
        return f.numpy(), int(s)

    val_small = va_bal[::max(1, len(va_bal) // 48)][:48]
    cache_small = [E.compose(pool, i, a.regime)[0] for i in val_small]

    def quick():
        return float(np.mean([predict(x)[1] == lab[i] for x, i in zip(cache_small, val_small)]))

    t0 = time.perf_counter()
    for ep in range(1, a.epochs + 1):
        dls, sls = [], []
        for k in rng.permutation(len(tr_idx)):
            x, y, _ = E.compose(pool, tr_idx[k], a.regime)
            if a.augment:                                  # dihedral transform, identical for inputs and targets; the severity label is unchanged
                q, fl = int(rng.integers(4)), bool(rng.integers(2))
                x, y = np.rot90(x, q, axes=(1, 2)), np.rot90(y, q, axes=(1, 2))
                if fl:
                    x, y = x[:, :, ::-1], y[:, :, ::-1]
                x, y = np.ascontiguousarray(x), np.ascontiguousarray(y)
            dl, sl = step(tf.constant(x[None], tf.float32), tf.constant(y, tf.float32), tf.constant([int(lab[tr_idx[k]])], tf.int32))
            dls.append(float(dl)); sls.append(float(sl))
        log(f"epoch {ep:2d}/{a.epochs}  depth/persist {np.mean(dls):.3f}  sev loss {np.mean(sls):.3f}  val acc(48 balanced) {quick():.3f}  ({(time.perf_counter() - t0) / ep:.0f}s/epoch avg)")

    # ------------------------------------------------------------------ evaluation
    def evaluate_set(idx):
        preds, cls, pcls, X, P = [], [], [], [], []
        for i in idx:
            x, y, _ = E.compose(pool, i, a.regime)
            f, s = predict(x)
            cls.append(s)
            pcls.append(depth_to_severity(float(f.max())))
            X.append(x); P.append(f)
        return np.array(cls), np.array(pcls), X, P

    res = {"id": rid, "regime": a.regime, "head": a.head, "augment": a.augment, "fold": a.fold, "K": a.K, "seed": seed, "epochs": a.epochs, "held_out_tiles": sorted(held)}
    ev = {}
    for name, idx in (("balanced", va_bal), ("natural", va_nat)):
        cls, pcls, X, P = evaluate_set(idx)
        y_true, y_base = lab[idx], base[idx]
        chg = y_true != y_base
        conf = np.zeros((4, 4), int)
        for t, p in zip(y_true, cls):
            conf[t, p] += 1
        recall = [float(conf[c, c] / max(1, conf[c].sum())) for c in range(4)]
        ev[name] = dict(n=len(idx), acc=float(np.mean(cls == y_true)), baseline_last_frame=float(np.mean(y_base == y_true)),
                        forecast_threshold=float(np.mean(pcls == y_true)), macro_recall=float(np.mean(recall)), recall=recall, confusion=conf.tolist(),
                        n_changing=int(chg.sum()), acc_changing=float(np.mean(cls[chg] == y_true[chg])) if chg.any() else None,
                        forecast_threshold_changing=float(np.mean(pcls[chg] == y_true[chg])) if chg.any() else None,
                        acc_unchanged=float(np.mean(cls[~chg] == y_true[~chg])) if (~chg).any() else None,
                        per_window=dict(idx=[int(i) for i in idx], cls=cls.tolist(), pcls=pcls.tolist(), true=y_true.tolist(), base=y_base.tolist()))
        log(f"{name}: capsule/{a.head} acc {ev[name]['acc']:.3f} | last-frame threshold {ev[name]['baseline_last_frame']:.3f} | forecast threshold {ev[name]['forecast_threshold']:.3f} | "
            f"macro recall {ev[name]['macro_recall']:.3f} | changing n={ev[name]['n_changing']} acc {ev[name]['acc_changing']}")
        if name == "natural":
            nat_X, nat_P, nat_cls, nat_idx = X, P, cls, idx
    res["eval"] = ev

    # ------------------------------------------------------------------ symbolic rule layer (natural windows)
    from real_data_loader import RegionData
    rd_lu = RegionData(root_dir=E.ROOT, region="Australia", fidelity_dir=E.FID[30])
    hours = HORIZON * 6 * 300 / 3600.0
    feats = np.array([rule_inputs(pool, rd_lu, i, x, p, hours) for i, x, p in zip(nat_idx, nat_X, nat_P)])
    true_rate = np.array([(np.array(pool["df"][i])[-1].mean() - np.array(pool["dh"][i])[-1].mean()) / hours for i in nat_idx])
    # thresholds re-derived on the fold's training windows (true future depth): midpoint of the moderate and severe class means of each feature
    trf = []
    for i in tr_idx:
        r, c, _ = (int(v) for v in pool["meta"][i])
        x = E.compose(pool, i, a.regime)[0]
        rate = float((np.array(pool["df"][i])[-1].mean() - np.array(pool["dh"][i])[-1].mean()) / hours)
        from predict_severity import _compute_slope_pct, _compute_drainage_proximity_m
        dr = _compute_drainage_proximity_m(rd_lu.load_landuse_patch(r, c, 128), 30.0)
        trf.append((rate, float(x[T_IN - 1, ..., 2].mean()), _compute_slope_pct(np.array(pool["dem"][r:r + 128, c:c + 128]), 30.0), 0.0 if np.isnan(dr) else dr))
    trf = np.array(trf); trl = lab[tr_idx]
    mid = lambda col: float((trf[trl == 2, col].mean() + trf[trl == 3, col].mean()) / 2) if (trl == 2).any() and (trl == 3).any() else float("nan")
    recal = dict(rate=mid(0), rain=mid(1), slope=mid(2), drain=mid(3))
    orig = dict(rate=SRL.DEPTH_RATE_ESCALATE_THRESHOLD, rain=SRL.RAINFALL_CORROBORATE_THRESHOLD, slope=SRL.SLOPE_HIGH_ENERGY_THRESHOLD, drain=SRL.DRAINAGE_FLOODPLAIN_THRESHOLD)
    rules = {}
    for nm, th in (("original_thresholds", orig), ("recalibrated_on_fold_training_windows", recal)):
        SRL.DEPTH_RATE_ESCALATE_THRESHOLD, SRL.RAINFALL_CORROBORATE_THRESHOLD = th["rate"], th["rain"]
        SRL.SLOPE_HIGH_ENERGY_THRESHOLD, SRL.DRAINAGE_FLOODPLAIN_THRESHOLD = th["slope"], th["drain"]
        newc, trig = [], []
        for c0, (rate, rain, slope, dr) in zip(nat_cls, feats):
            c1, reasons = apply_symbolic_rules(int(c0), rate, rain, slope, dr)
            newc.append(c1); trig.append(not reasons[0].startswith("no rule"))
        newc = np.array(newc); yt = lab[nat_idx]
        changed = newc != nat_cls
        rules[nm] = dict(thresholds=th, acc_before=float(np.mean(nat_cls == yt)), acc_after=float(np.mean(newc == yt)), n_changed=int(changed.sum()),
                         n_fixed=int(np.sum(changed & (newc == yt))), n_broke=int(np.sum(changed & (nat_cls == yt))), n_triggered=int(np.sum(trig)))
        log(f"rules [{nm}]: acc {rules[nm]['acc_before']:.3f} -> {rules[nm]['acc_after']:.3f}; changed {rules[nm]['n_changed']} (fixed {rules[nm]['n_fixed']}, broke {rules[nm]['n_broke']})")
    SRL.DEPTH_RATE_ESCALATE_THRESHOLD, SRL.RAINFALL_CORROBORATE_THRESHOLD = orig["rate"], orig["rain"]
    SRL.SLOPE_HIGH_ENERGY_THRESHOLD, SRL.DRAINAGE_FLOODPLAIN_THRESHOLD = orig["slope"], orig["drain"]
    pr = feats[:, 0]
    rules["reason_truth"] = dict(sign_agreement_rate_predicted_vs_true_depth_trend=float(np.mean(np.sign(pr) == np.sign(true_rate))),
                                 corr_rate=float(np.corrcoef(pr, true_rate)[0, 1]) if pr.std() > 0 and true_rate.std() > 0 else None,
                                 n_rising_claims=int(np.sum(pr > 0)), rising_claims_true=float(np.mean(true_rate[pr > 0] > 0)) if np.any(pr > 0) else None,
                                 n_receding_claims=int(np.sum(pr < -1e-4)), receding_claims_true=float(np.mean(true_rate[pr < -1e-4] < 0)) if np.any(pr < -1e-4) else None)
    res["rules"] = rules

    # ------------------------------------------------------------------ Grad-CAM on the encoder's hidden map (full resolution), redesigned after code review F5
    # Two maps per window, both at the shared encoder's hidden map h (128x128x8):
    #   "depth"    : target = signed sum of the predicted depth over ALL leads (direction = sign of the predicted net change);
    #   "severity" : target = the score of the predicted severity class (capsule length / dense logit) - what "Grad-CAM of the severity head" means.
    # Deletion (faithfulness): remove the depth + rain history at the top-10% heat pixels (only pixels with heat > 0, so ties at zero cannot enter) and measure
    #   depth: the L1 change of the predicted CHANGE maps (forecast minus its own anchor; all leads, all pixels);  severity: the change of the predicted class's score,
    # against two controls of the same size: (a) random pixels with the SAME wet/dry mix; (b) the top mask moved 40 pixels (same shape, another place).
    @tf.function(input_signature=[xs])
    def sev_score(x):
        h = enc(x, training=False)
        return sev_out(h if not isinstance(h, (list, tuple)) else h[0], False)[0]

    def cam_of(x, kind):
        xb = tf.constant(x[None], tf.float32)
        with tf.GradientTape() as tape:
            h = enc(xb, training=False)
            hh = h[0] if isinstance(h, (list, tuple)) else h
            tape.watch(hh)
            if kind == "depth":
                frames = m2.decode_horizon_local(dec, [hh], xb, HORIZON, training=False)
                net = tf.reduce_sum(frames[0]) - HORIZON * tf.reduce_sum(xb[0, T_IN - 1, ..., 0])
                target = tf.stop_gradient(tf.where(net >= 0, 1.0, -1.0)) * tf.reduce_sum(frames[0])
            else:
                o = sev_out(hh, False)[0]
                target = tf.gather(o, tf.argmax(o))
        g = tape.gradient(target, hh)
        w = tf.reduce_mean(g, axis=(1, 2))
        cam = tf.nn.relu(tf.reduce_sum(hh[0] * w[0], axis=-1))
        return (cam / (tf.reduce_max(cam) + 1e-9)).numpy()

    def change_maps(xx):
        return predict(xx)[0] - xx[T_IN - 1, ..., 0][None]

    cam_idx = sorted(int(i) for i in np.random.default_rng(777).choice(va_all, size=min(40, len(va_all)), replace=False))   # FIXED windows, independent of the seed
    crng = np.random.default_rng(1)
    store = {kd: dict(ratio=[], eff=[], cams=[]) for kd in ("depth", "severity")}
    for i_win in cam_idx:
        x = E.compose(pool, i_win, a.regime)[0]
        wet = x[T_IN - 1, ..., 0] > 0.05
        wet_ids, dry_ids = np.flatnonzero(wet.ravel()), np.flatnonzero(~wet.ravel())
        base_change, base_score = change_maps(x), sev_score(tf.constant(x[None], tf.float32)).numpy()
        cls_pred = int(np.argmax(base_score))
        for kd in ("depth", "severity"):
            cam = cam_of(x, kd)
            store[kd]["cams"].append(cam.astype(np.float16))
            if wet.any() and (~wet).any():
                store[kd]["ratio"].append(float(cam[wet].mean() / max(cam[~wet].mean(), 1e-9)))
            pos = np.flatnonzero(cam.ravel() > 0)
            k = min(int(0.10 * cam.size), len(pos))
            if k < 20:
                continue
            top_flat = pos[np.argsort(cam.ravel()[pos])[-k:]]
            top = np.unravel_index(top_flat, cam.shape)
            n_wet_top = int(wet[top].sum())

            def effect(ids, kd=kd):
                xm = x.copy(); xm[:, ids[0], ids[1], 0] = 0.0; xm[:, ids[0], ids[1], 2] = 0.0
                if kd == "depth":
                    return float(np.abs(change_maps(xm) - base_change).sum())
                return float(abs(sev_score(tf.constant(xm[None], tf.float32)).numpy()[cls_pred] - base_score[cls_pred]))

            def random_matched():
                pick = np.concatenate([crng.choice(wet_ids, min(n_wet_top, len(wet_ids)), replace=False),
                                       crng.choice(dry_ids, min(k - n_wet_top, len(dry_ids)), replace=False)])
                return np.unravel_index(pick, cam.shape)
            e_top, e_rand = effect(top), float(np.mean([effect(random_matched()) for _ in range(3)]))
            e_roll = effect(((top[0] + 40) % 128, (top[1] + 40) % 128))
            store[kd]["eff"].append((e_top, e_rand, e_roll))
    res["gradcam"] = {}
    for kd in ("depth", "severity"):
        eff, rat = np.array(store[kd]["eff"]).reshape(-1, 3), store[kd]["ratio"]
        res["gradcam"][kd] = dict(n_windows=len(cam_idx), n_with_effect=int(len(eff)), frac_windows_wet_hotter=float(np.mean(np.array(rat) > 1.0)) if rat else None,
                                  median_wet_dry_ratio=float(np.median(rat)) if rat else None,
                                  deletion_effect_top=float(eff[:, 0].mean()) if len(eff) else None, deletion_effect_random=float(eff[:, 1].mean()) if len(eff) else None,
                                  deletion_effect_rolled=float(eff[:, 2].mean()) if len(eff) else None,
                                  frac_top_exceeds_random=float(np.mean(eff[:, 0] > eff[:, 1])) if len(eff) else None,
                                  frac_top_exceeds_both=float(np.mean((eff[:, 0] > eff[:, 1]) & (eff[:, 0] > eff[:, 2]))) if len(eff) else None)
        g = res["gradcam"][kd]
        log(f"Grad-CAM [{kd}]: wet hotter in {g['frac_windows_wet_hotter']} of windows (median ratio {g['median_wet_dry_ratio']}); deletion effect top {g['deletion_effect_top']} vs random {g['deletion_effect_random']} vs shifted {g['deletion_effect_rolled']}; top beats both controls in {g['frac_top_exceeds_both']}")
    res["gradcam_windows"] = cam_idx
    np.savez_compressed(os.path.join(OUT, rid + "_cams.npz"), cams_depth=np.stack(store["depth"]["cams"]), cams_sev=np.stack(store["severity"]["cams"]), idx=np.array(cam_idx))
    json.dump(res, open(out_p, "w"), default=float)
    log("saved " + out_p)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile completion/visualize.py
"""Figures for the corrected study (used by flood_forecasting_study_colab.ipynb; also runnable locally).

  plot_ladder()          skill of every rung (copy the last frame ... ConvLSTM) on the never-seen region, from completion/results/report.json
  plot_ablations()       what the trained models use (test-time removal of one input), from the run files
  plot_regimes()         causal / hybrid / forced
  plot_curves(...)       training curves of one run (train loss / persistence, validation skill per epoch)
  plot_forecast(...)     example forecasts on never-seen windows: last observed depth, true depth at the last lead, forecast, forecast error, persistence error
                         (needs the weights saved by `engine.py run ... --save-weights`)
  plot_gradcam(...)      Grad-CAM of the depth forecast at the ConvLSTM hidden map for a few never-seen windows (same weights)
Every function returns the matplotlib figure and saves a PNG under completion/results/figures/.
"""
import glob
import json
import os
import sys

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import matplotlib
if "google.colab" not in sys.modules and not os.environ.get("MPLBACKEND"):
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

import engine as E  # noqa: E402  (chdirs to the project root)

FIG = "completion/results/figures"
NAMES = {"E60": "Australia -> UK, 60 m", "E30": "Australia -> UK, 30 m", "E480": "Pakistan -> Mozambique, 480 m"}
COL = ["#1B4F72", "#5DA9C9", "#8FB88F", "#C98B5D", "#7A6F9B", "#B03A2E"]


def _save(fig, name):
    os.makedirs(FIG, exist_ok=True)
    fig.savefig(os.path.join(FIG, name), dpi=110, bbox_inches="tight")
    return fig


def _report():
    return json.load(open("completion/results/report.json"))


def plot_ladder(exps=("E60", "E30", "E480"), regime="hybrid"):
    rep = _report()
    h, b = rep["headline"], rep["baselines"]
    rows = [("copy the\nlast frame", lambda e: 0.0), ("straight-line\ntrend", lambda e: (b.get(f"{e}|{regime}|S1") or {}).get("test")),
            ("2-number\nlookup table", lambda e: (b.get(f"{e}|{regime}|S5") or {}).get("test")), ("per-pixel\nMLP", lambda e: (b.get(f"{e}|{regime}|S4c") or {}).get("test")),
            ("original\nCNN-LSTM (v1)", lambda e: (h.get(f"{e}|v1|{regime}") or {}).get("test_mean")), ("local CNN", lambda e: (h.get(f"{e}|local|{regime}") or {}).get("test_mean")),
            ("ConvLSTM\n(final)", lambda e: (h.get(f"{e}|clstm|{regime}") or {}).get("test_mean"))]
    rows = [(n, f) for n, f in rows if any(f(e) is not None for e in exps)]
    fig, ax = plt.subplots(figsize=(11, 4.6))
    w = 0.8 / len(exps)
    for k, e in enumerate(exps):
        vals = [f(e) if f(e) is not None else np.nan for _, f in rows]
        bars = ax.bar(np.arange(len(rows)) + k * w, vals, w, label=NAMES[e], color=COL[k])
        for x, v in zip(np.arange(len(rows)) + k * w, vals):
            if not np.isnan(v):
                ax.text(x, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)
    ax.set_xticks(np.arange(len(rows)) + w * (len(exps) - 1) / 2)
    ax.set_xticklabels([n for n, _ in rows], fontsize=9)
    ax.set_ylabel("skill vs 'copy the last frame'  (0 = same, 1 = perfect)")
    ax.set_title(f"Skill on the never-seen region ({regime} rain input, mean of 3 tile folds)")
    ax.axhline(0, color="grey", lw=0.8); ax.legend(fontsize=9); ax.set_ylim(-0.05, 1.0)
    fig.tight_layout()
    return _save(fig, "ladder.png")


def plot_ablations(exps=("E60", "E480"), archs=("clstm", "local"), regime="hybrid"):
    g = __import__("d_aggregate").load_runs()
    keys = [("none", "normal"), ("rain_removed", "rain\nremoved"), ("terrain_removed", "terrain\nremoved"), ("depth_hist_frozen", "depth history\nfrozen"), ("rain_shifted_earlier", "rain strictly\npast")]
    fig, ax = plt.subplots(figsize=(11, 4.4))
    combos = [(e, a) for e in exps for a in archs]
    w = 0.8 / max(1, len(combos))
    for k, (e, a) in enumerate(combos):
        runs = g.get((e, a, regime, "full", False, False, E.__dict__.get("EPOCHS_F", 30) if False else int(os.environ.get("FINAL_EPOCHS", 30)), int(os.environ.get("FINAL_NTRAIN", 300))), [])
        vals = [np.mean([r["ablation_test"][key] for r in runs if r.get("ablation_test") and key in r["ablation_test"]]) if runs and all(key in (r.get("ablation_test") or {}) for r in runs) else np.nan for key, _ in keys]
        ax.bar(np.arange(len(keys)) + k * w, vals, w, label=f"{a} / {NAMES[e]}", color=COL[k % len(COL)])
    ax.set_xticks(np.arange(len(keys)) + w * (len(combos) - 1) / 2)
    ax.set_xticklabels([n for _, n in keys], fontsize=9)
    ax.set_ylim(-0.2, 1.0); ax.axhline(0, color="grey", lw=0.8)
    ax.set_ylabel("skill when that input is removed at test time")
    ax.set_title("What the trained models use: the depth history, almost nothing else")
    ax.legend(fontsize=8)
    fig.tight_layout()
    return _save(fig, "ablations.png")


def plot_regimes(exps=("E60", "E480"), archs=("local", "clstm")):
    rep = _report()["headline"]
    fig, ax = plt.subplots(figsize=(10, 4.2))
    combos = [(e, a) for e in exps for a in archs]
    w = 0.8 / len(combos)
    for k, (e, a) in enumerate(combos):
        vals = [(rep.get(f"{e}|{a}|{r}") or {}).get("test_mean", np.nan) for r in ("causal", "hybrid", "forced")]
        ax.bar(np.arange(3) + k * w, vals, w, label=f"{a} / {NAMES[e]}", color=COL[k % len(COL)])
    ax.set_xticks(np.arange(3) + w * (len(combos) - 1) / 2)
    ax.set_xticklabels(["causal\n(past rain only)", "hybrid\n(the original input)", "forced\n(future rain too)"])
    ax.set_ylim(0.5, 1.0); ax.set_ylabel("skill on the never-seen region"); ax.set_title("Rain settings (axis starts at 0.5)"); ax.legend(fontsize=8)
    fig.tight_layout()
    return _save(fig, "regimes.png")


def plot_curves(exp="E60", arch="clstm", regime="hybrid", fold=0):
    fs = sorted(glob.glob(f"completion/results/runs/{exp}_{arch}_{regime}_full_f{fold}of3_*.json"))
    fs = [f for f in fs if "_blk" not in f and "_b8" not in f and "_lr" not in f]
    if not fs:
        raise FileNotFoundError("no run file")
    r = json.load(open(fs[0]))
    ep, tl, vs = zip(*r["history"])
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    axes[0].plot(ep, tl); axes[0].axhline(1.0, color="grey", ls="--", label="= copying the last frame"); axes[0].set_title("training loss / persistence loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
    axes[1].plot(ep, vs); axes[1].axhline(0, color="grey", ls="--"); axes[1].set_title("skill on 48 held-out windows (higher is better)"); axes[1].set_xlabel("epoch")
    fig.suptitle(f"{arch} - {NAMES[exp]} - {regime} - fold {fold}")
    fig.tight_layout()
    return _save(fig, f"curves_{exp}_{arch}_{regime}_f{fold}.png")


def _load_model(exp, arch, regime, fold, epochs, K=3, batch=1):
    seed = 1000 + 10 * fold + K
    rid = f"{exp}_{arch}_{regime}_full_f{fold}of{K}_s{seed}_ep{epochs}"
    wd = os.path.join(E.CACHE, "weights")
    ep, dp = os.path.join(wd, rid + "_enc.weights.h5"), os.path.join(wd, rid + "_dec.weights.h5")
    if not (os.path.exists(ep) and os.path.exists(dp)):
        raise FileNotFoundError(f"weights not found ({rid}); train with --save-weights")
    tf = E._tf()
    T = E.T_IN + (E.HORIZON if regime == "forced" else 0)
    enc, dec, dech, _ = E.build_nets(arch, T)
    enc.load_weights(ep); dec.load_weights(dp)
    return tf, enc, dec, dech


def plot_forecast(exp="E60", arch="clstm", regime="hybrid", fold=0, epochs=30, n=3):
    tf, enc, dec, dech = _load_model(exp, arch, regime, fold, epochs)
    tr_pool, te_pool, tr_idx, va_idx, te_idx, held, seed = E.split_for(exp, fold, 3, n_train=int(os.environ.get("FINAL_NTRAIN", 300)), log=lambda s: None)
    # pick wet windows with visible change: the largest persistence error among a spread of never-seen windows
    cand = te_idx[::max(1, len(te_idx) // 40)]
    err = []
    for i in cand:
        x, y, m = E.compose(te_pool, i, regime)
        err.append(float(((x[E.T_IN - 1, ..., 0][None] - y) ** 2)[:, m].sum()))
    pick = [cand[j] for j in np.argsort(err)[::-1][:n]]
    fig, axes = plt.subplots(n, 5, figsize=(16, 3.2 * n))
    axes = np.atleast_2d(axes)
    for r_, i in enumerate(pick):
        x, y, m = E.compose(te_pool, i, regime)
        p = dech(dec, enc(tf.constant(x[None]), training=False), tf.constant(x[None]), E.HORIZON, training=False).numpy()[0]
        last, k = x[E.T_IN - 1, ..., 0], E.HORIZON - 1
        vmax = max(float(y[k].max()), float(last.max()), 1e-3)
        emax = max(float(np.abs(p[k] - y[k]).max()), float(np.abs(last - y[k]).max()), 1e-4)
        panels = [(last, "last observed depth (m)", "Blues", 0, vmax), (y[k], "TRUE depth 2.5 h later (m)", "Blues", 0, vmax), (p[k], f"{arch} forecast 2.5 h (m)", "Blues", 0, vmax),
                  (p[k] - y[k], "forecast error (m)", "RdBu_r", -emax, emax), (last - y[k], "'copy the last frame' error (m)", "RdBu_r", -emax, emax)]
        for c, (img, title, cmap, lo, hi) in enumerate(panels):
            a_ = axes[r_, c]
            im = a_.imshow(np.where(m, img, np.nan), cmap=cmap, vmin=lo, vmax=hi)
            a_.set_title(title if r_ == 0 else "", fontsize=10); a_.axis("off")
            if c in (2, 4):
                fig.colorbar(im, ax=a_, fraction=0.046)
        axes[r_, 0].text(-0.05, 0.5, f"window {i}", transform=axes[r_, 0].transAxes, rotation=90, va="center", ha="right", fontsize=9)
    fig.suptitle(f"Never-seen region ({NAMES[exp]}): forecasts of a {arch} model vs the truth and vs copying the last frame ({regime})", y=1.0)
    fig.tight_layout()
    return _save(fig, f"forecast_{exp}_{arch}_{regime}_f{fold}.png")


def plot_gradcam(exp="E60", arch="clstm", regime="hybrid", fold=0, epochs=30, n=3):
    tf, enc, dec, dech = _load_model(exp, arch, regime, fold, epochs)
    import model_v2 as m2
    tr_pool, te_pool, tr_idx, va_idx, te_idx, held, seed = E.split_for(exp, fold, 3, n_train=int(os.environ.get("FINAL_NTRAIN", 300)), log=lambda s: None)
    cand = te_idx[::max(1, len(te_idx) // 40)]
    err = [float(((lambda x, y, m: ((x[E.T_IN - 1, ..., 0][None] - y) ** 2)[:, m].sum())(*E.compose(te_pool, i, regime)))) for i in cand]
    pick = [cand[j] for j in np.argsort(err)[::-1][:n]]
    fig, axes = plt.subplots(2, n, figsize=(4.6 * n, 8))
    for c, i in enumerate(pick):
        x, y, m = E.compose(te_pool, i, regime)
        xb = tf.constant(x[None], tf.float32)
        with tf.GradientTape() as tape:
            h = enc(xb, training=False)
            hh = h[0] if isinstance(h, (list, tuple)) else h
            tape.watch(hh)
            frames = m2.decode_horizon_local(dec, [hh], xb, E.HORIZON, training=False)
            net = tf.reduce_sum(frames[0]) - E.HORIZON * tf.reduce_sum(xb[0, E.T_IN - 1, ..., 0])
            target = tf.stop_gradient(tf.where(net >= 0, 1.0, -1.0)) * tf.reduce_sum(frames[0])
        g = tape.gradient(target, hh)
        cam = tf.nn.relu(tf.reduce_sum(hh[0] * tf.reduce_mean(g, axis=(1, 2))[0], axis=-1)).numpy()
        cam = cam / (cam.max() + 1e-9)
        last = x[E.T_IN - 1, ..., 0]
        axes[0, c].imshow(np.where(m, last, np.nan), cmap="Blues"); axes[0, c].set_title("last observed depth", fontsize=10); axes[0, c].axis("off")
        axes[1, c].imshow(np.where(m, last, np.nan), cmap="Greys", alpha=0.6)
        axes[1, c].imshow(np.where(m, cam, np.nan), cmap="jet", alpha=0.55, vmin=0, vmax=1); axes[1, c].set_title("Grad-CAM: where the forecast's evidence is", fontsize=10); axes[1, c].axis("off")
    fig.suptitle(f"Grad-CAM of the depth forecast ({arch}, {NAMES[exp]}); hot = evidence for the predicted change", y=1.0)
    fig.tight_layout()
    return _save(fig, f"gradcam_{exp}_{arch}_{regime}_f{fold}.png")


## Build the window caches and check the input builder

`selftest` verifies, on real windows, that the study's input builder reproduces the original loader exactly, that the causal / forced rainfall
inputs equal the raw rainfall files of the intended intervals, and that an untrained model equals persistence in every regime.

In [ ]:
import os, subprocess, sys
ENV = dict(os.environ, TF_THREADS=str(THREADS))
def run(args, tail=12):
    p = subprocess.run([sys.executable, "-u"] + args, env=ENV, capture_output=True, text=True)
    lines = [l for l in (p.stdout + p.stderr).splitlines() if not l.startswith(("I0000", "W0000", "WARNING"))]
    print("\n".join(lines[-tail:]))
    if p.returncode != 0:
        raise RuntimeError(f"command failed: {args}")
for e in EXPERIMENTS:
    run(["completion/engine.py", "cache", e], tail=4)
ENV["SELFTEST_EXPS"] = ",".join(EXPERIMENTS)
run(["completion/engine.py", "selftest"], tail=8)

## Baselines

Persistence is skill 0 by definition. `S1` linear on depth history only; `S3` linear with rainfall, state and trend; `S5` a two-feature lookup table (current depth x last
one-step change); `S4` a small per-pixel MLP (no convolution, no recurrence). The `c` versions clip the forecast at zero depth.

In [ ]:
if RUN_BASELINES:
    for e in EXPERIMENTS:
        for r in REGIMES:
            for f in FOLDS:
                print(f"--- baselines {e} {r} fold {f}")
                run(["completion/b_baselines.py", e, r, str(f), "--n-train", str(N_TRAIN)], tail=3)

## Train and evaluate the neural models

One run = one (experiment, regime, fold, model): train on the tiles outside the fold, score the held-out tiles and every window of the never-seen region, then the
test-time ablations. Weights are saved (they are tiny) so the forecast maps and Grad-CAM below can be drawn. Results are written to `completion/results/runs/*.json` (a finished run is skipped, so the cell can be re-run after a disconnect).

In [ ]:
for e in EXPERIMENTS:
    for r in REGIMES:
        for f in FOLDS:
            for a in MODELS:
                print(f"--- {e} {r} fold {f} model {a}")
                run(["completion/engine.py", "run", e, r, str(f), "--arch", a, "--epochs", str(EPOCHS), "--n-train", str(N_TRAIN), "--save-weights"], tail=6)

## Results

Skill = 1 - SSE_model / SSE_persistence (0 = copying the last frame). Intervals are cluster bootstraps (tiles for the held-out score, start times for the never-seen region).
The hypotheses block reports the pre-registered checks (H1-H6) for the groups that were run.

In [ ]:
ENV["FINAL_EPOCHS"], ENV["FINAL_NTRAIN"] = str(EPOCHS), str(N_TRAIN)
p = subprocess.run([sys.executable, "completion/g_report.py"], env=ENV, capture_output=True, text=True)
print(p.stdout[-12000:] if p.returncode == 0 else p.stderr[-3000:])

## Figures

1. **Skill ladder** - every rung from "copy the last frame" to the ConvLSTM on the never-seen region (the original CNN-LSTM row appears if you also ran `--arch v1`).
2. **What the trained models use** - skill when one input is removed at test time: the depth history is what they use.
3. **Rain settings** - causal / hybrid / forced.
4. **Training curves** of the first run.
5. **Example forecasts** on never-seen windows: last observed depth, the TRUE depth 2.5 h later, the model's forecast, and the forecast error next to the error of "copy the last frame".
6. **Grad-CAM** of the depth forecast: where the model's evidence for the predicted change sits.

In [ ]:
import os, sys
os.environ["FINAL_EPOCHS"], os.environ["FINAL_NTRAIN"] = str(EPOCHS), str(N_TRAIN)
sys.path.insert(0, "completion")
import matplotlib.pyplot as plt
import visualize as V

def show(fn, *a, **k):
    try:
        fn(*a, **k)
        plt.show()
    except Exception as ex:                     # a figure needs its data: say what is missing instead of failing the notebook
        print(f"[{fn.__name__} skipped: {type(ex).__name__}: {ex}]")

_arch = "clstm" if "clstm" in MODELS else MODELS[0]
show(V.plot_ladder, tuple(EXPERIMENTS))
show(V.plot_ablations, tuple(EXPERIMENTS), tuple(MODELS))
show(V.plot_regimes, tuple(EXPERIMENTS), tuple(MODELS))
show(V.plot_curves, EXPERIMENTS[0], _arch, REGIMES[0], FOLDS[0])
show(V.plot_forecast, EXPERIMENTS[0], _arch, REGIMES[0], FOLDS[0], EPOCHS)
show(V.plot_gradcam, EXPERIMENTS[0], _arch, REGIMES[0], FOLDS[0], EPOCHS)

## Optional: severity, symbolic rules and Grad-CAM (Australia at 30 m)

The severity label is only informative for Australia at 30 m. This trains the joint model (a small ConvLSTM encoder shared by the depth head and a severity head)
with either the **capsule** head or a plain **dense** head, and reports accuracy against the two threshold baselines, the symbolic rule layer's effect
(original vs re-derived thresholds, and whether its stated reasons are true), and Grad-CAM faithfulness. Needs `RUN_SEVERITY = True` and `"E30"` data.

In [ ]:
if RUN_SEVERITY:
    for r in ("hybrid",):
        for h in ("capsule", "dense"):
            for f in FOLDS:
                print(f"--- severity {r} {h} fold {f}")
                run(["completion/severity_study.py", r, h, str(f), "--epochs", str(EPOCHS)], tail=8)

In [ ]:
import shutil
shutil.make_archive("study_results", "zip", "completion/results")
print("wrote study_results.zip")
try:
    from google.colab import files
    files.download("study_results.zip")
except Exception:
    pass